# Library

In [ ]:
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import pprint
import numpy as np
from matplotlib.patches import Rectangle, Circle,Polygon
from itertools import permutations
from IPython.display import display, clear_output
import time
import math
import heapq
from lxml import etree
from PIL import Image
import numpy as np
import os
import shutil
import csv

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Arc, Polygon
import math
import re
from matplotlib.patches import Polygon
import itertools
import matplotlib.patches as patches


import csv
import time
import requests
from urllib.parse import urlparse, parse_qs, urlunparse, urlencode, urljoin
from lxml import html
from requests.adapters import HTTPAdapter, Retry

import os
import csv
import pathlib
import requests
from urllib.parse import urlparse


# Switch to a backend that supports pop-out windows (like Qt5Agg, TkAgg, MacOSX)
# You might need to install the backend if you don't have it (e.g., pip install pyqt5 for Qt5Agg)
matplotlib.use('Agg') # Or try 'TkAgg' or 'MacOSX'

# Layout (.brd) Extract Function

In [ ]:
# Convert XML to dict
def xml_to_dict(element):
    """
    Recursively converts an XML element and its children into a dictionary.
    """
    # Convert attributes and text of the element to a dictionary
    result = {element.tag: {} if element.attrib else None}

    # Add element attributes to the dictionary
    if element.attrib:
        result[element.tag].update((key, value) for key, value in element.attrib.items())

    # Add element text to the dictionary if it exists
    if element.text and element.text.strip():
        text = element.text.strip()
        if result[element.tag]:
            result[element.tag]['text'] = text
        else:
            result[element.tag] = text

    # Convert child elements
    children = list(element)
    if children:
        child_dict = {}
        for child in children:
            child_result = xml_to_dict(child)
            if child.tag in child_dict:
                if isinstance(child_dict[child.tag], list):
                    child_dict[child.tag].append(child_result[child.tag])
                else:
                    child_dict[child.tag] = [child_dict[child.tag], child_result[child.tag]]
            else:
                child_dict.update(child_result)
        if result[element.tag]:
            result[element.tag].update(child_dict)
        else:
            result[element.tag] = child_dict

    return result

# Output elements from a file according the path
def get_elements(file,root):
    list_elements = []
    for elements in file.getroot().findall(root):
        for element in elements:
            list_elements.append(element)
    return list_elements

# print list of elements attribute
def print_elements(list_elements):
    for element in list_elements:
        print(element,element.attrib.items())

# Output list of elements according the tag name
def get_tag_name(list_elements,tag_name):
    return [elem for elem in list_elements if elem.tag == tag_name]

# Output wire from an element
def get_wire(wire):
    return [[float(wire.get('x1')),float(wire.get('y1'))],[float(wire.get('x2')),float(wire.get('y2'))]]
    # return [[float(wire.get('x1')),float(wire.get('x2'))],[float(wire.get('y1')),float(wire.get('y2'))]]

# Output wires from list of  element
def get_wires(elements):
    wires =[]
    for element in elements:
        wire = get_wire(element)
        wires.append(wire)
    return wires

# Output smd of an element
def get_smd(ICsmd):
    smd_names = []
    # print('HWE',ICsmd)
    if isinstance(ICsmd, list):
        for smd in ICsmd:
            # print('HWE',smd.keys())
            x = float(smd.get('x'))
            y = float(smd.get('y'))
            tmp_smd = {}
            tmp_smd['name'] = smd.get('name')
            tmp_smd['pos'] = [x,y]
            tmp_smd['wid_len'] = [float(smd.get('dx')),float(smd.get('dy'))]
            tmp_smd['rot'] = 'R0'
            if 'rot' in smd.keys():
                tmp_smd['rot'] = smd.get('rot')
                if tmp_smd['rot'].upper().startswith('M'):
                    tmp_smd['is_mirror'] = True
                else:
                    tmp_smd['is_mirror'] = False
            smd_names.append(tmp_smd)
            
    else:
        x = float(ICsmd.get('x'))
        y = float(ICsmd.get('y'))
        tmp_smd = {}
        tmp_smd['name'] = ICsmd.get('name')
        tmp_smd['pos'] = [x, y]
        tmp_smd['wid_len'] = [float(ICsmd.get('dx')), float(ICsmd.get('dy'))]
        tmp_smd['rot'] = 'R0'
        if 'rot' in ICsmd.keys():
            tmp_smd['rot'] = ICsmd.get('rot')
            if tmp_smd['rot'].upper().startswith('M'):
                tmp_smd['is_mirror'] = True
            else:
                tmp_smd['is_mirror'] = False
        smd_names.append(tmp_smd)
    return smd_names


def get_pad(ICpad):
    # print(ICpad)
    pads= []
    # print('HWE',ICpad)
    if isinstance(ICpad, list):
        for pad in ICpad:
            # print('HWE',smd.keys())
            x = float(pad.get('x'))
            y = float(pad.get('y'))
            tmp_pad = {}
            tmp_pad['name'] = pad.get('name')
            tmp_pad['pos'] = [x,y]
            tmp_pad['drill'] = pad.get('drill')
            tmp_pad['diameter'] = pad.get('diameter')
            pads.append(tmp_pad)
            
    else:
        x = float(ICpad.get('x'))
        y = float(ICpad.get('y'))
        tmp_pad = {}
        tmp_pad['name'] = ICpad.get('name')
        tmp_pad['pos'] = [x, y]
        tmp_pad['drill'] = ICpad.get('drill')
        tmp_pad['diameter'] = ICpad.get('diameter')

        pads.append(tmp_pad)
    return pads

def get_one_ic(pack,library_name):
    IC = {}
    IC['name'] = pack['name']
    IC['library'] = library_name


    if 'wire' in pack.keys():
        IC['wire'] = get_wires(pack['wire'])
    else:
        IC['wire'] = []

    if 'smd' in pack.keys():
        smd = pack['smd']
        # print(smd)
        IC['smd'] = get_smd(smd)
    else:
        IC['smd'] = []
    
    if 'pad' in pack.keys():
        pad = pack['pad']
        IC['pad'] = get_pad(pad)
    else:
        IC['pad'] = []


    return IC

# Output library from libraries
def get_IC_info(list_library):
    ICs = []
    for library in list_library:
        if isinstance(library['packages']['package'], list):
            for pack in library['packages']['package']:
                # print(pack['name'],library['name'] ,pack.keys())
                ICs.append(get_one_ic(pack, library['name']))
        else:
            # print(library['packages']['package']['name'],library['name'],library['packages']['package'].keys())
            # print(library['packages']['package']['smd'])
            ICs.append(get_one_ic(library['packages']['package'],library['name']))

    return ICs

# Output element information
def get_element_info(list_element):
    elements = []
    for ele in list_element:
        element = {}

        element['name'] = ele['name']
        element['pos'] = [ele['x'],ele['y']]
        element['package'] = ele['package']
        if 'rot' in ele.keys():
            element['rot'] = ele['rot']
            if element['rot'].upper().startswith('M'):
                element['is_mirror'] = True
            else:
                element['is_mirror'] = False
        else:
            element['rot'] = 'R0'
        elements.append(element)

    return elements

# Output position of contactref
def generate_wire_terminal(connection_point, elements, ic_library):
    element_map = {el['name']: el for el in elements}
    
    def get_position(element, pad_name):
        element_pos = [float(coord) for coord in element['pos']]
        rotation = int(element.get('rot', 'R0')[1:]) if element.get('rot', '').startswith('R') else 0
        is_mirror = True if element.get('rot', '').startswith('M') else False
        ic = next((ic for ic in ic_library if ic['name'] == element['package']), None)
        
        if ic:
            pad = next((smd for smd in ic.get('smd', []) if smd['name'] == pad_name), None)
            if pad:
                rotated_pad = rotate_point(pad['pos'], [0, 0], rotation)
                if is_mirror:
                    rotated_pad[0] = -rotated_pad[0]
                return [
                    element_pos[0] + rotated_pad[0],
                    element_pos[1] + rotated_pad[1]
                ]
        return element_pos  # If no pad is found, return the element position itself
    
    # Extract the element and pad from the input
    element_name = connection_point['ele']
    pad_name = connection_point['pad']
    
    # Get the corresponding element from the elements list
    element = element_map[element_name]
    
    # Calculate the position of the pad
    position = get_position(element, pad_name)
    
    return position

# Output contactref from signals
def get_contactref(list_contactref,elements,ic_library):
    contactrefs = []
    
    if  isinstance(list_contactref, list):
        for contact in list_contactref:
            # print("contact:",contact)
            contactref = {}
            contactref['ele'] = contact['element']
            contactref['pad'] = contact['pad']
            contactref['pos'] = generate_wire_terminal( {'ele': contact['element'], 'pad': contact['pad']}, elements, ic_library)
            contactrefs.append(contactref)
    else:
        # print("contact2:",list_contactref)
        contactref = {}
        contactref['ele'] = list_contactref['element']
        contactref['pad'] = list_contactref['pad']
        contactref['pos'] = generate_wire_terminal( {'ele': list_contactref['element'], 'pad': list_contactref['pad']}, elements, ic_library)
        contactrefs.append(contactref)

    return contactrefs

# Output wire from signals
def get_wire_signals(list_wires):
    wires = []

    if isinstance(list_wires, list):
        for w in list_wires:
            if w['layer'] == '19':
                wire = {}
                wire['wire'] = get_wire(w)
                wires.append(wire)
    else:
        if list_wires['layer'] == '19':
                wire = {}
                wire['wire'] = get_wire(list_wires)
                wires.append(wire)

    return wires


# Output Signal
def get_signal_info(list_signal,elements,ic_library):
    
    signals = []
    
    for sig in list_signal:
        # print('name',sig['name'])
        signal = {}
        signal['name'] = sig['name']
        signal['contactref'] = []
        if 'contactref' in sig.keys():
            signal['contactref'] = get_contactref(sig['contactref'],elements,ic_library)
        
        signals.append(signal)
        
    return signals

def rotate_point(point, origin=[0, 0], angle=0):
    """Rotate a point counterclockwise by a given angle around a specified origin."""
    radians = np.deg2rad(angle)
    ox, oy = origin
    x, y = point

    x_new = ox + np.cos(radians) * (x - ox) - np.sin(radians) * (y - oy)
    y_new = oy + np.sin(radians) * (x - ox) + np.cos(radians) * (y - oy)

    return [x_new, y_new]


In [ ]:


# Plot wires from a list of wires
def plot_wires(list_wires):
    plt.figure(figsize=(6, 4))
    for wire in list_wires:
        plt.plot(wire[0], wire[1], marker='o')

    # Set the aspect of the plot to be equal
    plt.gca().set_aspect('equal', adjustable='box')

    # Add labels and title
    plt.xlabel('X coordinate')
    plt.ylabel('Y coordinate')
    plt.title('Wires Visualization')

    # Show the grid
    plt.grid(True)

    # Display the plot
    plt.show()


def plot_pcb_and_elements(ax, border_segments, elements, ic_library, signals=None, mirrored=False, legend=False, show_rectangles=True, show_wires=True):
    # Extract the coordinates from the border segments
    x_coords_border = [coord[0] for segment in border_segments for coord in segment]
    y_coords_border = [coord[1] for segment in border_segments for coord in segment]

    x_min, x_max = min(x_coords_border), max(x_coords_border)
    y_min, y_max = min(y_coords_border), max(y_coords_border)

    # Handle mirroring if necessary
    if mirrored:
        x_coords_border = [-x for x in x_coords_border]
        border_segments = [[[-coord[0], coord[1]] for coord in segment] for segment in border_segments]

    # Plot the PCB border
    for segment in border_segments:
        x_values = [point[0] for point in segment]
        y_values = [point[1] for point in segment]
        ax.plot(x_values, y_values, color='black', linestyle='-', linewidth=2, label='PCB Border')

    # Define a colormap for the elements
    num_elements = len(elements)
    color_map = plt.cm.get_cmap('tab20', num_elements)

    for idx, element in enumerate(elements):
        is_mirrored = element.get('rot', '').startswith('MR')

        # Skip elements that don't match the current mirroring state
        if is_mirrored != mirrored:
            continue

        ic = next((item for item in ic_library if item['name'] == element['package']), None)
        if ic is None:
            continue

        element_pos = [float(coord) for coord in element['pos']]
        element_rotation = int(element.get('rot', 'R0')[1:]) if element.get('rot', '').startswith('R') else 0

        if mirrored:
            element_pos[0] = -element_pos[0]
            element_rotation = -element_rotation

        circle = Circle(element_pos, radius=0.05, color=color_map(idx), alpha=0.8, label=f"{element['name']} Position")
        ax.add_patch(circle)

        # Plot pads with drill holes as filled rings (annuli)
        for pad in ic.get('pad', []):
            pad_pos = [pad['pos'][0] + element_pos[0], pad['pos'][1] + element_pos[1]]
            if 'drill' in pad and 'diameter' in pad:
                outer_radius = float(pad['diameter']) / 2
                drill_radius = float(pad['drill']) / 2
                
                # Draw the outer circle of the pad
                pad_circle_outer = Circle(pad_pos, radius=outer_radius, color='blue', fill=True, alpha=0.5)
                ax.add_patch(pad_circle_outer)
                
                # Draw the inner (drill) circle as an empty space
                pad_circle_drill = Circle(pad_pos, radius=drill_radius, color='white', fill=True)
                ax.add_patch(pad_circle_drill)

        # Rotate and plot SMDs
        for smd in ic['smd']:
            smd_rotation_str = smd.get('rot', 'R0') if smd.get('rot') else 'R0'
            smd_rotation = int(smd_rotation_str[1:]) if smd_rotation_str.startswith('R') else 0

            smd_relative_pos = smd['pos']
            smd_wid_len = smd.get('wid_len', [0.1, 0.1])

            smd_corners = [
                [-smd_wid_len[0] / 2, -smd_wid_len[1] / 2],
                [smd_wid_len[0] / 2, -smd_wid_len[1] / 2],
                [smd_wid_len[0] / 2, smd_wid_len[1] / 2],
                [-smd_wid_len[0] / 2, smd_wid_len[1] / 2]
            ]

            smd_corners_rotated = [rotate_point(corner, [0, 0], smd_rotation) for corner in smd_corners]
            smd_corners_rotated = [[corner[0] + smd_relative_pos[0], corner[1] + smd_relative_pos[1]] for corner in smd_corners_rotated]
            smd_corners_final = [rotate_point(corner, [0, 0], element_rotation) for corner in smd_corners_rotated]
            smd_corners_final = [[corner[0] + element_pos[0], corner[1] + element_pos[1]] for corner in smd_corners_final]

            if show_rectangles:
                polygon = Polygon(smd_corners_final, color=color_map(idx), alpha=0.5)
                ax.add_patch(polygon)
            else:
                ax.scatter(*smd_corners_final[0], color=color_map(idx), label=f"{element['name']} SMD {smd['name']}")

            smd_label_x = np.mean([corner[0] for corner in smd_corners_final])
            smd_label_y = np.mean([corner[1] for corner in smd_corners_final])
            ax.text(smd_label_x, smd_label_y, f"{smd['name']}", fontsize=8, ha='center', va='center', color='white')

    # Plot positions of terminals if signals are provided
    if signals:
        for signal in signals:
            for contact in signal['contactref']:
                # Check if the current contact belongs to a mirrored element
                contact_element = next((el for el in elements if el['name'] == contact['ele']), None)
                contact_is_mirrored = contact_element and contact_element.get('rot', '').startswith('MR')

                # Only plot terminals that match the current mirroring state
                if contact_is_mirrored == mirrored:
                    pos = contact['pos']
                    if mirrored:
                        pos = [-pos[0], pos[1]]
                    ax.add_patch(Circle(pos, radius=0.1, color='red', fill=True, alpha=0.6))

    # Set the limits of the plot
    margin = 5.0
    ax.set_xlim(min(x_coords_border) - margin, max(x_coords_border) + margin)
    ax.set_ylim(y_min - margin, y_max + margin)

    # Set the aspect of the plot to be equal
    ax.set_aspect('equal', adjustable='box')

    # Add labels and title
    plot_type = "Mirrored" if mirrored else "Regular"
    ax.set_xlabel('X coordinate')
    ax.set_ylabel('Y coordinate')
    ax.set_title(f'PCB and SMD Positions ({plot_type})')

    # Show the grid
    ax.grid(True, which='both', axis='both', color='gray', linestyle='--', linewidth=0.5)

    # Set mirrored grid settings if necessary
    if mirrored:
        ax.set_xticks(np.arange(x_min - margin, x_max + margin, 1) * -1)
    else:
        ax.set_xticks(np.arange(x_min - margin, x_max + margin, 1))

    ax.set_yticks(np.arange(y_min - margin, y_max + margin, 1))

    # Show legend if requested
    if legend:
        ax.legend(loc='upper right', fontsize='small', markerscale=0.5)

               
def plot_all_elements(border_segments, elements, ic_library, signals=None, view="both", legend=False, show_rectangles=True):
    """Plot all elements with options to display mirrored, regular, or both elements."""
    
    # Determine the number of subplots based on the view parameter
    if view == "both":
        num_plots = 2
    else:
        num_plots = 1
    
    # Create subplots
    fig, axes = plt.subplots(num_plots, 1, figsize=(10, 8 * num_plots))
    
    # If only one plot, axes is not a list
    if not isinstance(axes, np.ndarray):
        axes = [axes]

    # Plot based on the view parameter
    if view in ["regular", "both"]:
        plot_pcb_and_elements(axes[0], border_segments, elements, ic_library, signals=signals, mirrored=False, legend=legend, show_rectangles=show_rectangles)

    if view in ["mirrored", "both"]:
        plot_pcb_and_elements(axes[-1], border_segments, elements, ic_library, signals=signals, mirrored=True, legend=legend, show_rectangles=show_rectangles)

    # Adjust layout to avoid overlap
    plt.tight_layout()

    # plt.show()
    
    # Display the plots
    plt.savefig('pcb_plot.png', dpi=300, bbox_inches='tight')


def process_board_file(file_path):
    """
    Process the board file and extract relevant information.

    Args:
        file_path (str): Path to the board file.

    Returns:
        dict: A dictionary containing board information.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    xml_dict = xml_to_dict(root)

    board_dimension = xml_dict['eagle']['drawing']['board']['plain']['wire']
    board_library = xml_dict['eagle']['drawing']['board']['libraries']['library']
    board_element = xml_dict['eagle']['drawing']['board']['elements']['element']
    board_signal = xml_dict['eagle']['drawing']['board']['signals']['signal']

    board_info = {}
    board_info['board dimension'] = get_wires(board_dimension)
    board_info['IC_library'] = get_IC_info(board_library)
    board_info['elements'] = get_element_info(board_element)
    board_info['signals'] = get_signal_info(board_signal, board_info['elements'], board_info['IC_library'])

    return board_info


# Sample

In [ ]:
if __name__ == "__main__":
    # Load and parse the XML file for the specified IC
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\Module_library"
    IC_name = "ICM-20948"
    file_name = f"{folder_path}\\{IC_name}\\{IC_name}.brd"

    # file_name = r"/Users/linkaiyuan/Documents/EAGLE/projects/sparkfunmicropractis/SparkFun_IMU_Breakout_ICM-20948.brd"
    # Process the board file and plot
    board_info = process_board_file(file_name)
    print(board_info['signals'][1])

    plot_all_elements(border_segments=board_info['board dimension'],
                    elements=board_info['elements'],
                    ic_library=board_info['IC_library'],
                    signals=board_info['signals'],
                    view="both", legend=False, show_rectangles=True)


# Schematic (.sch) Extract Function

In [ ]:
def retrieve_sheet_info(schematic_sheets):
    return schematic_sheets

In [ ]:
# Convert XML to dict
def xml_to_dict(element):
    """
    Recursively converts an XML element and its children into a dictionary.
    """
    # Convert attributes and text of the element to a dictionary
    result = {element.tag: {} if element.attrib else None}

    # Add element attributes to the dictionary
    if element.attrib:
        result[element.tag].update((key, value) for key, value in element.attrib.items())

    # Add element text to the dictionary if it exists
    if element.text and element.text.strip():
        text = element.text.strip()
        if result[element.tag]:
            result[element.tag]['text'] = text
        else:
            result[element.tag] = text

    # Convert child elements
    children = list(element)
    if children:
        child_dict = {}
        for child in children:
            child_result = xml_to_dict(child)
            if child.tag in child_dict:
                if isinstance(child_dict[child.tag], list):
                    child_dict[child.tag].append(child_result[child.tag])
                else:
                    child_dict[child.tag] = [child_dict[child.tag], child_result[child.tag]]
            else:
                child_dict.update(child_result)
        if result[element.tag]:
            result[element.tag].update(child_dict)
        else:
            result[element.tag] = child_dict

    return result


def get_deviceset_element(schematic_library, library_name, deviceset_name):
    """
    Extract the entire element of the deviceset from the schematic library.

    Args:
        schematic_library (list): List of libraries from the schematic XML.
        library_name (str): Name of the library containing the deviceset.
        deviceset_name (str): Name of the deviceset to extract.

    Returns:
        dict: The entire element of the deviceset, or None if not found.
    """
    # Find the library with the specified name
    library = next((lib for lib in schematic_library if lib['name'] == library_name), None)
    if not library:
        print(f"Library '{library_name}' not found.")
        return None
    

    # Find the deviceset within the library
    devicesets = library.get('devicesets', {}).get('deviceset', [])
    if not isinstance(devicesets, list):
        devicesets = [devicesets]

    deviceset = next((ds for ds in devicesets if ds['name'] == deviceset_name), None)
    if not deviceset:
        print(f"Deviceset '{deviceset_name}' not found in library '{library_name}'.")
        return None

    return deviceset


def extract_deviceset_info(deviceset_xml, device_name):
    """
    Extract deviceset name, symbol, and package from a deviceset XML.

    Args:
        deviceset_xml (dict): The deviceset XML as a dictionary.
        device_name (str): The name of the device to extract information for.

    Returns:
        dict: A dictionary containing the deviceset name, symbol, and package.
    """
    deviceset_info = {
        'name': deviceset_xml.get('name', 'Unknown'),
        'symbols': 'Unknown',
        'packages': ['Unknown']
    }

    # Extract gates and their symbols
    gates = deviceset_xml.get('gates', {}).get('gate', [])
    if isinstance(gates, dict):
        gates = [gates]
    if gates:
        deviceset_info['symbols'] = gates[0].get('symbol', 'Unknown')

    # Extract devices and their packages
    devices = deviceset_xml.get('devices', {}).get('device', [])
    if isinstance(devices, dict):
        devices = [devices]

    # Find the device with the specified device_name
    device_info = next((device for device in devices if device.get('name') == device_name), None)
    if device_info:
        deviceset_info['packages'] = device_info.get('package', 'Unknown')

    return deviceset_info


def map_part_to_deviceset(schematic_parts, schematic_library, schematic_instance):
    """
    Maps parts to their corresponding deviceset names, libraries, and additional attributes,
    including a list of instances with gate-specific information.

    Args:
        schematic_parts (list): List of parts containing deviceset, library, and value information.
        schematic_instance (list): List of instances containing part names and other attributes.

    Returns:
        dict: A dictionary where keys are part names and values are dictionaries containing deviceset, library,
              device, value, symbol, package, and a list of instances with gate-specific information.
    """
    # Create a mapping of part names to deviceset names, libraries, devices, and values
    # part_to_deviceset_and_library = {
    #     part['name']: {
    #         'deviceset': part['deviceset'],
    #         'library': part.get('library', 'Unknown'),
    #         'device': part.get('device', 'Unknown'),  # Include device information
    #         'value': '' if 'PowerSymbols' in part.get('library', 'Unknown') else part.get('value', ''),  # Conditionally include value information
    #         'package': 'Unknown',  # Initialize package information
    #         'instances': []  # Initialize an empty list for instances
    #     }
    #     for part in schematic_parts
    # }
    part_to_deviceset_and_library = {}

    for part in schematic_parts:
        part_name = part['name']
        part_info = {
            'deviceset': part['deviceset'],
            'library': part.get('library', 'Unknown'),
            'device': part.get('device', 'Unknown'),  # Include device information
            # 'value': '' if 'PowerSymbols' in part.get('library', 'Unknown') else part.get('value', ''),  # Conditionally include value information
            'value': part.get('value', part['deviceset']+part.get('device', 'Unknown')),  # Include value information
            'package': 'Unknown',  # Initialize package information
            'instances': []  # Initialize an empty list for instances
        }


        # Optional attributes: add only if they exist
        optional_keys = ['library_urn', 'package3d_urn']
        for key in optional_keys:
            if key in part:
                part_info[key] = part[key]

        part_to_deviceset_and_library[part_name] = part_info
        
    # Map instances to their corresponding parts and add gate-specific information
    for instance in schematic_instance:
        
        part_name = instance['part']
        if part_name in part_to_deviceset_and_library:
            library_name = part_to_deviceset_and_library[part_name]['library']
            deviceset_name = part_to_deviceset_and_library[part_name]['deviceset']
            device_name = part_to_deviceset_and_library[part_name]['device']
            deviceset_element = get_deviceset_element(schematic_library, library_name, deviceset_name)

            # Find the symbol for the instance based on the gate name
            gate_name = instance.get('gate', 'Unknown')
            if deviceset_element is None:
                continue
            gates = deviceset_element.get('gates', {}).get('gate', [])
            if isinstance(gates, dict):
                gates = [gates]
            symbol = next((gate.get('symbol', 'Unknown') for gate in gates if gate.get('name') == gate_name), 'Unknown')

            instance_data = {
                'gate': gate_name,
                'x': instance.get('x', 'Unknown'),
                'y': instance.get('y', 'Unknown'),
                'rot': instance.get('rot', 'R0'),
                'symbol': symbol,
                "attribute": instance.get('attribute', [])
            }
            
            part_to_deviceset_and_library[part_name]['instances'].append(instance_data)

            # Update package information if not already set
            if part_to_deviceset_and_library[part_name]['package'] == 'Unknown':
                deviceset_info = extract_deviceset_info(deviceset_element, device_name)
                part_to_deviceset_and_library[part_name]['package'] = deviceset_info['packages']

    # for part in part_to_deviceset_and_library:
    #     print("part: ",part, part_to_deviceset_and_library[part])
        # print("--------------------------------------------------")

    return part_to_deviceset_and_library


def get_symbol_element_of_instance_from_library(library_xml, part_name, instance_gate_name, part_library):
    """
    Extract the symbol element from the library XML based on the part name and instance gate name.

    Args:
        library_xml (list): List of libraries from the schematic XML.
        part_name (str): Name of the part to extract the symbol for.
        instance_gate_name (str): Name of the gate to match the instance.
        part_library (dict): Dictionary containing part names mapped to their library, symbol, and other details.

    Returns:
        dict: The symbol element, or None if not found.
    """
    # Find the library name and symbol using the part name from the part library
    part_info = part_library.get(part_name)
    if not part_info:
        print(f"Part '{part_name}' not found in part library.")
        return None

    library_name = part_info.get('library')
    instances = part_info.get('instances', [])
    if not library_name or not instances:
        print(f"Library or instances not found for part '{part_name}'.")
        return None

    # Find the library with the specified name
    library = next((lib for lib in library_xml if lib['name'] == library_name), None)
    if not library:
        print(f"Library '{library_name}' not found.")
        return None

    # Extract symbols for the instance matching the specified gate name
    for instance in instances:
        
        if instance.get('gate') != instance_gate_name:
            continue
        

        symbol_name = instance.get('symbol')
        if not symbol_name:
            print(f"Symbol not found for instance in part '{part_name}'.")
            continue

        # Find the symbol in the library
        library_symbols = library.get('symbols', {}).get('symbol', [])
        if not isinstance(library_symbols, list):
            library_symbols = [library_symbols]

        symbol_element = next((sym for sym in library_symbols if sym['name'] == symbol_name), None)
        if not symbol_element:
            print(f"Symbol '{symbol_name}' not found in library '{library_name}'.")
            continue


        
        
        # print("instance:",instance)
        # print("------------------------")
        # Convert the symbol element into the desired dictionary format
        symbol_data = {
            'name': symbol_element.get('name', 'Unknown'),
            'wire': symbol_element.get('wire', []) if isinstance(symbol_element.get('wire', []), list) else [symbol_element.get('wire', [])],
            'text': instance.get('attribute', []) if isinstance(instance.get('attribute', []), list) else [instance.get('attribute', [])],
            'pin': symbol_element.get('pin', []) if isinstance(symbol_element.get('pin', []), list) else [symbol_element.get('pin', [])],
            'rectangle': symbol_element.get('rectangle', []) if isinstance(symbol_element.get('rectangle', []), list) else [symbol_element.get('rectangle', [])],
            'circle': symbol_element.get('circle', []) if isinstance(symbol_element.get('circle', []), list) else [symbol_element.get('circle', [])],
            'polygon': symbol_element.get('polygon', []) if isinstance(symbol_element.get('polygon', []), list) else [symbol_element.get('polygon', [])],
            'attribute': instance.get('attribute', []),
            'symbol_text': symbol_element.get('text',[]) if isinstance(symbol_element.get('text',[]), list) else [symbol_element.get('text',[])]
            
        }

        # print('symbol_data:', symbol_data)
        # if (symbol_element.get('name', 'Unknown') == "I2C_STANDARD"):
        #     import json
        #     with open("example_symbol_element.json", "w") as f:
        #                 json.dump(symbol_data, f, indent=4)
        # print("symbol_data:",symbol_data)
        # print("--------------------------------------------------")
        return symbol_data
    
    print(f"No matching symbol found for gate '{instance_gate_name}' in part '{part_name}'.")
    return None



import json

def find_connects(data, library_name, deviceset_name, device_name=""):
    """
    Extracts the 'connects' field from the specified device in the deviceset.

    Args:
        data (dict): The full JSON-parsed data structure.
        library_name (str): The name of the library (e.g. "40xx").
        deviceset_name (str): The name of the deviceset (e.g. "4066").
        device_name (str): The name of the specific device (e.g. "", "D", "N").

    Returns:
        list of dicts: The list of 'connect' entries, or None if not found.
    """
    for library in data.get("IC_library", []):
        if library.get("name") == library_name:
            devicesets = library.get("devicesets", {}).get("deviceset", {})
            if isinstance(devicesets, list):
                for ds in devicesets:
                    if ds.get("name") == deviceset_name:
                        # import json
                        # with open("text.json", "w") as f:
                        #     json.dump(ds, f, indent=2)
                        devices = ds.get("devices", {}).get("device")
                        if isinstance(devices, dict):
                            devices = [devices]
                        for device in devices:
                            if device.get("name") == device_name:
                                return device.get("connects", {}).get("connect", [])
            elif devicesets.get("name") == deviceset_name:
                devices = devicesets.get("devices", {}).get("device", [])
                if isinstance(devices, dict):
                    devices = [devices]
                for device in devices:
                    if device.get("name") == device_name:
                        return device.get("connects", {}).get("connect", [])
    return None

  
def retrieve_net_info(schematic_net):
    """
    Extract net information from the schematic net data.

    Args:
        schematic_net (list): List of nets from the schematic XML.

    Returns:
        dict: A dictionary where keys are net names and values are lists of segments,
              with each segment containing lists of pinrefs, wires, labels, and junctions.
    """
    net_info = {}

    # Ensure schematic_net is a list
    if not isinstance(schematic_net, list):
        schematic_net = [schematic_net]

    for net in schematic_net:
        net_name = net.get('name', 'Unknown')
        segments = net.get('segment', [])
        if not isinstance(segments, list):
            segments = [segments]

        # Process each segment
        processed_segments = []
        for segment in segments:
            processed_segment = {
                'pinref': segment.get('pinref', []) if isinstance(segment.get('pinref', []), list) else [segment.get('pinref', [])],
                'wire': segment.get('wire', []) if isinstance(segment.get('wire', []), list) else [segment.get('wire', [])],
                'label': segment.get('label', []) if isinstance(segment.get('label', []), list) else [segment.get('label', [])],
                'junction': segment.get('junction', []) if isinstance(segment.get('junction', []), list) else [segment.get('junction', [])]
            }
            processed_segments.append(processed_segment)

        net_info[net_name] = processed_segments

    return net_info



def process_schematic_file(file_path):
    """
    Process the board file and extract relevant information.

    Args:
        file_path (str): Path to the board file.

    Returns:
        dict: A dictionary containing board information.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    xml_dict = xml_to_dict(root)



    check_sheet = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", [])
    if isinstance(check_sheet, list):
        raise ValueError(f"Multiple sheets found: {len(check_sheet)}")
    
    
    check_busses = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get('sheet', {}).get('busses', {})
    if check_busses:
        raise ValueError("Busses found")
    

    # schematic_library = xml_dict['eagle']['drawing']['schematic']['libraries']['library']
    # schematic_parts = xml_dict['eagle']['drawing']['schematic']['parts']["part"]
    # schematic_instance = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['instances']['instance']
    # schematic_net = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['nets']['net']

        # Handle cases where instances, parts, or nets might not exist
    schematic_library = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('libraries', {})
    if schematic_library is None:
        schematic_library = []
    else:
        schematic_library = schematic_library.get("library", []) or []
        if isinstance(schematic_library, dict):
            schematic_library = [schematic_library]

    schematic_parts = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('parts', {})
    if schematic_parts is None:
        schematic_parts = []
    else:
        schematic_parts = schematic_parts.get("part", []) or []
        if isinstance(schematic_parts, dict):
            schematic_parts = [schematic_parts]


        

    schematic_instance = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", {}).get('instances', {})
    

    if schematic_instance is None:
        schematic_instance = []
    else:
        schematic_instance = schematic_instance.get("instance", []) or []
        if isinstance(schematic_instance, dict):
            schematic_instance = [schematic_instance]
    
    schematic_net = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", {}).get('nets', {})
    if schematic_net is None:
        schematic_net = []
    else:
        schematic_net = schematic_net.get("net", []) or []


    schematic_sheets = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get('sheet', {}).get('plain', {})
    schematic_setting = xml_dict.get('eagle', {}).get('drawing', {}).get('grid', {})

    # print("schematic_library:", schematic_library)
    # print("schematic_parts:", schematic_parts)
    # print("schematic_instance:", schematic_instance)
    # print("schematic_net:", schematic_net)



    schematic_info = {}
    schematic_info['IC_library'] = schematic_library
    schematic_info['parts'] = map_part_to_deviceset(schematic_parts, schematic_library, schematic_instance)
    schematic_info['nets'] = retrieve_net_info(schematic_net) if schematic_net else {}
    schematic_info['SheetInfo'] = schematic_sheets if schematic_sheets else {}
    schematic_info['setting'] = schematic_setting
    # for net_name, segments in schematic_info['nets'].items():
    #     print("Net name:", net_name)
    #     print("Segments:")
    #     for segment in segments:
    #         print(segment)
    
    return schematic_info



# Example

In [ ]:
if __name__ == "__main__":
    # <pinref part="C2" gate="G$1" pin="1"/>
    # <wire x1="53.34" y1="220.98" x2="53.34" y2="210.82" width="0.1524" layer="91"/>
    # Output elements from a file according the path
    # Load and parse the XML file for the specified IC
    folder_path = r"F:\GitHub\IMG2SCH\sample\ArtemisDevKit.sch"
    # folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    # IC_name = "ICM-20948"
    # sch_file_name = f"{folder_path}/{IC_name}/{IC_name}.sch"
    sch_file_name = folder_path
    # print("schematic file name:", sch_file_name)
    # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"
    # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\temp_add.sch"
    sch_file = "F:\GitHub\IMG2SCH\sample\Arduino-Fio-v23 - 副本.sch"
    # sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\train\sch\weather-bit#SparkFun-Connectors#MICRO_BIT.sch"
    # sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\yolo_training_24\dataset\val\sch\TLC5940-Breakout-v12.sch"
    sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\data\full sch\Arduino_Nano_Every.sch"

    schematic_info = process_schematic_file(sch_file)
    

In [ ]:
schematic_info['parts']

# draw schematic 

### Draw Sheet
*Layers: 97 (Info), 94 (Symbols)*

In [ ]:
from matplotlib.patches import Arc

def draw_sheet_wire(sheet_wire, alpha=1.0, ax=None):
    """
    Draw a wire (line segment) on the given axes.
    """
    # print("sheet_wire:", sheet_wire)
    x1 = float(sheet_wire['x1'])
    y1 = float(sheet_wire['y1'])
    x2 = float(sheet_wire['x2'])
    y2 = float(sheet_wire['y2'])
    

    layer = sheet_wire.get('layer', '-1')  # Default to layer '-1' if not specified

    color = 'black'
    if layer == '97':
        color = 'gray'
    elif layer == '94':
        color = 'red'
    elif layer == '91':
        color = 'green'

    style = sheet_wire.get('style', 'continuous')
    linestyle = '-'
    if style == 'longdash':
        linestyle = (0, (10, 5))  # long dashes
    elif style == 'shortdash':
        linestyle = (0, (4, 2))   # shorter dashes
    elif style == 'dashdot':
        linestyle = (0, (6, 3, 1, 3))  # dash-dot pattern
    else:
        linestyle = '-'  # continuous

    c = sheet_wire.get('curve', 0)
    curve_deg = float(c)

    if abs(curve_deg) < 1e-9:
        plt.plot([x1, x2], [y1, y2], color = color, linestyle = linestyle)
        return
    chord_len = math.hypot(x2 - x1, y2 - y1) # use original points
    
    if chord_len < 1e-9:
        # Degenerate case: start == end
        return
    
    theta = math.radians(abs(curve_deg))
    R = chord_len / (2.0 * math.sin(theta / 2.0))
    mx_orig = (x1 + x2) / 2.0
    my_orig = (y1 + y2) / 2.0
    d = math.sqrt(R*R - (chord_len / 2.0)**2)
    vx_orig = x2 - x1
    vy_orig = y2 - y1
    nx_orig = -vy_orig
    ny_orig =  vx_orig
    length_n_orig = math.hypot(nx_orig, ny_orig)
    nx_orig /= length_n_orig
    ny_orig /= length_n_orig


    if curve_deg > 0:
        cx_orig = mx_orig + d * nx_orig
        cy_orig = my_orig + d * ny_orig
    else:
        cx_orig = mx_orig - d * nx_orig
        cy_orig = my_orig - d * ny_orig

    cx = cx_orig
    cy = cy_orig

    start_angle = math.degrees(math.atan2(y1 - cy, x1 - cx))
    end_angle   = math.degrees(math.atan2(y2 - cy, x2 - cx))


    def normalize_angle(a):
        return a % 360

    start_angle = normalize_angle(start_angle)
    end_angle   = normalize_angle(end_angle)



    if curve_deg > 0:
        diff = end_angle - start_angle
        if diff < 0:
            diff += 360
        if abs(diff - curve_deg) > 1e-3:
            if diff < curve_deg:
                end_angle += 360
            else:
                end_angle -= 360
    else:
        temp = start_angle
        start_angle = end_angle
        end_angle   = temp

    arc = Arc(
        (cx, cy),           # center
        2*R, 2*R,           # width, height
        angle=0,            # rotation of the whole arc ellipse (0 for a circle)
        theta1=start_angle, # start angle in degrees
        theta2=end_angle,   # end angle in degrees
        edgecolor=color,
        facecolor = color,
        linestyle=linestyle, 
    )


    if ax is None:
        ax = plt.gca()
    # ax.plot([x1, x2], [y1, y2], color=color, alpha=alpha, linestyle=linestyle)
    ax.add_patch(arc)
def get_text_alignment(align, rot_angle_deg, is_mirrored):
    """
    Returns horizontal and vertical alignment for text based on align type,
    rotation angle, and optional mirroring.

    Parameters
    ----------
    align : str
        The alignment keyword (e.g., 'bottom-center', 'top-left').
    rot_angle_deg : int
        Rotation angle (0, 90, 180, 270).
    is_mirrored : bool, optional
        Whether the text is mirrored horizontally.

    Returns
    -------
    (ha, va) : tuple
        Horizontal alignment ('left', 'center', 'right')
        and vertical alignment ('top', 'center', 'bottom').
    """

    base_align_map = {
        0:   ('left',  'bottom'),
        90:  ('right', 'bottom'),
        180: ('right', 'top'),
        270: ('left',  'top'),
    }

    align_maps = {
        'bottom-center': {
            0:   ('center', 'bottom'),
            90:  ('right',  'center'),
            180: ('center', 'top'),
            270: ('left',   'center'),
        },
        'bottom-right': {
            0:   ('right', 'bottom'),
            90:  ('right', 'top'),
            180: ('left',  'top'),
            270: ('left',  'bottom'),
        },
        'center-left': {
            0:   ('left',   'center'),
            90:  ('center', 'bottom'),
            180: ('right',  'center'),
            270: ('center', 'top'),
        },
        'center': {
            0:   ('center', 'center'),
            90:  ('center', 'center'),
            180: ('center', 'center'),
            270: ('center', 'center'),
        },
        'center-right': {
            0:   ('right',  'center'),
            90:  ('center', 'top'),
            180: ('left',   'center'),
            270: ('center', 'bottom'),
        },
        'top-left': {
            0:   ('left',  'top'),
            90:  ('left', 'bottom'),
            180: ('right', 'bottom'),
            270: ('right',  'top'),
        },
        'top-center': {
            0:   ('center', 'top'),
            90:  ('left',   'center'),
            180: ('center', 'bottom'),
            270: ('right',  'center'),
        },
        'top-right': {
            0:   ('right', 'top'),
            90:  ('left', 'top'),
            180: ('left',  'bottom'),
            270: ('right', 'bottom'),
        }
    }

    # Default to base map
    ha, va = base_align_map.get(rot_angle_deg % 360, ('center', 'center'))

    # Override if align is in align_maps
    if align in align_maps:
        ha, va = align_maps[align].get(rot_angle_deg % 360, ('center', 'center'))

    # Handle mirrored flip
    exceptions = {
        ('center', 'center'),
        ('center-right', 'center'),
        ('center-left', 'center'),
        ('bottom-center', 'center'),
        ('top-center', 'center'),
    }
    if is_mirrored and (align, ha) not in exceptions:
        ha = 'right' if ha == 'left' else 'left'

    return ha, va


def draw_sheet_text(sheet_text, alpha=1.0, ax=None):


    x = float(sheet_text.get('x', 0))
    y = float(sheet_text.get('y', 0))
    size = float(sheet_text.get('size', 1.0)) * 10
    text = sheet_text.get('text', '')
    layer = sheet_text.get('layer', '-1')
    text_rot = sheet_text.get('rot', 'R0')
    align = sheet_text.get('align', 'bottom-left')
    
    # Color selection
    color = 'black'
    if layer == '97':
        color = 'gray'
    elif layer == '94':
        color = 'red'
    elif layer == '98':
        color = '#d7d769'
    elif layer == '91':
        color = 'green'

    # Rotation and mirroring
    rot_angle_deg = 0.0
    is_mirrored = False
    if text_rot.upper().startswith('R'):
        rot_angle_deg = float(text_rot[1:])
    elif text_rot.upper().startswith('MR'):
        rot_angle_deg = float(text_rot[2:])
        is_mirrored = True

    ha, va = get_text_alignment(align, rot_angle_deg, is_mirrored)
    
    # Keep text upright where needed
    if rot_angle_deg == 180:
        rot_angle_final = 0   # keep upright
    elif rot_angle_deg == 270:
        rot_angle_final = 90
    else:
        rot_angle_final = rot_angle_deg

    if ax is None:
        ax = plt.gca()
    ax.text(x, y, text,
            color=color, fontsize=size,
            rotation=rot_angle_final,
            ha=ha, va=va, alpha=alpha)

    # Draw cross
    cross_size = 0.3
    ax.plot([x - cross_size, x + cross_size], [y, y],
            color=color, linewidth=1, zorder=0)
    ax.plot([x, x], [y + cross_size, y - cross_size],
            color=color, linewidth=1, zorder=0)


def visualize_sheet_texts(sheetinfo_texts,ax=None):
    if not isinstance(sheetinfo_texts, list):
        sheetinfo_texts = [sheetinfo_texts]

    for text in sheetinfo_texts:
        draw_sheet_text(text, alpha=1.0,ax=ax)


def visualize_sheet_wires(sheetinfo_wires,ax=None):
    if not isinstance(sheetinfo_wires, list):
        sheetinfo_wires = [sheetinfo_wires]
        
    for wire in sheetinfo_wires:
        # print("dummy")
        draw_sheet_wire(wire, alpha=0.5,ax=ax)

### Draw Setting-Grid

In [ ]:
def visualize_grid_from_setting(setting, ax=None, draw_grid=True):
    """
    Draws a grid on the given matplotlib axes based on the provided setting dictionary.
    Supports 'lines' or 'dots' styles.
    """
    if setting.get('display', 'yes').lower() != 'yes' or not draw_grid:
        return  # Do not draw grid if display is not 'yes'

    if ax is None:
        ax = plt.gca()

    distance = float(setting.get('distance', 5))
    grid_style = setting.get('style', 'lines').lower()
    multiple = int(setting.get('multiple', 1))

    # Grid spacing
    grid_spacing = distance * multiple

    # Get current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    if grid_style == 'lines':
        # Draw vertical grid lines
        for x in np.arange(xlim[0], xlim[1] + grid_spacing, grid_spacing):
            ax.axvline(x, color='dimgray', linestyle='-', linewidth=0.5, zorder=0)
        # Draw horizontal grid lines
        for y in np.arange(ylim[0], ylim[1] + grid_spacing, grid_spacing):
            ax.axhline(y, color='dimgray', linestyle='-', linewidth=0.5, zorder=0)

    elif grid_style == 'dots':
        # Create grid of points
        xs = np.arange(xlim[0], xlim[1] + grid_spacing, grid_spacing)
        ys = np.arange(ylim[0], ylim[1] + grid_spacing, grid_spacing)
        X, Y = np.meshgrid(xs, ys)
        ax.scatter(X, Y, color='dimgray', s=5, marker='.', zorder=0)

    else:
        print(f"Unknown grid style '{grid_style}', skipping grid.")


### Bouding box Symbol Comp.

In [ ]:
def draw_anchor_box(ax, x, y, ha='center', va='center', width=10, height=5,
                    edgecolor='green', facecolor='none', alpha=1.0, linewidth=2,
                    rot=0, padding=1.0):
    """
    Draw a rectangle with padding. (x, y) is the anchor inside the original box.
    Gray box = original, Green box = padded.

    Parameters
    ----------
    x, y : float
        Anchor point coordinates.
    ha, va : str
        Horizontal/vertical alignment of the anchor in the box.
    width, height : float
        Original box dimensions.
    rot : int
        Rotation angle in degrees; swaps width/height if 90 or 270.
    padding : float
        Extra margin added OUTSIDE the original box.
    show_gray : bool
        Whether to draw the original box in gray for reference.
    """

    rot = rot % 360
    if rot in (90, 270):
        width, height = height, width

    # Horizontal alignment offset
    if ha == 'left':
        offset_x = 0
    elif ha == 'center':
        offset_x = -width / 2
    elif ha == 'right':
        offset_x = -width
    else:
        raise ValueError("ha must be 'left', 'center', or 'right'")

    # Vertical alignment offset
    if va == 'bottom':
        offset_y = 0
    elif va == 'center':
        offset_y = -height / 2
    elif va == 'top':
        offset_y = -height
    else:
        raise ValueError("va must be 'bottom', 'center', or 'top'")

    # Lower-left corner of original box
    lower_left_x = x + offset_x
    lower_left_y = y + offset_y


    # Expand with padding
    lower_left_x -= padding
    lower_left_y -= padding
    width_padded = width + 2 * padding
    height_padded = height + 2 * padding

    # Draw padded green box
    rect_green = patches.Rectangle(
        (lower_left_x, lower_left_y),
        width_padded, height_padded,
        linewidth=linewidth,
        edgecolor=edgecolor,
        facecolor=facecolor,
        alpha=alpha
    )
    ax.add_patch(rect_green)


    return rect_green



def get_text_corners_helper(x, y, ha='center', va='center', width=10, height=5,
                     rot=0, padding=1.0):
    """
    Compute the four corners of a text bounding box with padding.

    Parameters
    ----------
    x, y : float
        Anchor point coordinates.
    ha, va : str
        Horizontal/vertical alignment of the anchor in the box.
    width, height : float
        Original box dimensions.
    rot : int
        Rotation angle in degrees; swaps width/height if 90 or 270.
    padding : float
        Extra margin added OUTSIDE the original box.

    Returns
    -------
    xs, ys : list
        Lists of x and y coordinates of the four corners (in order).
    """

    rot = rot % 360
    if rot in (90, 270):
        width, height = height, width

    # Horizontal alignment offset
    if ha == 'left':
        offset_x = 0
    elif ha == 'center':
        offset_x = -width / 2
    elif ha == 'right':
        offset_x = -width
    else:
        raise ValueError("ha must be 'left', 'center', or 'right'")

    # Vertical alignment offset
    if va == 'bottom':
        offset_y = 0
    elif va == 'center':
        offset_y = -height / 2
    elif va == 'top':
        offset_y = -height
    else:
        raise ValueError("va must be 'bottom', 'center', or 'top'")

    # Lower-left corner of original box
    lower_left_x = x + offset_x
    lower_left_y = y + offset_y

    # Expand with padding
    lower_left_x -= padding
    lower_left_y -= padding
    width_padded = width + 2 * padding
    height_padded = height + 2 * padding

    # Define corners
    corners = np.array([
        [lower_left_x, lower_left_y],
        [lower_left_x + width_padded, lower_left_y],
        [lower_left_x + width_padded, lower_left_y + height_padded],
        [lower_left_x, lower_left_y + height_padded]
    ])

    xs, ys = corners[:, 0].tolist(), corners[:, 1].tolist()
    return xs, ys



def get_text_corners(attr_x, attr_y, align, rot_deg, is_mirrored, text_width, text_height,ax = None):
    """
    Compute bounding box corners of rotated and mirrored text.
    ha and va are used as anchor positions inside the bounding box.

    Parameters
    ----------
    attr_x, attr_y : float
        Anchor position (center of rotation).
    align : str
        Text alignment string (e.g., 'bottom-left', 'top-center', etc.).
    rot_deg : int
        Rotation angle in degrees.
    is_mirrored : bool
        Whether the text is mirrored horizontally.
    text_width, text_height : float
        Size of the text box.

    Returns
    -------
    xs, ys : list of float
        Rotated and transformed bounding box corner coordinates.
    """
    # Draw cross at center
    # cross_size = min(2, 2) * 0.1
    # ax.plot([attr_x - cross_size, attr_x + cross_size], [attr_y, attr_y], color="blue", linewidth=3, alpha=0.2)
    # ax.plot([attr_x, attr_x], [attr_y - cross_size, attr_y + cross_size], color="blue", linewidth=3, alpha=0.2)

    rot_deg = rot_deg % 360
    align = align.lower()

    # # Map alignment to anchor position inside the bounding box
    # ha_va_ratio_map = {
    #     'left':   0.0,
    #     'center': 0.5,
    #     'right':  1.0,
    #     'bottom': 0.0,
    #     'top':    1.0
    # }

    align_maps = {
        'bottom-left':  {0: ('left', 'bottom'), 90: ('right', 'bottom'), 180: ('right', 'top'), 270: ('left', 'top')},
        'bottom-center':{0: ('center','bottom'),90: ('right','center'),180: ('center','top'),270: ('left','center')},
        'bottom-right': {0: ('right','bottom'), 90: ('right','top'),   180: ('left','top'),    270: ('left','bottom')},
        'center-left':  {0: ('left','center'),  90: ('center','bottom'),180: ('right','center'),270: ('center','top')},
        'center':       {0: ('center','center'),90: ('center','center'),180: ('center','center'),270: ('center','center')},
        'center-right': {0: ('right','center'), 90: ('center','top'),   180: ('left','center'),270: ('center','bottom')},
        'top-left':     {0: ('left','top'),     90: ('left','bottom'), 180: ('right','bottom'),270: ('right','top')},
        'top-center':   {0: ('center','top'),   90: ('left','center'), 180: ('center','bottom'),270: ('right','center')},
        'top-right':    {0: ('right','top'),    90: ('left','top'),    180: ('left','bottom'), 270: ('right','bottom')}
    }

    base_align = ('center', 'center')
    if align in align_maps:
        base_align = align_maps[align].get(rot_deg, ('center', 'center'))

    ha_str, va_str = base_align

    # Mirror ha if needed
    exceptions = {
        ('center', 'center'),
        ('center-right', 'center'),
        ('center-left', 'center'),
        ('bottom-center', 'center'),
        ('top-center', 'center'),
    }
    if is_mirrored and (align, ha_str) not in exceptions:
        ha_str = 'right' if ha_str == 'left' else 'left'

    
    # draw_anchor_box(ax, attr_x, attr_y, ha=ha_str, va=va_str,width=text_width, height=text_height,edgecolor="blue",rot=rot_deg,padding=0.5)
    
    xs, ys = get_text_corners_helper(attr_x, attr_y, ha_str, va_str, text_width, text_height, rot_deg, padding=0)

    return xs, ys


In [ ]:
def get_wire_bounding_box(wire, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute bounding box of a wire (straight or curved) with rotation and mirror.
    Returns: element_min_x, element_max_x, element_min_y, element_max_y
    """

    # Parse rotation + mirror
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    cosθ, sinθ = math.cos(theta), math.sin(theta)
    rot_matrix = [[cosθ, -sinθ], [sinθ, cosθ]]
    mirror_matrix = [[-1, 0], [0, 1]] if is_mirrored else [[1, 0], [0, 1]]

    def transform_point(px, py):
        # rotate
        rx = px * rot_matrix[0][0] + py * rot_matrix[0][1]
        ry = px * rot_matrix[1][0] + py * rot_matrix[1][1]
        # mirror
        mx = rx * mirror_matrix[0][0] + ry * mirror_matrix[0][1]
        my = rx * mirror_matrix[1][0] + ry * mirror_matrix[1][1]
        # translate
        return x + mx, y + my

    # Endpoints
    x1_orig, y1_orig = float(wire['x1']), float(wire['y1'])
    x2_orig, y2_orig = float(wire['x2']), float(wire['y2'])
    x1, y1 = transform_point(x1_orig, y1_orig)
    x2, y2 = transform_point(x2_orig, y2_orig)

    curve_deg = float(wire.get('curve', 0))

    # Straight wire
    if abs(curve_deg) < 1e-9:
        return min(x1, x2), max(x1, x2), min(y1, y2), max(y1, y2)

    # Arc case
    chord_len = math.hypot(x2_orig - x1_orig, y2_orig - y1_orig)
    if chord_len < 1e-9:
        return x1, x1, y1, y1

    theta_arc = math.radians(abs(curve_deg))
    R = chord_len / (2.0 * math.sin(theta_arc / 2.0))
    mx_orig = (x1_orig + x2_orig) / 2.0
    my_orig = (y1_orig + y2_orig) / 2.0
    d = math.sqrt(R*R - (chord_len/2.0)**2)

    vx_orig, vy_orig = x2_orig - x1_orig, y2_orig - y1_orig
    nx_orig, ny_orig = -vy_orig, vx_orig
    length_n = math.hypot(nx_orig, ny_orig)
    nx_orig, ny_orig = nx_orig/length_n, ny_orig/length_n

    if curve_deg > 0:
        cx_orig = mx_orig + d * nx_orig
        cy_orig = my_orig + d * ny_orig
    else:
        cx_orig = mx_orig - d * nx_orig
        cy_orig = my_orig - d * ny_orig

    cx, cy = transform_point(cx_orig, cy_orig)

    # Angles of endpoints
    start_angle = math.degrees(math.atan2(y1 - cy, x1 - cx)) % 360
    end_angle   = math.degrees(math.atan2(y2 - cy, x2 - cx)) % 360

    if curve_deg > 0:
        diff = end_angle - start_angle
        if diff < 0: diff += 360
        if abs(diff - curve_deg) > 1e-3:
            if diff < curve_deg:
                end_angle += 360
            else:
                end_angle -= 360
    else:
        start_angle, end_angle = end_angle, start_angle

    def point_on_circle(angle_deg):
        rad = math.radians(angle_deg)
        return cx + R * math.cos(rad), cy + R * math.sin(rad)

    # Points: start, end, and possible extrema at 0/90/180/270
    arc_points = [point_on_circle(start_angle), point_on_circle(end_angle)]
    for ang in [0, 90, 180, 270]:
        norm_ang = ang % 360
        norm_s, norm_e = start_angle % 360, end_angle % 360
        if start_angle < end_angle:
            in_range = norm_s <= norm_ang <= norm_e
        else:
            in_range = norm_ang >= norm_s or norm_ang <= norm_e
        if in_range:
            arc_points.append(point_on_circle(ang))

    xs, ys = zip(*arc_points)
    return min(xs), max(xs), min(ys), max(ys)


def get_rectangle_bounding_box(rect_data, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute the bounding box of a rectangle with rotation.

    Parameters
    ----------
    rect_data : dict
        Dictionary with 'x1','y1','x2','y2' defining opposite corners.
    x, y : float
        Translation offsets.
    rot_angle_rad : str
        Rotation string like 'R0', 'R90', 'MR0', etc.

    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    # Extract rectangle corners
    x1 = float(rect_data['x1'])
    y1 = float(rect_data['y1'])
    x2 = float(rect_data['x2'])
    y2 = float(rect_data['y2'])

    # All four corners before rotation
    corners = [
        (x1, y1),
        (x1, y2),
        (x2, y1),
        (x2, y2)
    ]

    rotated_corners = []
    for cx, cy in corners:
        # Rotate
        rx = x + (cx * math.cos(theta) - cy * math.sin(theta))
        ry = y + (cx * math.sin(theta) + cy * math.cos(theta))
        # Mirror horizontally if needed
        if is_mirrored:
            rx = 2*x - rx
        rotated_corners.append((rx, ry))

    xs, ys = zip(*rotated_corners)
    return min(xs), max(xs), min(ys), max(ys)


def get_pin_bounding_box(pin, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute bounding box for a single pin with rotation & mirroring.
    """
    # Parse rotation and mirror
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    rot_angle_rad = math.radians(rot_angle_deg)

    # Length mapping
    length_map = {
        'point': 0.0,
        'short': 2.54,
        'middle': 2.54 * 2,
        'long': 2.54 * 3
    }

    # Base
    x_base = float(pin['x'])
    y_base = float(pin['y'])

    # Rotate base
    rotated_x_base = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
    rotated_y_base = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))

    # Extension
    length_type = pin.get('length', 'long').lower()
    extension = length_map.get(length_type, 2.54)

    # Pin angle
    rot_pin_str = pin.get('rot', 'R0')
    pin_angle_deg = float(rot_pin_str[1:]) if rot_pin_str.upper().startswith('R') else float(rot_pin_str[2:])
    pin_angle_rad = math.radians(pin_angle_deg) + rot_angle_rad

    # End point
    x_end = rotated_x_base + extension * math.cos(pin_angle_rad)
    y_end = rotated_y_base + extension * math.sin(pin_angle_rad)

    # Apply mirroring consistently to both base and end
    if is_mirrored:
        rotated_x_base = 2*x - rotated_x_base
        x_end = 2*x - x_end

    # Collect points
    points = [(rotated_x_base, rotated_y_base), (x_end, y_end)]

    # Dotted pins: add a small circle radius
    if pin.get('function', '') == 'dot':
        circle_radius = 1.0
        points.extend([
            (x_end + circle_radius, y_end + circle_radius),
            (x_end - circle_radius, y_end - circle_radius)
        ])

    xs, ys = zip(*points)
    return min(xs), max(xs), min(ys), max(ys)


def get_symbol_text_bounding_box(attribute, x=0, y=0, rot_angle_rad='R0', ax=None):
    """
    Compute the bounding box for a single text attribute.
    Considers symbol rotation, text local rotation, mirroring, alignment,
    and multiline text.
    Returns: element_min_x, element_max_x, element_min_y, element_max_y
    """

    if not attribute or attribute.get('display', 'on').lower() == 'off':
        return None

    # Attribute fields
    attr_name = attribute.get('text', '')
    attr_local_x = float(attribute.get('x', 0))
    attr_local_y = float(attribute.get('y', 0))
    attr_size = float(attribute.get('size', 1.0))
    attr_align = str(attribute.get('align', 'bottom-left')).lower()
    attr_rot_str = str(attribute.get('rot', 'R0')).upper()

    # Parse local text rotation
    local_mirror = False
    local_rot_deg = 0.0
    if attr_rot_str.startswith('MR'):
        local_mirror = True
        local_rot_deg = float(attr_rot_str[2:]) if len(attr_rot_str) > 2 else 0.0
    elif attr_rot_str.startswith('R'):
        local_rot_deg = float(attr_rot_str[1:]) if len(attr_rot_str) > 1 else 0.0

    # Parse symbol rotation
    is_mirrored = False
    symbol_rot_deg = 0.0
    rot_angle_str = str(rot_angle_rad).upper()
    if rot_angle_str.startswith('MR'):
        is_mirrored = True
        symbol_rot_deg = float(rot_angle_str[2:]) if len(rot_angle_str) > 2 else 0.0
    elif rot_angle_str.startswith('R'):
        symbol_rot_deg = float(rot_angle_str[1:]) if len(rot_angle_str) > 1 else 0.0

    # Combined mirror: XOR of symbol and local
    final_mirror = is_mirrored ^ local_mirror
    final_rot_deg = (symbol_rot_deg + local_rot_deg) % 360

    theta = math.radians(symbol_rot_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)

    # Rotate + mirror local text coordinates into global
    rx = attr_local_x * cos_t - attr_local_y * sin_t
    ry = attr_local_x * sin_t + attr_local_y * cos_t
    if is_mirrored:
        rx = -rx
    attr_global_x = x + rx
    attr_global_y = y + ry

    # Handle multiline text
    lines = attr_name.splitlines()
    num_lines = len(lines)
    max_line_len = max((len(line) for line in lines), default=0)

    # Estimate text dimensions
    fontsize = attr_size * 10
    text_width = int(max_line_len) * fontsize * 0.6 / 10.0
    text_height = int(num_lines) * fontsize * 1.2 / 10.0

    # Compute bounding box corners
    xs, ys = get_text_corners(
        attr_global_x, attr_global_y,
        attr_align, final_rot_deg, final_mirror,
        text_width, text_height, ax
    )

    return min(xs), max(xs), min(ys), max(ys)

def get_attribute_text_bounding_box(attribute, part_name="Unknown", part_value="Unknown", symbol_name="Unknown",ax=None):
    """
    Compute bounding box for a single attribute text with rotation and mirroring.
    Handles tricky cases like rot=90 with bottom-left align.
    Returns (min_x, max_x, min_y, max_y).
    """

    if not attribute or attribute.get('display', 'on').lower() == 'off':
        return None

    # Attribute fields
    attr_name = attribute.get('name', '')
    attr_x = float(attribute.get('x', 0))
    attr_y = float(attribute.get('y', 0))
    attr_size = float(attribute.get('size', 1.0))
    attr_rot = str(attribute.get('rot', 'R0')).upper()
    attr_align = str(attribute.get('align', 'bottom-left')).lower()

    # Mirroring and rotation
    is_mirrored = attr_rot.startswith('MR')
    rot_val = attr_rot[2:] if is_mirrored else attr_rot[1:]
    try:
        rot_deg = int(rot_val)
    except ValueError:
        rot_deg = 0
    rot_deg = rot_deg % 360

    # Decide text to draw
    if attr_name == "NAME" and part_name != "Unknown":
        display_text = part_name
    elif attr_name == "VALUE":
        display_text = part_value if part_value not in ["Unknown", ""] else symbol_name
        if 'GND' in part_name.upper():
            display_text = re.sub(r'\d+', '', part_name)
    else:
        if symbol_name.upper() == "GND":
            return None
        display_text = attr_name

    if not display_text:
        return None

    # Estimate unrotated text dimensions
    fontsize = attr_size * 10
    text_width = len(display_text) * fontsize * 0.6 / 10.0
    text_height = fontsize * 1.2 / 10.0

    xs, ys = get_text_corners(attr_x, attr_y, attr_align, rot_deg, is_mirrored, text_width, text_height,ax)
    
    return min(xs), max(xs), min(ys), max(ys)

def get_circle_bounding_box(circle_data, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute the bounding box of a circle with rotation and mirroring.

    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        rot_angle_deg = float(rot_angle_rad[2:])
        is_mirrored = True
    rot_angle_rad = math.radians(rot_angle_deg)

    # Extract circle parameters
    x_center = float(circle_data['x'])
    y_center = float(circle_data['y'])
    radius   = float(circle_data['radius'])

    # Rotate and mirror center
    rotated_x_center = x + (x_center * math.cos(rot_angle_rad) - y_center * math.sin(rot_angle_rad))
    rotated_y_center = y + (x_center * math.sin(rot_angle_rad) + y_center * math.cos(rot_angle_rad))

    if is_mirrored:
        rotated_x_center = 2*x - rotated_x_center

    # Bounding box
    element_min_x = rotated_x_center - radius
    element_max_x = rotated_x_center + radius
    element_min_y = rotated_y_center - radius
    element_max_y = rotated_y_center + radius

    return element_min_x, element_max_x, element_min_y, element_max_y

def get_polygon_bounding_box(polygons, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute the bounding box for a list of polygons with rotation and mirroring.
    
    Parameters
    ----------
    polygons : list
        Each dict has 'vertex': list of {'x': str, 'y': str}
    x, y : float
        Anchor position for the symbol instance.
    rot_angle_rad : str
        Rotation string like 'R0', 'R90', 'MR0', 'MR90'.
        
    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y : float
        The bounding box coordinates of the polygon set.
    """

    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    cos_t, sin_t = math.cos(theta), math.sin(theta)

    all_x, all_y = [], []


    vertices = polygons.get('vertex', [])
    for v in vertices:
        px = float(v['x'])
        py = float(v['y'])

        # Apply rotation
        rx = px * cos_t - py * sin_t
        ry = px * sin_t + py * cos_t

        # Apply mirroring across vertical axis
        if is_mirrored:
            rx = -rx

        # Translate
        rotated_x = x + rx
        rotated_y = y + ry

        all_x.append(rotated_x)
        all_y.append(rotated_y)

    if not all_x or not all_y:
        return None  # no vertices

    return min(all_x), max(all_x), min(all_y), max(all_y)




### Bounding box Symbol

In [ ]:
def draw_one_box(x_center, y_center, length, width, ax, color='orange', thickness=1.5):
    x_min = x_center - length / 2 
    y_min = y_center - width / 2 

    rect = Rectangle(
        (x_min, y_min), length, width,
        linewidth=thickness, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)

    # Draw cross at center
    cross_size = min(length, width) * 0.1
    ax.plot([x_center - cross_size, x_center + cross_size], [y_center, y_center], color=color, linewidth=1.2)
    ax.plot([x_center, x_center], [y_center - cross_size, y_center + cross_size], color=color, linewidth=1.2)


def get_bounding_box_for_symbol_instance2(symbol_element, instance_x, instance_y, rot="R0", part_name="Unknown", part_value="Unknown",symbol_name="Unknown",space=10,ax=None):
    
    """
    Calculates the bounding box (xlim, ylim) for a single symbol instance.
    (No changes in this function)
    """
    # print("get_bounding_box_for_symbol_instance2")

        
    min_x_symbol = float('inf')
    max_x_symbol = float('-inf')
    min_y_symbol = float('inf')
    max_y_symbol = float('-inf')

    rot_angle_deg = 0.0
    if rot.upper().startswith('R180'):
        rot_angle_deg = 0
    elif rot.upper().startswith('R270'):
        rot_angle_deg = 90
    elif rot.upper().startswith('R'):
        rot_angle_deg = float(rot[1:])
    elif rot.upper().startswith('MR'):
        rot_angle_deg = 360 - float(rot[2:])
    rot_angle_rad = math.radians(rot_angle_deg)
        

    elements_list = []
    for key in ['pin', 'wire', 'rectangle', 'circle', 'polygon', 'text', 'symbol_text']:
        for element in symbol_element.get(key, []):
            element_with_type = element.copy()
            element_with_type['element_type'] = key
            elements_list.append([element_with_type])

    # return elements_list

    # print("elements_list:", elements_list)
    for elements in elements_list:
        # print("elements:", elements)
        for element in elements:
            # print("element:", element)
            element_type = element.get('element_type', 'unknown')

            # print(element_type," element:", element)
            element_min_x = float('inf')
            element_max_x = float('-inf')
            element_min_y = float('inf')
            element_max_y = float('-inf')

            if element_type == 'pin':
                element_min_x, element_max_x, element_min_y, element_max_y = get_pin_bounding_box(element, instance_x, instance_y, rot)

            elif element_type == 'wire':
                element_min_x, element_max_x, element_min_y, element_max_y = get_wire_bounding_box(element, instance_x, instance_y, rot)
            

            elif element_type == 'rectangle':
                element_min_x, element_max_x, element_min_y, element_max_y = get_rectangle_bounding_box(element, instance_x, instance_y, rot)


            elif element_type == 'circle':
                element_min_x, element_max_x, element_min_y, element_max_y = get_circle_bounding_box(element, instance_x, instance_y, rot)


            elif element_type == 'polygon':
                element_min_x, element_max_x, element_min_y, element_max_y = get_polygon_bounding_box(element, instance_x, instance_y, rot)

            
            elif (element_type == 'text' and element.get('display', 'on').lower() != 'off'):
                element_min_x, element_max_x, element_min_y, element_max_y = get_attribute_text_bounding_box(element,part_name, part_value, symbol_name,ax)


            elif (element_type == 'symbol_text' and element.get('display', 'on').lower() != 'off' and not element.get('text', '').upper().startswith('>')):
                # print("symbol_text: ",element)
                element_min_x, element_max_x, element_min_y, element_max_y = get_symbol_text_bounding_box(element, instance_x, instance_y, rot)




            min_x_symbol = min(min_x_symbol, element_min_x)
            max_x_symbol = max(max_x_symbol, element_max_x)
            min_y_symbol = min(min_y_symbol, element_min_y)
            max_y_symbol = max(max_y_symbol, element_max_y)
        

            # if element_type in ['circle']:
            #     # Draw bounding box for each element if ax is provided
            #     if element_min_x != float('inf') and element_max_x != float('-inf'):
            #         x_center = (element_min_x + element_max_x) / 2
            #         y_center = (element_min_y + element_max_y) / 2
            #         length = element_max_x - element_min_x
            #         width = element_max_y - element_min_y
            #         draw_one_box(x_center, y_center, length+1, width+1, ax, color="#CE5192", thickness=2)
            

                    
       
    # print("---------------------------------")

    if min_x_symbol == float('inf'): # No elements found
        return ((instance_x, instance_x + 200), (instance_y, instance_y + 200)) # Return a default range around instance position
    else:
        xlim = (min_x_symbol-space, max_x_symbol+space)
        ylim = (min_y_symbol-space, max_y_symbol+space)



            
        return xlim, ylim, elements_list


def draw_element_list(element_lists, draw_list,ax = None):
    for element_list in element_lists:
        instance_x = element_list['x_instance']
        instance_y = element_list['y_instance']
        rot = element_list['rot']
        ele_list = element_list['list']
        min_x_symbol = float('inf')
        max_x_symbol = float('-inf')
        min_y_symbol = float('inf')
        max_y_symbol = float('-inf')
        part_name = element_list['part_name']
        part_value = element_list['part_value']
        symbol_name = element_list['symbol_name']
        for elements in ele_list:
            # print("elements:", elements)
            for element in elements:
                # print("element:", element)
                element_type = element.get('element_type', 'unknown')

                # print(element_type," element:", element)
                element_min_x = float('inf')
                element_max_x = float('-inf')
                element_min_y = float('inf')
                element_max_y = float('-inf')

                if element_type == 'pin':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_pin_bounding_box(element, instance_x, instance_y, rot)

                elif element_type == 'wire':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_wire_bounding_box(element, instance_x, instance_y, rot)
                

                elif element_type == 'rectangle':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_rectangle_bounding_box(element, instance_x, instance_y, rot)


                elif element_type == 'circle':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_circle_bounding_box(element, instance_x, instance_y, rot)


                elif element_type == 'polygon':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_polygon_bounding_box(element, instance_x, instance_y, rot)

                
                elif (element_type == 'text' and element.get('display', 'on').lower() != 'off'):
                    element_min_x, element_max_x, element_min_y, element_max_y = get_attribute_text_bounding_box(element,part_name, part_value, symbol_name,ax)


                elif (element_type == 'symbol_text' and element.get('display', 'on').lower() != 'off' and not element.get('text', '').upper().startswith('>')):
                    # print("symbol_text: ",element)
                    element_min_x, element_max_x, element_min_y, element_max_y = get_symbol_text_bounding_box(element, instance_x, instance_y, rot)




                min_x_symbol = min(min_x_symbol, element_min_x)
                max_x_symbol = max(max_x_symbol, element_max_x)
                min_y_symbol = min(min_y_symbol, element_min_y)
                max_y_symbol = max(max_y_symbol, element_max_y)
            

                if element_type in draw_list:
                    # Draw bounding box for each element if ax is provided
                    if element_min_x != float('inf') and element_max_x != float('-inf'):
                        x_center = (element_min_x + element_max_x) / 2
                        y_center = (element_min_y + element_max_y) / 2
                        length = element_max_x - element_min_x
                        width = element_max_y - element_min_y
                        draw_one_box(x_center, y_center, length+1, width+1, ax, color="#CE5192", thickness=2)



### get bounding box for full sch

In [ ]:
def get_bounding_box_full_sch(boxes):
    if not boxes:
        return None  

    # Initialize with the first box limits
    min_x = float(boxes[0]['x_center']) - float(boxes[0]['length'])/2
    max_x = float(boxes[0]['x_center']) + float(boxes[0]['length'])/2
    min_y = float(boxes[0]['y_center']) - float(boxes[0]['width'])/2
    max_y = float(boxes[0]['y_center']) + float(boxes[0]['width'])/2
    for box in boxes[1:]:
        min_x = min(min_x, float(box['x_center']) - float(box['length'])/2)
        max_x = max(max_x, float(box['x_center']) + float(box['length'])/2)
        min_y = min(min_y, float(box['y_center']) - float(box['width'])/2)
        max_y = max(max_y, float(box['y_center']) + float(box['width'])/2)

    # Construct final bounding box
    final_box = {
        'part_name': 'SheetInfo',
        'net_name': 'final_bounding_box',
        'x_center': (min_x + max_x) / 2,
        'y_center': (min_y + max_y) / 2,
        'length': max_x - min_x,
        'width': max_y - min_y,
    }

    return [final_box]

In [ ]:

def get_list_of_symbol_bounding_boxes(schematic_info,space=5):
    """
    Returns a list of bounding box information for each symbol instance in the schematic.

    Args:
        schematic_info (dict): The schematic information dictionary containing 'parts', 'nets', and 'IC_library'.

    Returns:
        list: A list of dictionaries, each containing:
              {
                  'part_name': str,
                  'gate_instance': str,
                  'xlim': (min_x, max_x),
                  'ylim': (min_y, max_y)
              }
    """
    parts_info = schematic_info.get('parts')
    ic_library = schematic_info.get('IC_library')

    list_of_bounding_boxes = []
    element_lists = []

    # --- Calculate bounding box for parts using get_bounding_box_for_symbol_instance ---
    if parts_info:
        for part_name, part_data in parts_info.items():
            # print(f"Processing part '{part_name}': {part_data}")
            part_value = part_data.get('value', 'Unknown')
            
            for instance in part_data.get('instances', []):
                gate_instance = instance.get('gate', '') # Handle cases where gate might be missing
                symbol_element = get_symbol_element_of_instance_from_library(
                    ic_library, part_name, gate_instance, parts_info
                )
                

                symbol_name = instance['symbol']

                if symbol_element:
                    rot_instance = instance['rot']
                    x_instance = float(instance['x'])
                    y_instance = float(instance['y'])

                    symbol_instance_bbox = get_bounding_box_for_symbol_instance2(
                        symbol_element, x_instance, y_instance, rot_instance,part_name, part_value, symbol_name, space=space
                    )
                    xlim_instance, ylim_instance, element_list = symbol_instance_bbox

                    ele_list_pack = {}
                    ele_list_pack['list'] = element_list
                    
                    ele_list_pack['x_instance'] = x_instance
                    ele_list_pack['y_instance'] = y_instance
                    ele_list_pack['rot'] = rot_instance
                    ele_list_pack['part_name'] = part_name
                    ele_list_pack['part_value'] = part_value
                    ele_list_pack['symbol_name'] = symbol_name

                    element_lists.append(ele_list_pack)

                    length = xlim_instance[1] - xlim_instance[0]
                    width = ylim_instance[1] - ylim_instance[0]

                    lib         = part_data['library']
                    symbol       = instance['symbol']
                    # base_name    = os.path.splitext(os.path.basename(sch_file))[0]
                    key          = f"{lib}#{symbol}"

                    bbox_info = {
                        'part_name': part_name,
                        'gate_instance': gate_instance,
                        'x_center': (xlim_instance[0] + xlim_instance[1]) / 2,
                        'y_center': (ylim_instance[0] + ylim_instance[1]) / 2,
                        'length': length,
                        'width': width,
                        'key':key,
                        'cat':'symbol'
                    }

                    list_of_bounding_boxes.append(bbox_info)

    return list_of_bounding_boxes, element_lists


def draw_list_of_bounding_boxes(bounding_boxes, ax, color='orange'):
    """
    Draws bounding boxes for each symbol instance on the given Matplotlib axis,
    and marks the center (x_center, y_center) with a cross.

    Args:
        bounding_boxes (list): A list of bounding box dictionaries with keys:
            'part_name', 'gate_instance', 'length', 'width', 'x_center', 'y_center'.
        ax (matplotlib.axes.Axes): The axis to draw the bounding boxes on.
        color (str): The color of the bounding box (default: 'orange').
    """
    for bbox in bounding_boxes:
        x_center = bbox['x_center']
        y_center = bbox['y_center']
        length = bbox['length']
        width = bbox['width']


        # Print part name, gate instance, center coordinates and dimensions for each bounding box
        # print(f"Part: {bbox['part_name']}, Instance: {bbox['gate_instance']}, Center: ({x_center:.2f}, {y_center:.2f}), Length: {length:.2f}, Width: {width:.2f}")

        draw_one_box(
            x_center, y_center, length, width, ax, color=color
        )

### Bounding box Sheet

In [ ]:
def get_one_sheet_text_bounding_box(element, ax=None):
    """
    Compute the bounding box for a sheet text element.
    Handles multiline text, rotation, alignment, and mirroring.

    Parameters
    ----------
    element : dict
        Dictionary with keys:
          - 'text': the text string (may contain newlines)
          - 'x', 'y': coordinates (strings or floats)
          - 'size': text size scale
          - 'align': alignment (e.g., 'bottom-left', 'center')
          - 'rot': rotation string (e.g., 'R0', 'R90', 'MR0', 'MR90')
    ax : matplotlib axis (optional)
        Axis, if you want to debug by drawing.

    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y
    """

    # Extract fields safely
    text = str(element.get('text', ''))
    x = float(element.get('x', 0))
    y = float(element.get('y', 0))
    size = float(element.get('size', 1.0))
    align = str(element.get('align', 'bottom-left')).lower()
    rot_str = str(element.get('rot', 'R0')).upper()

    # Handle mirroring and rotation
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_str.startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_str[2:]) if len(rot_str) > 2 else 0.0
    elif rot_str.startswith('R'):
        rot_angle_deg = float(rot_str[1:]) if len(rot_str) > 1 else 0.0

    # Split text into lines
    lines = text.splitlines()
    num_lines = len(lines)
    max_line_len = max((len(line) for line in lines), default=0)

    # Estimate dimensions
    fontsize = size * 10
    text_width = int(max_line_len) * fontsize * 0.6 / 10.0
    text_height = int(num_lines) * fontsize * 1.2 / 10.0

    # Get polygon corners for the text box
    xs, ys = get_text_corners(
        x, y,
        align,
        rot_angle_deg,
        is_mirrored,
        text_width,
        text_height,
    )

    return min(xs), max(xs), min(ys), max(ys)

def get_list_of_sheet_text_bounding_boxes(schematic_info, space=1, ax=None):
    """
    Return a list of bounding box dictionaries for each sheet text element in the schematic.
    Each dictionary contains center coordinates, length, width, and metadata.
    Optionally draws boxes on the provided axis.
    """
    sheetinfo_texts = schematic_info['SheetInfo'].get('text', [])
    if not isinstance(sheetinfo_texts, list):
        sheetinfo_texts = [sheetinfo_texts]

    list_of_bounding_boxes = []

    for element in sheetinfo_texts:
        bbox = get_one_sheet_text_bounding_box(element)
        if bbox is None:
            continue
        element_min_x, element_max_x, element_min_y, element_max_y = bbox

        x_center = (element_min_x + element_max_x) / 2
        y_center = (element_min_y + element_max_y) / 2
        length = element_max_x - element_min_x
        width = element_max_y - element_min_y

        # Draw the bounding box (optional)
        # if ax is not None:
        #     draw_one_box(x_center, y_center, length + 1, width + 1, ax, color="#EEEA0A", thickness=2)

        bbox_info = {
            'part_name': 'SheetInfo',
            'gate_instance': element.get('text', ''),
            'x_center': x_center,
            'y_center': y_center,
            'length': length + 2 * space,
            'width': width + 2 * space,
            'key': 'text'
        }

        list_of_bounding_boxes.append(bbox_info)


    return list_of_bounding_boxes

def get_list_of_sheet_wire_bounding_boxes(schematic_info, space=1.0,ax=None):
    """
    Compute bounding boxes for each sheet wire segment.

    Parameters
    ----------
    sheet_wires : list or dict
        A list of wire segments (or a single wire dict) from the sheet.
    margin : float
        Padding around each bounding box.

    Returns
    -------
    list of dict
        Each dict contains bounding box info:
        {
            'type': 'sheet_wire',
            'x_center': ...,
            'y_center': ...,
            'length': ...,
            'width': ...,
            'xlim': (...),
            'ylim': (...)
        }
    """
    sheetinfo_wires = schematic_info['SheetInfo'].get('wire', [])
    if not isinstance(sheetinfo_wires, list):
        sheetinfo_wires = [sheetinfo_wires]

    list_of_bounding_boxes = []

    for wire in sheetinfo_wires:
        x1 = float(wire['x1'])
        y1 = float(wire['y1'])
        x2 = float(wire['x2'])
        y2 = float(wire['y2'])

        min_x = min(x1, x2) - space
        max_x = max(x1, x2) + space
        min_y = min(y1, y2) - space
        max_y = max(y1, y2) + space

        bbox = {
            'part_name': 'sheet_wire',
            'type': 'sheet_wire',
            'x_center': (min_x + max_x) / 2,
            'y_center': (min_y + max_y) / 2,
            'length': max_x - min_x,
            'width': max_y - min_y,
            'xlim': (min_x, max_x),
            'ylim': (min_y, max_y),
            'key': 'wire'
        }

        list_of_bounding_boxes.append(bbox)

    return list_of_bounding_boxes


### Bounding box Net

In [ ]:
def get_bounding_boxes_for_net_elements(net_data,net_name, space=1.0,ax=None):
    """
    Returns a list of bounding boxes for wires, junctions, and labels in a net.

    Parameters
    ----------
    data : list
        List of segments/items (from net['segment']).
    net_name : str
        The name of the net (used for metadata).
    margin : float
        Margin to pad around each bounding box.

    Returns
    -------
    list of dict
        Each dict contains:
        {
            'type': 'wire' | 'junction' | 'label',
            'net_name': net_name,
            'x_center': ...,
            'y_center': ...,
            'length': ...,
            'width': ...,
            'xlim': (...),
            'ylim': (...)
        }
    """

    bbox_list = []

    for item in net_data:
        # ------------------- WIRE -------------------
        if 'wire' in item:
            wires = item['wire'] if isinstance(item['wire'], list) else [item['wire']]
            for wire in wires:
                # print("Processing wire:", wire)
                x1, y1 = float(wire['x1']), float(wire['y1'])
                x2, y2 = float(wire['x2']), float(wire['y2'])

                min_x = min(x1, x2) - space
                max_x = max(x1, x2) + space
                min_y = min(y1, y2) - space
                max_y = max(y1, y2) + space

                 
                bbox ={
                    'part_name': 'wire',
                    'type': 'wire',
                    'net_name': net_name,
                    'x_center': (min_x + max_x) / 2,
                    'y_center': (min_y + max_y) / 2,
                    'length': max_x - min_x,
                    'width': max_y - min_y,
                    'xlim': (min_x, max_x),
                    'ylim': (min_y, max_y),
                    'key': 'wire'
                }
                bbox_list.append(bbox)

                # # Optional: draw the wire and its bounding box
                # if ax is not None:
                #     ax.plot([x1, x2], [y1, y2], linewidth=2, color='green')
                #     draw_one_box(bbox['x_center'], bbox['y_center'],
                #                 bbox['length'], bbox['width'], ax=ax,
                #                 color="#00E5FF", thickness=2)
    
        # ------------------- JUNCTION -------------------
        if 'junction' in item:
            junctions = item['junction'] if isinstance(item['junction'], list) else [item['junction']]
            for junction in junctions:
                x, y = float(junction['x']), float(junction['y'])

                bbox ={
                    'part_name': 'junction',
                    'type': 'junction',
                    'net_name': net_name,
                    'x_center': x,
                    'y_center': y,
                    'length': space * 2,
                    'width': space * 2,
                    'xlim': (x - space, x + space),
                    'ylim': (y - space, y + space),
                    'key': 'junction'
                }

                bbox_list.append(bbox)

                # # Optional: draw the wire and its bounding box
                # if ax is not None:
                #     ax.plot([x1, x2], [y1, y2], linewidth=2, color='green')
                #     draw_one_box(bbox['x_center'], bbox['y_center'],
                #                 bbox['length'], bbox['width'], ax=ax,
                #                 color="#00E5FF", thickness=2)


        # ------------------- LABEL -------------------
        if 'label' in item:
            labels = item['label'] if isinstance(item['label'], list) else [item['label']]
            for label in labels:
                x = float(label['x'])
                y = float(label['y'])

                bbox ={
                    'part_name': 'label',
                    'type': 'label',
                    'net_name': net_name,
                    'x_center': x,
                    'y_center': y,
                    'length': space * 2,
                    'width': space * 2,
                    'xlim': (x - space, x + space),
                    'ylim': (y - space, y + space),
                    'key':'labels'
                }

                bbox_list.append(bbox)

                # Optional: draw the wire and its bounding box
                # if ax is not None:
                #     ax.plot([x1, x2], [y1, y2], linewidth=2, color='green')
                #     draw_one_box(bbox['x_center'], bbox['y_center'],
                #                 bbox['length'], bbox['width'], ax=ax,
                #                 color="#00E5FF", thickness=2)


    return bbox_list


### Draw Component

In [ ]:
def draw_polygons(polygons, x=0, y=0, rot_angle_rad='R0',
                  color='red', fill=False, facecolor=None, ax=None):
    """
    Draw polygons (e.g., triangles in EAGLE symbols) with rotation and mirroring.

    Parameters
    ----------
    polygons : list
        Each dict has 'width' and a list of 'vertex' dicts with x, y.
    x, y : float
        Anchor position for the symbol instance.
    rot_angle_rad : str
        Rotation string like 'R0', 'R90', 'MR0', 'MR90'.
    color : str
        Outline color.
    fill : bool
        If True, fill the polygon.
    facecolor : str
        Color for fill (used if fill=True).
    ax : matplotlib axis
        Axis to draw on.
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    cos_t, sin_t = math.cos(theta), math.sin(theta)

    if ax is None:
        ax = plt.gca()

    for poly in polygons:
        vertices = poly.get('vertex', [])
        rotated_pts = []

        for v in vertices:
            px = float(v['x'])
            py = float(v['y'])

            # Apply rotation
            rx = px * cos_t - py * sin_t
            ry = px * sin_t + py * cos_t

            # Apply mirroring across vertical axis
            if is_mirrored:
                rx = -rx

            # Translate
            rotated_x = x + rx
            rotated_y = y + ry

            rotated_pts.append((rotated_x, rotated_y))

        width = float(poly.get('width', '0.1524'))

        poly_patch = Polygon(
            rotated_pts,
            closed=True,
            fill=fill,
            facecolor=facecolor if (fill and facecolor is not None) else 'none',
            edgecolor=color,
            linewidth=width
        )
        ax.add_patch(poly_patch)


def draw_circle(circle_data, x=0, y=0, rot_angle_rad='R0', ax = None):
    """
    Draws a circle with rotation.
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        rot_angle_deg = float(rot_angle_rad[2:])
        is_mirrored = True 
    rot_angle_rad = math.radians(rot_angle_deg)
    
    # Extract circle parameters
    x_center = float(circle_data['x'])
    y_center = float(circle_data['y'])
    radius   = float(circle_data['radius'])
    line_w   = float(circle_data['width'])

    # Apply rotation to the circle center
    rotated_x_center = x + (x_center * math.cos(rot_angle_rad) - y_center * math.sin(rot_angle_rad))
    if is_mirrored:
        rotated_x_center = x + (x_center * math.cos(rot_angle_rad) + y_center * math.sin(rot_angle_rad))
    else:
        rotated_x_center = x + (x_center * math.cos(rot_angle_rad) - y_center * math.sin(rot_angle_rad))

    rotated_y_center = y + (x_center * math.sin(rot_angle_rad) + y_center * math.cos(rot_angle_rad))


    # Create a Circle patch (unfilled by default)
    circle_patch = Circle(
        (rotated_x_center, rotated_y_center),
        radius,
        fill=False,         
        linewidth=line_w*10,
        edgecolor='red'     
    )

    # Add to the current axes
    if ax is None:
        ax = plt.gca()
    ax.add_patch(circle_patch)


def compress_pad_string(pad_str):
    """
    Given a space-separated pad string like '3 4 5',
    returns a compressed format like '3*3'
    """
    parts = pad_str.strip().split()
    
    if not parts:
        return ""
    

    first = parts[0]
    count = len(parts)
    if count == 1:
        return first
    return f"{first}*{count}"


def draw_pins(pins, x=0, y=0, rot_angle_rad='R0', ax=None, schematic_info = None, lib_name = None, de_set_name = None, device_name = None, scale = 1.0, gate = None):
    """
    Draws pins with rotation.
    """
    if ax is None:
        ax = plt.gca()
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    rot_angle_rad = math.radians(rot_angle_deg)
    # font_size = 12.0/100.0 * float(scale)

    # if is_mirrored:
    #     print("is_mirrored:", is_mirrored, "rot_angle_rad:", rot_angle_rad, "rot_angle_deg:", rot_angle_deg)
    # Define the mapping from length strings to extension distances
    length_map = {
        'point': 0.0,
        'short': 2.54,
        'middle': 2.54 * 2,
        'long': 2.54 * 3
    }

    rotated_pin_positions = []
    for pin in pins:
        x_base = float(pin['x'])
        y_base = float(pin['y'])
        x_rot = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
        y_rot = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))
        if is_mirrored:
            x_rot = 2*x - x_rot
        rotated_pin_positions.append((x_rot, y_rot))

    def compute_min_spacing(pin_positions):
        min_dist = float('inf')
        for (x1, y1), (x2, y2) in itertools.combinations(pin_positions, 2):
            dx = x2 - x1
            dy = y2 - y1
            dist = math.hypot(dx, dy)
            if dist > 1e-6:
                min_dist = min(min_dist, dist)
        return min_dist
    min_spacing = compute_min_spacing(rotated_pin_positions)
    font_size = max(min(min_spacing * 100, 12), 10)


    for pin in pins:
        # print("pin name:", pin['name'])
        # Get base coordinates
        x_base = float(pin['x'])
        # if is_mirrored:
        #     x_base = -x_base
        y_base = float(pin['y'])

        dotted = pin.get('function','') == 'dot'

        # Apply rotation to the pin base
        rotated_x_base = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
        rotated_y_base = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))

        # Determine extension length
        length_type = pin.get('length', 'long').lower()
        extension = length_map.get(length_type, 2.54)

        # Compute pin angle
        rot_pin_str = pin.get('rot', 'R0')
        pin_angle_deg = float(rot_pin_str[1:]) if rot_pin_str.upper().startswith('R') else (float(rot_pin_str[2:]))
        pin_angle_rad = math.radians(pin_angle_deg) + rot_angle_rad
        x_end = rotated_x_base + extension * math.cos(pin_angle_rad)
        y_end = rotated_y_base + extension * math.sin(pin_angle_rad)
        # Compute end of pin


        
        if is_mirrored and dotted:
            circle_centerx = rotated_x_base + (extension - 1) * math.cos(pin_angle_rad)
            circle_centery = rotated_y_base + (extension - 1) * math.sin(pin_angle_rad)
            x_end = rotated_x_base + (extension - 2) * math.cos(pin_angle_rad)
            y_end = rotated_y_base + (extension - 2) * math.sin(pin_angle_rad)

            x_end = 2*x - x_end
            rotated_x_base = 2*x - rotated_x_base
            #print("pin name:", pin['name'], "pin not_mirrored:", pin['name'], "rotated_x_base:", rotated_x_base, "rotated_y_base:", rotated_y_base, "x_end:", x_end, "y_end:", y_end, "calculated angle:", pin_angle_rad)
        elif is_mirrored:
            # x_end = rotated_x_base - extension * math.cos(pin_angle_rad)
            x_end = 2*x - x_end
            rotated_x_base = 2*x - rotated_x_base
            #print("pin name:", pin['name'], "pin is_mirrored:", pin['name'], "rotated_x_base:", rotated_x_base, "rotated_y_base:", rotated_y_base, "x_end:", x_end, "y_end:", y_end, "calculated angle:", pin_angle_rad)
        elif dotted:
            circle_centerx = rotated_x_base + (extension - 1) * math.cos(pin_angle_rad)
            circle_centery = rotated_y_base + (extension - 1) * math.sin(pin_angle_rad)
            x_end = rotated_x_base + (extension - 2) * math.cos(pin_angle_rad)
            y_end = rotated_y_base + (extension - 2) * math.sin(pin_angle_rad)
        
            #print("pin name:", pin['name'], "pin not_mirrored:", pin['name'], "rotated_x_base:", rotated_x_base, "rotated_y_base:", rotated_y_base, "x_end:", x_end, "y_end:", y_end, "calculated angle:", pin_angle_rad)
        
        dx = x_end - rotated_x_base
        dy = y_end - rotated_y_base
        length = math.hypot(dx, dy)
        if length > 1e-6:
            ux, uy = dx/length, dy/length
        else:
            ux, uy = 1.0, 0.0
    # change
        if length_type == 'point':
            offset_pts = -8
        elif length_type == 'short':
            offset_pts = -47
        elif length_type == 'middle':
            offset_pts = -75
        elif length_type == 'long':   
            offset_pts = -103
        offset_x = -ux * offset_pts
        # if is_mirrored:
        #     offset_x = -offset_x
        offset_y = -uy * (offset_pts - 8)
        pin_angle_deg_final = math.degrees(pin_angle_rad)
        pin_angle_deg_final = pin_angle_deg_final % 360
        align_map = {
            0:     ('left',  'center'),
            90:    ('left', 'center'),
            180:   ('right', 'center'),
            270:   ('right',  'center'),
        }
        ha, va = align_map.get(pin_angle_deg_final, ('center', 'center'))

        if is_mirrored:
            ha = 'right' if ha == 'left' else 'left'
        angel_drawed = pin_angle_deg_final
        if pin_angle_deg_final == 180:
            angel_drawed = 0

        if pin_angle_deg_final == 270:
            angel_drawed = 90



        # Draw the pin line and base marker
        if dotted:
            theta = np.linspace(0, 2 * np.pi, 100)
            ax.plot(circle_centerx + np.cos(theta), circle_centery + np.sin(theta), label = 'circle', color='red', lw=1, clip_on=True)
            ax.plot([rotated_x_base, x_end], [rotated_y_base, y_end], color='red', lw=1, clip_on=True)
        else:
            ax.plot([rotated_x_base, x_end], [rotated_y_base, y_end], color='red', clip_on=False, lw=1, zorder=20)
        ax.scatter([rotated_x_base], [rotated_y_base], color='blue', zorder=5, clip_on=True)
        
        # print('pin Name:', pin['name'], 'visible:', pin.get('visible', ''))

        if pin.get('visible', '').lower() in ['pin', 'both', '']:
                # ax.annotate(
                #     pin['name'],
                #     (rotated_x_base, rotated_y_base),
                #     textcoords="offset points",
                #     xytext=(offset_x, offset_y),
                #     ha=ha, va=va,
                #     fontsize=font_size, 
                #     color = 'Gray',
                #     rotation=angel_drawed,
                #     rotation_mode='anchor',    
                #     clip_on=True
                # )
                pin_name = pin['name'].split('@', 1)[0]
                ax.annotate(
                    pin_name,
                    (rotated_x_base, rotated_y_base),
                    textcoords="offset points",
                    xytext=(offset_x, offset_y),
                    ha=ha, va=va,
                    fontsize=font_size, 
                    color='Gray',
                    rotation=angel_drawed,
                    rotation_mode='anchor',    
                    clip_on=True
                )
        if pin.get('visible', '').lower() in ['pad', 'both', '']:
            # print("Pad Drawn: pin name:", pin['name'], "pin_angle_deg_final:", pin_angle_deg_final, "ha:", ha, "va:", va)
            connects = find_connects(schematic_info, library_name=lib_name, deviceset_name=de_set_name, device_name=device_name)
            with open('connects.json', 'w') as f:
                json.dump(connects, f, indent=2)
            for connect in connects:
                if connect['pin'] == pin['name'] and (connect['gate'] == gate or connect['gate'] is None or connect['gate'] == '' or gate is None or gate == ''):
                    if ha == 'left':
                        ha = 'right'
                    elif ha == 'right':
                        ha = 'left'
                    if va == 'top':
                        va = 'bottom'
                    elif va == 'bottom':
                        va = 'top'
                    if ha == 'left' and pin_angle_deg_final !=90 and pin_angle_deg_final != 270:
                        # print("left 0 or 180 Here")
                        if length_type == 'point' or length_type == 'short':
                            offset_x = offset_x + 40    
                        elif length_type == 'middle' or length_type == 'long':
                            offset_x = offset_x + 50
                    elif ha == 'right' and pin_angle_deg_final == 0:
                        if length_type == 'point' or length_type == 'short':
                            offset_x = offset_x -40
                        elif length_type == 'middle' or length_type == 'long':
                            offset_x = offset_x -60
                    elif ha == 'right' and pin_angle_deg_final == 180:
                        if length_type == 'point' or length_type == 'short':
                            offset_x = offset_x -40
                        elif length_type == 'middle' or length_type == 'long':
                            offset_x = offset_x -60



                    
                    if pin_angle_deg_final == 90:
                        
                        if length_type == 'point' or length_type == 'short':
                            offset_y = offset_y -50
                        elif length_type == 'middle' or length_type == 'long':
                            offset_y = offset_y -60
                    if pin_angle_deg_final == 270:
                        if length_type == 'point' or length_type == 'short':
                            offset_y = offset_y + 40    
                        elif length_type == 'middle' or length_type == 'long':
                            offset_y = offset_y + 50
                        # print("270 de pin/pad:",ha, va)

                    # elif pin_angle_deg_final == 90:
                    #     offset_x = offset_x  -100
                    # elif pin_angle_deg_final == 270:
                    #     offset_x = offset_x + 5
                      

                    if angel_drawed == 90 :
                        va = 'bottom'
                    # elif pin_angle_deg_final == 270:
                    #     va = 'bottom'
                    # elif ha == 'center' and va == 'bottom':
                    #     offset_x = offset_x
                    # elif ha == 'center' and va == 'top':
                    #     offset_x = offset_x +20

                    if va == 'center':
                        offset_y = offset_y -10
                    angel_drawed = pin_angle_deg_final
                    if pin_angle_deg_final == 180:
                        angel_drawed = 0

                    if pin_angle_deg_final == 270:
                        angel_drawed = 90
                    draw_text = compress_pad_string(connect['pad'])
                    if dotted and pin_angle_deg_final == 180:
                        
                        ax.annotate(
                            draw_text,
                            (x_end - 2, y_end),
                            textcoords="offset points",
                            xytext=(-offset_x, -offset_y),
                            ha=ha, va=va,
                            fontsize=font_size, 
                            color = 'Gray',
                            rotation=angel_drawed,            # ← rotate text
                            rotation_mode='anchor',    
                            clip_on=True
                        )
                    elif dotted and pin_angle_deg_final == 0:
                        ax.annotate(
                            draw_text,
                            (x_end + 1, y_end),
                            textcoords="offset points",
                            xytext=(-offset_x, -offset_y),
                            ha=ha, va=va,
                            fontsize=font_size, 
                            color = 'Gray',
                            rotation=angel_drawed,            # ← rotate text
                            rotation_mode='anchor',    
                            clip_on=True
                        )
                    else:
                        ax.annotate(
                            draw_text,
                            (x_end, y_end),
                            textcoords="offset points",
                            xytext=(-offset_x, -offset_y),
                            ha=ha, va=va,
                            fontsize=font_size, 
                            color = 'Gray',
                            rotation=angel_drawed,            # ← rotate text
                            rotation_mode='anchor',    
                            clip_on=True
                        )
 

    return rot_angle_rad


def draw_rectangle(rect_data, x=0, y=0, rot_angle_rad='R0', color='red', alpha=1.0, ax=None):
    """
    Draw a rectangle that supports rotation and mirroring (Rθ / MRθ).
    The rectangle is defined by two opposite corners (x1,y1)-(x2,y2) in local coords.
    Transform order: mirror -> rotate -> translate (about origin x,y).
    """
    if ax is None:
        ax = plt.gca()

    # Parse rotation/mirror
    rot_str = str(rot_angle_rad).upper()
    mirrored = False
    rot_deg = 0.0
    if rot_str.startswith('MR'):
        mirrored = True
        rot_deg = float(rot_str[2:] or 0.0)
    elif rot_str.startswith('R'):
        rot_deg = float(rot_str[1:] or 0.0)

    theta = math.radians(rot_deg)
    c, s = math.cos(theta), math.sin(theta)

    # Local rect corners (axis-aligned in local space)
    x1 = float(rect_data['x1']); y1 = float(rect_data['y1'])
    x2 = float(rect_data['x2']); y2 = float(rect_data['y2'])
    corners = [(x1,y1), (x1,y2), (x2,y2), (x2,y1)]

    def transform(px, py):
        # mirror in local space
        if mirrored:
            px = -px
        # rotate about origin
        rx = px * c - py * s
        ry = px * s + py * c
        # translate to (x, y)
        return (x + rx, y + ry)

    world_pts = [transform(px, py) for (px, py) in corners]

    patch = Polygon(world_pts, closed=True, facecolor=color, edgecolor='none', alpha=alpha)
    ax.add_patch(patch)


def get_arrow_triangle(x, y, angle_deg = 0, size = 20, mirror=False):
    tri = np.array([
        [-size / 2, 0],            # Tip of arrow
        [0, size / 2],             # Left base
        [0, -size / 2],            # Right base
    ])

    # Rotate the triangle around (0,0)
    theta = np.radians(angle_deg)
    rot_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])
    tri_rotated = tri @ rot_matrix.T

    # Mirror in local frame (flip x-values at local origin)
    if mirror:
        tri_rotated[:, 0] *= -1  # flip x

    # Now translate to (x, y)
    tri_translated = tri_rotated + np.array([x, y])
    return tri_translated



def draw_part_attributes(attributes, part_name="Unknown", part_value="Unknown", symbol_name="Unknown", ax = None, rot_angle=None, gate = None):
    """
    Draws part attributes such as name and value on the plot.

    Args:
        attributes (list or dict): List or dictionary of attributes to draw.
        x (float): X-coordinate of the center.
        y (float): Y-coordinate of the center.
        rot_angle_rad (float): Rotation angle in radians.
        part_name (str): Name of the part.
        part_value (str): Value of the part.
        symbol_name (str): Name of the symbol.
    """

    if not attributes:  # Check if attributes is empty
        return
    if ax is None:
        ax = plt.gca()
    if isinstance(attributes, dict):  # Convert dictionary to list for uniform processing
        attributes = [attributes]

    for attribute in attributes:
        if attribute.get('display', 'on').lower() == 'off':
            continue

        attr_name = attribute.get('name', '')
        attr_x = float(attribute.get('x', 0))
        attr_y = float(attribute.get('y', 0))
        attr_size = float(attribute.get('size', 1.0))
        xref_yes  = attribute.get('xref', 'no').lower() == 'yes'
        attr_rot = attribute.get('rot', 'R0')
        attr_align = attribute.get('align', 'bottom-left').lower()
        attr_rot = attr_rot.upper()
        is_mirrored = attr_rot.startswith('MR')
        rot_val = attr_rot[2:] if is_mirrored else attr_rot[1:]
        rot_deg = int(rot_val)




        ha, va = get_text_alignment(attr_align, rot_deg, is_mirrored)
        
        attr_rot_deg = rot_deg


        if attr_rot_deg == 180:
            attr_rot_deg = 0
        elif attr_rot_deg == 270:
            attr_rot_deg = 90
        
        bbox_kw = None
        if xref_yes:
            bbox_kw = dict(
                boxstyle="square,pad=0.2",
                facecolor="white",
                edgecolor="gray",
                linewidth=1
            )

        # print("Drawing attribute:", attr_name, "at position:", attr_x, attr_y, "with rotation:", attr_rot_deg, "and size:", attr_size, 'ha', ha, 'va:', va)
        # Determine if the attribute is for part name or part value
        if attr_name == "NAME" and part_name != "Unknown":

            ax.text(attr_x, attr_y, part_name, fontsize= float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color='blue')
            
        elif attr_name == "VALUE":
            # Use symbol_name as part_value if part_value is unknown
            # print(f"Drawing attribute '{attr_name}' for part '{part_name}' with value '{part_value}'")
            display_value = part_value if part_value not in ["Unknown", ""] else None

            if 'GND' in part_name.upper():
                # Remove all digits but keep letters, dashes, etc.
                display_value = re.sub(r'\d+', '', part_name)
                

            ax.text(attr_x, attr_y, display_value, fontsize= float(attr_size) * 10, ha=ha, va=va, 
                    rotation=attr_rot_deg, color='green')
            
        elif  "GND" != symbol_name:
            # print(f"Drawing attribute '{attr_name}' for part '{part_name}'")
            ax.text(attr_x, attr_y, attr_name, fontsize= float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color='red')
            

        cross_size = 0.3
        ax.plot([attr_x - cross_size, attr_x + cross_size],
        [attr_y, attr_y],
        color='lightgray', linewidth=1, zorder=0)


        ax.plot([attr_x, attr_x],
                [attr_y + cross_size, attr_y - cross_size],
                color='lightgray', linewidth=1, zorder=0)

     
def draw_texts(attributes, part_name="Unknown", part_value="Unknown", symbol_name="Unknown", ax = None, rot_angle_rad=None, gate = None, x = 0, y = 0):
    """
    Draws part attributes such as name and value on the plot.

    Args:
        attributes (list or dict): List or dictionary of attributes to draw.
        x (float): X-coordinate of the center.
        y (float): Y-coordinate of the center.
        rot_angle_rad (float): Rotation angle in radians.
        part_name (str): Name of the part.
        part_value (str): Value of the part.
        symbol_name (str): Name of the symbol.
    """
    # if part_name == "J6":
    #     import json
    #     with open("example_text_part.json", "w") as f:
    #                     json.dump(attributes, f, indent=4)
    if not attributes:  # Check if attributes is empty
        return
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    rot_angle_rad = math.radians(rot_angle_deg)

    if ax is None:
        ax = plt.gca()
    if isinstance(attributes, dict):  # Convert dictionary to list for uniform processing
        attributes = [attributes]

    for attribute in attributes:
        if attribute.get('display', 'on').lower() == 'off':
            continue
        attr_name = attribute.get('text', '')
        if attr_name.startswith('>'):
            continue
        # print("symbol_text:",attribute)
        x_base = float(attribute.get('x', 0))
        y_base = float(attribute.get('y', 0))
        attr_size = float(attribute.get('size', 1.0))
        xref_yes  = attribute.get('xref', 'no').lower() == 'yes'
        attr_rot = attribute.get('rot', 'R0')
        attr_align = attribute.get('align', 'bottom-left').lower()

        pin_angle_deg = float(attr_rot[1:]) if attr_rot.upper().startswith('R') else (float(attr_rot[2:]))
        pin_angle_rad = math.radians(pin_angle_deg) + rot_angle_rad
        pin_angle_deg_final = math.degrees(pin_angle_rad)

        x_rot = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
        y_rot = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))
        if is_mirrored:
            x_rot = 2*x - x_rot


        attr_x = x_rot
        attr_y = y_rot

        ha,va = get_text_alignment(attr_align, pin_angle_deg_final, is_mirrored)

        attr_rot_deg = pin_angle_deg_final


        if attr_rot_deg == 180:
            attr_rot_deg = 0
        elif attr_rot_deg == 270:
            attr_rot_deg = 90
        
        bbox_kw = None
        if xref_yes:
            bbox_kw = dict(
                boxstyle="square,pad=0.2",
                facecolor="white",
                edgecolor="gray",
                linewidth=1
            )

        # print("Drawing attribute:", attr_name, "at position:", attr_x, attr_y, "with rotation:", attr_rot_deg, "and size:", attr_size, 'ha', ha, 'va:', va)
        # Determine if the attribute is for part name or part value


        layer = attribute.get('layer', '-1')  # Default to layer '-1' if not specified

        color = 'black'
        if layer  in ['97','95','96']:
            color = 'gray'
        elif layer == '94':
            color = 'red'
        elif layer == '98':
            color = '#d7d769'

        ax.text(attr_x, attr_y, attr_name, fontsize=float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color=color)
            

        cross_size = 0.3
        ax.plot([attr_x - cross_size, attr_x + cross_size],
        [attr_y, attr_y],
        color='lightgray', linewidth=1, zorder=0)


        ax.plot([attr_x, attr_x],
                [attr_y + cross_size, attr_y - cross_size],
                color='lightgray', linewidth=1, zorder=0)


def draw_wire_eagle_style(wire, x=0, y=0, rot_angle_rad='R0', ax=None):
    """
    Draws a wire or arc with support for rotation and mirroring (EAGLE style).
    Fixes mirroring bug for arcs (curve sign inversion).
    """
    import math
    from matplotlib.patches import Arc
    import matplotlib.pyplot as plt
    # print("wire drawn: x: ", x,",y: ", y, "rot_angle_rad: ", rot_angle_rad)
    # Parse rotation & mirror
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    # Rotation matrix
    cosθ, sinθ = math.cos(theta), math.sin(theta)
    rot_matrix = [[cosθ, -sinθ], [sinθ, cosθ]]

    # Mirror matrix
    mirror_matrix = [[-1, 0], [0, 1]] if is_mirrored else [[1, 0], [0, 1]]
    # mirror_matrix = [[1, 0], [0, 1]]
    def transform_point(px, py):
        # First apply mirroring
        if is_mirrored:
            px = -px
        # Then apply rotation
        rx = px * rot_matrix[0][0] + py * rot_matrix[0][1]
        ry = px * rot_matrix[1][0] + py * rot_matrix[1][1]
        return x + rx, y + ry

    def transform_point(px, py):
        # Apply rotation
        rx = px * rot_matrix[0][0] + py * rot_matrix[0][1]
        ry = px * rot_matrix[1][0] + py * rot_matrix[1][1]
        # Apply mirror
        mx = rx * mirror_matrix[0][0] + ry * mirror_matrix[0][1]
        my = rx * mirror_matrix[1][0] + ry * mirror_matrix[1][1]
        # Translate
        return x + mx, y + my
    
    # Extract and transform endpoints
    x1_orig, y1_orig = float(wire['x1']), float(wire['y1'])
    x2_orig, y2_orig = float(wire['x2']), float(wire['y2'])
    x1, y1 = transform_point(x1_orig, y1_orig)
    x2, y2 = transform_point(x2_orig, y2_orig)
    
    # if is_mirrored:
    #     x2, y2 = transform_point(x1_orig, y1_orig)
    #     x1, y1 = transform_point(x2_orig, y2_orig)

    # Get curve (adjust if mirrored)
    curve_deg = float(wire.get('curve', 0))
    if is_mirrored:
        curve_deg *= -1  # 🔁 key fix: mirror inverts curve direction

    if abs(curve_deg) < 1e-9:
        if ax is None: ax = plt.gca()
        ax.plot([x1, x2], [y1, y2], 'r-')
        return

    #  Arc geometry (based on original coords)
    chord_len = math.hypot(x2_orig - x1_orig, y2_orig - y1_orig)
    if chord_len < 1e-9:
        return

    theta_arc = math.radians(abs(curve_deg))
    R = chord_len / (2.0 * math.sin(theta_arc / 2.0))
    mx_orig = (x1_orig + x2_orig) / 2.0
    my_orig = (y1_orig + y2_orig) / 2.0
    d = math.sqrt(R * R - (chord_len / 2.0) ** 2)

    vx, vy = x2_orig - x1_orig, y2_orig - y1_orig
    nx, ny = -vy, vx
    length_n = math.hypot(nx, ny)
    nx, ny = nx / length_n, ny / length_n
    if not is_mirrored:
        if curve_deg > 0:
            cx_orig = mx_orig + d * nx
            cy_orig = my_orig + d * ny
        else:
            cx_orig = mx_orig - d * nx
            cy_orig = my_orig - d * ny
    else:
        if curve_deg > 0:
            cx_orig = mx_orig - d * nx
            cy_orig = my_orig - d * ny
        else:
            cx_orig = mx_orig + d * nx
            cy_orig = my_orig + d * ny

    # Transform arc center
    cx, cy = transform_point(cx_orig, cy_orig)

    # Compute angles (in transformed space)
    start_angle = math.degrees(math.atan2(y1 - cy, x1 - cx)) % 360
    end_angle = math.degrees(math.atan2(y2 - cy, x2 - cx)) % 360

    if curve_deg > 0:
        if end_angle < start_angle:
            end_angle = end_angle + 360
    else:
        if start_angle < end_angle:
            start_angle = start_angle + 360
        start_angle, end_angle = end_angle, start_angle

    # Draw arc
    linewidth = float(wire.get('width', 0.1524))
    if ax is None: ax = plt.gca()
    arc = Arc((cx, cy), 2 * R, 2 * R, angle=0,
              theta1=start_angle, theta2=end_angle,
              color='red', linewidth=linewidth * 10)
    ax.add_patch(arc)
    # ax.plot(cx, cy, 'go', markersize=8)
    # ax.plot(x1, y1, 'go', markersize=1)
    # ax.plot(x2, y2, 'go', markersize=1)
    # ax.plot([x1, x2], [y1, y2], 'r-', linewidth=linewidth * 10)
    

### Draw Symbol

In [ ]:
def visualize_wires_junctions(data, net_name, ax = None):
    """
    Visualizes wires, junctions, and labels from the given data using matplotlib.
    """
    if ax is None:
        ax = plt.gca()  # Get current axes
    ax.yaxis.set_inverted(True)  # Invert y-axis here, within the function itself
    ax.set_aspect('equal', adjustable='box')

    min_x, max_x, min_y, max_y = float('inf'), float('-inf'), float('inf'), float('-inf')
    data_found = False  # Flag to track if any wire/junction data was processed

    for item in data:
        if 'wire' in item:
            data_found = True  # Set flag to True because we found wire data
            if isinstance(item['wire'], list):
                for wire_seg in item['wire']:
                    x1 = float(wire_seg['x1'])
                    y1 = float(wire_seg['y1'])
                    x2 = float(wire_seg['x2'])
                    y2 = float(wire_seg['y2'])
                    width = float(wire_seg['width']) if 'width' in wire_seg else 0.1524
                    layer = wire_seg['layer'] if 'layer' in wire_seg else '91'
                    ax.plot([x1, x2], [y1, y2], linewidth=width * 10, color='green')
                    min_x = min(min_x, x1, x2)
                    max_x = max(max_x, x1, x2)
                    min_y = min(min_y, y1, y2)
                    max_y = max(max_y, y1, y2)
            elif isinstance(item['wire'], dict):
                wire_seg = item['wire']
                x1 = float(wire_seg['x1'])
                y1 = float(wire_seg['y1'])
                x2 = float(wire_seg['x2'])
                y2 = float(wire_seg['y2'])
                width = float(wire_seg['width']) if 'width' in wire_seg else 0.1524
                layer = wire_seg['layer'] if 'layer' in wire_seg else '91'
                ax.plot([x1, x2], [y1, y2], linewidth=width * 20, color='green')
                min_x = min(min_x, x1, x2)
                max_x = max(max_x, x1, x2)
                min_y = min(min_y, y1, y2)
                max_y = max(max_y, y1, y2)

        if 'junction' in item:
            data_found = True  # Set flag to True because we found junction data
            if isinstance(item['junction'], list):
                for junction in item['junction']:
                    x = float(junction['x'])
                    y = float(junction['y'])
                    ax.plot(x, y, marker='o', markersize=5, color='green')
                    min_x = min(min_x, x)
                    max_x = max(max_x, x)
                    min_y = min(min_y, y)
                    max_y = max(max_y, y)
            elif isinstance(item['junction'], dict):
                junction = item['junction']
                x = float(junction['x'])
                y = float(junction['y'])
                ax.plot(x, y, marker='o', markersize=5, color='green')
                min_x = min(min_x, x)
                max_x = max(max_x, x)
                min_y = min(min_y, y)
                max_y = max(max_y, y)

        if 'label' in item:
            data_found = True  # Set flag to True because we found label data
            if isinstance(item['label'], list):
                for label in item['label']:
                    x = float(label['x'])
                    y = float(label['y'])
                    font_size = float(label.get('size', 8)) * 15  # Dynamically get font size, default to 8
                    rot = label.get('rot', 'R0').upper()
                    

                    if rot.startswith('MR'):
                        is_mirrored = True
                        rot = rot[2:]
                    else:
                        is_mirrored = False
                        rot = rot[1:]
                    rot = int(rot)
                    # print("label:", net_name, "rot:", rot)
                    # label_offset_x = -1 if rot != 'R180' else 1  # Adjust offset based on rotation
                    label_offset_y = 0  # No distance for y-coordinate
                    is_xref = label.get('xref', '').lower() == 'yes'

                    if not is_mirrored:
                        if rot == 0 or rot == 90:
                            ha = 'left'
                            label_offset_x = 1
                        elif  rot == 180 or rot == 270:
                            ha = 'right'
                            label_offset_x = -1
                        else:
                            ha = 'center'
                            label_offset_x = 0
                    else:
                        if rot == 0 or rot == 90:
                            ha = 'right'
                            label_offset_x = 1
                        elif  rot == 180 or rot == 270:
                            ha = 'left'
                            label_offset_x = -1
                        else:
                            ha = 'center'
                            label_offset_x = 0

                    if rot == 90:
                        va = 'center'
                        label_offset_x = 0
                    elif rot == 270:
                        va = 'center'
                        label_offset_x = 0
                    else:
                        va = 'center'
                        label_offset_y = 0
                    label_offset_y = 0

                    # if net_name == "ADC":
                    #     print("net found:", net_name, "rot:", rot, "y:", y + label_offset_y, 'va:', va, 'ha:', ha)
                    bbox_kw = dict(
                        boxstyle="square,pad=0.2",
                        facecolor="white",
                        edgecolor="gray",
                        linewidth=1
                    ) if is_xref else None


                    tri = get_arrow_triangle(x + label_offset_x, y + label_offset_y, rot, 2.5, is_mirrored)
                    if rot == 180:
                        rot = 0
                    elif rot == 270:
                        rot = 90

                    # if net_name == "LD1" or net_name == "SW3V3":
                    #     print("net found:", net_name, "rot:", rot, "y:", y + label_offset_y)
                    # print("rot angle:",rot)
                    ax.text(x + label_offset_x, y + label_offset_y, 
                            net_name, 
                            fontsize=font_size, 
                            ha=ha, 
                            color='gray', 
                            bbox = bbox_kw, 
                            clip_on=True, 
                            rotation=rot,
                        rotation_mode='anchor', va=va)
                   
                    if is_xref:
                        ax.add_patch(Polygon(tri, closed=True, color='gray'))
                    min_x = min(min_x, x + label_offset_x)
                    max_x = max(max_x, x + label_offset_x)
                    min_y = min(min_y, y)
                    max_y = max(max_y, y)
            elif isinstance(item['label'], dict):
                label = item['label']
                x = float(label['x'])
                y = float(label['y'])
                font_size = float(label.get('size', 8)) * 15  # Dynamically get font size, default to 8
                rot = label.get('rot', 'R0').upper()

                if rot.startswith('MR'):
                    is_mirrored = True
                    rot = rot[2:]
                else:
                    is_mirrored = False
                    rot = rot[1:]
                rot = int(rot) if not is_mirrored else 360 - int(rot)

                label_offset_y = 0  # No distance for y-coordinate
                is_xref = label.get('xref', '').lower() == 'yes'


                if not is_mirrored:
                    if rot == 0 or rot == 90:
                        ha = 'left'
                        label_offset_x = 1
                    elif  rot == 180 or rot == 270:
                        ha = 'right'
                        label_offset_x = -1
                    else:
                        ha = 'center'
                        label_offset_x = 0
                else:
                    if rot == 0 or rot == 90:
                        ha = 'right'
                        label_offset_x = 1
                    elif  rot == 180 or rot == 270:
                        ha = 'left'
                        label_offset_x = -1
                    else:
                        ha = 'center'
                        label_offset_x = 0

                if rot == 90:
                    va = 'center'
                    label_offset_x = 0
                    label_offset_y = 0
                elif rot == 270:
                    va = 'center'
                    label_offset_x = 0
                    label_offset_y = 0
                else:
                    va = 'center'
                    label_offset_y = 0
                bbox_kw = dict(
                    boxstyle="square,pad=0.2",
                    facecolor="white",
                    edgecolor="gray",
                    linewidth=1
                ) if is_xref else None



                tri = get_arrow_triangle(x + label_offset_x, y + label_offset_y, rot, 2.7)
                if rot == 180:
                    rot = 0
                elif rot == 270:
                    rot = 90



                ax.text(x + label_offset_x, y + label_offset_y, 
                            net_name, 
                            fontsize=font_size, 
                            ha=ha, 
                            color='gray', 
                            bbox = bbox_kw, 
                            clip_on=True, 
                            rotation=rot,
                        rotation_mode='anchor', va=va)
                
                min_x = min(min_x, x + label_offset_x)
                max_x = max(max_x, x + label_offset_x)
                min_y = min(min_y, y)
                max_y = max(max_y, y)


                if is_xref:
                    ax.add_patch(Polygon(tri, closed=True, color='gray'))



                # label_offset_x = 5 if rot != 'R180' else -5  # Adjust offset based on rotation
                # label_offset_y = 0  # No distance for y-coordinate

                # is_xref = label.get('xref', '').lower() == 'yes'




    if data_found:  # Only set limits and labels if data was found
        ax.set_xlim(min_x - 5, max_x + 5)
        ax.set_ylim(min_y - 5, max_y + 5)
        # ax.set_xlabel("X Coordinate")
        # ax.set_ylabel("Y Coordinate")
        # ax.set_title("Visualization of Wires, Junctions, and Labels")
        # plt.grid(True, linestyle='--', alpha=0.6)
    else:
        ax.text(0.5, 0.5, "No wire, junction, or label data to display", ha='center', va='center', fontsize=12, color='gray')  # Display message if no data


In [ ]:


# Function to visualize symbol with rotation
def visualize_symbol_from_dict(symbol_data, x=0, y=0, rot_angle_rad='R0', part_name="Unknown",display_part_name='Unknown', part_value="Unknown",symbol_name="Unknown", ax = None, gate = None, schematic_info  = None, lib_name = None, de_set_name = None, device_name = None):
    """
    Visualizes the symbol data with rotation.

    Args:
        symbol_data (dict): Dictionary containing symbol information.
        x (float): X-coordinate of the center.
        y (float): Y-coordinate of the center.
        rot (str): Rotation of the symbol, e.g., 'R90', 'R0', 'R270'.
    """
    # Convert rotation string to radians
    # rot_angle_deg = 0.0
    # if rot.upper().startswith('R'):
    #     rot_angle_deg = float(rot[1:])
    # rot_angle_rad = math.radians(rot_angle_rad)

    # Extract symbol elements
    pins = symbol_data.get('pin', [])
    # print("pins:",pins)
    wires = symbol_data.get('wire', [])
    rectangles = symbol_data.get('rectangle', [])
    circles = symbol_data.get('circle', [])
    polygons = symbol_data.get('polygon', [])
    attributes = symbol_data.get('attribute', [])
    texts = symbol_data.get('symbol_text', [])
    # print("here is the texts:",texts)
    # if part_name == "J6":
    #     import json
    #     with open("example_text_symbol.json", "w") as f:
    #                     json.dump(symbol_data, f, indent=4)
    if gate == "G$1":
        gate = None


    # Plot circles
    for circle in circles:
        
        draw_circle(circle, x, y, rot_angle_rad, ax)


    # Plot polygons
    if polygons:
        
        draw_polygons(polygons, x, y, rot_angle_rad, color='red', fill=True, facecolor='red', ax = ax)


    # Plot pins
    # print("start to draw pins")
    # print("pins")
    
    angle = draw_pins(pins, x, y, rot_angle_rad, ax, schematic_info = schematic_info, lib_name = lib_name, de_set_name = de_set_name, device_name = device_name, gate = gate)
    
    # Draw wires
    for wire in wires:
        draw_wire_eagle_style(wire, x, y, rot_angle_rad, ax)

                                                                
    # Draw rectangles

    for rect_data in rectangles:
        
        draw_rectangle(rect_data, x, y, rot_angle_rad, color='red', alpha=1.0, ax =ax)


    # print("here are the texts:",texts)
    draw_texts(texts, part_name, part_value, symbol_name, ax = ax, rot_angle_rad=rot_angle_rad, gate = gate, x = x, y = y)
    

    draw_part_attributes(attributes, display_part_name, part_value, symbol_name, ax = ax, rot_angle=angle, gate = gate)

    # Draw cross
    cross_size = 0.5
    ax.plot([x - cross_size, x + cross_size], [y, y],
            color="#ff6666", linewidth=1, zorder=0)
    ax.plot([x, x], [y + cross_size, y - cross_size],
            color="#ff6666", linewidth=1, zorder=0)

    


def visualize_schematic(schematic_file_path, save_path=None,draw_grid=True,draw_bounding_bbox=False):
    """
    Visualizes the schematic based on the provided schematic_info.

    Args:
        schematic_info (dict): Dictionary containing schematic information, including parts and nets.
    """
    schematic_info = process_schematic_file(schematic_file_path)
    list_of_bounding_box = []

    list_of_symbol_bounding_box, element_lists = get_list_of_symbol_bounding_boxes(schematic_info,space=1)

    list_of_sheet_text_bounding_box = get_list_of_sheet_text_bounding_boxes(schematic_info,space=1)

    list_of_sheet_wire_bounding_box = get_list_of_sheet_wire_bounding_boxes(schematic_info, space=1)

    list_of_net_bounding_box = []
    bounding_boxes = []

    bounding_boxes.extend(list_of_sheet_text_bounding_box)
    bounding_boxes.extend(list_of_symbol_bounding_box)
    bounding_boxes.extend(list_of_sheet_wire_bounding_box)
    bounding_box_full = get_bounding_box_full_sch(bounding_boxes)[0]

    if not bounding_box_full:
        print("Empty schematic: ", schematic_file_path)
        return None
         

    for net_name, net_data in schematic_info['nets'].items():
        list_of_net_bounding_box.extend(
            get_bounding_boxes_for_net_elements(net_data, net_name, space=0.5)
        )

    list_of_bounding_box.extend(list_of_symbol_bounding_box)
    list_of_bounding_box.extend(list_of_sheet_text_bounding_box)
    list_of_bounding_box.extend(list_of_sheet_wire_bounding_box)
    list_of_bounding_box.extend(list_of_net_bounding_box)
    # list_of_bounding_box.extend([bounding_box_full])

    x_center = bounding_box_full['x_center']
    y_center = bounding_box_full['y_center']
    sch_length = bounding_box_full['length']
    sch_width =  bounding_box_full['width']
    xlim = [x_center - sch_length/2, x_center + sch_length/2]
    ylim = [y_center - sch_width/2, y_center + sch_width/2]
    # (xmin, xmax), (ymin, ymax) = bounding_box
    raw_w  = sch_length
    raw_h  = sch_width
    # print("schematics_raw_dimensions:", raw_w, raw_h)
    fig_w    = raw_w  / 5.0
    fig_h    = raw_h  / 5.0

    
    # print("Dimensions of schematic:", x, y)
    # if (x == -1):
    #     print("Error: Invalid schematic file or empty schematic.")
    #     return
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=100)  # Create figure and axes
    # ax = plt.gca()  # Get current axes
    # print("Parts in schematic:", schematic_info['parts'])

      # Parse the schematic file
    import json
    with open("example_symbol_part.json", "w") as f:
                    json.dump(schematic_info, f, indent=4)
    # Visualize parts
    for part_name, part_data in schematic_info['parts'].items():
        part_value = part_data.get('value', 'Unknown')
        lib_name = part_data.get('library', 'Unknown')
        de_set_name = part_data.get('deviceset', 'Unknown')
        device_name = part_data.get('device', 'Unknown')

        

        for instance in part_data.get('instances', []):
            symbol_name = instance['symbol']
            x = float(instance['x'])
            y = float(instance['y'])
            rot = instance['rot']
            gate = instance['gate']
            display_part_name = part_name+gate if len(part_data.get('instances', []))>1 else part_name
            symbol_element = get_symbol_element_of_instance_from_library(
                schematic_info['IC_library'], part_name, gate, schematic_info['parts']
            )
            # print("part_data:", symbol_name)
            # test
            if symbol_element:
                visualize_symbol_from_dict(symbol_element, x, y, rot, part_name, display_part_name,part_value, symbol_name, gate = gate, schematic_info = schematic_info, ax = ax, lib_name = lib_name, de_set_name = de_set_name, device_name = device_name)

    # Visualize nets
    for net_name, net_data in schematic_info['nets'].items():
        visualize_wires_junctions(net_data, net_name)


    # Visualize sheet
    sheetinfo_wires = schematic_info['SheetInfo'].get('wire', [])
    sheetinfo_texts = schematic_info['SheetInfo'].get('text', [])
    
    visualize_sheet_wires(sheetinfo_wires,ax=ax)
    visualize_sheet_texts(sheetinfo_texts,ax=ax)
    
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # Draw bounding box in red dashed line
    rect = Rectangle(
        (xlim[0], ylim[0]),  # Bottom-left corner
        xlim[1] - xlim[0],   # Width
        ylim[1] - ylim[0],   # Height
        linewidth=10,
        edgecolor='red',
        linestyle='--',
        facecolor='none'
    )
    ax.add_patch(rect)

    if draw_bounding_bbox:
        draw_list_of_bounding_boxes(list_of_symbol_bounding_box,ax, color='orange')
        draw_list_of_bounding_boxes(list_of_sheet_text_bounding_box,ax, color='gray')
        draw_list_of_bounding_boxes(list_of_sheet_wire_bounding_box,ax, color='gray')
        draw_list_of_bounding_boxes(list_of_net_bounding_box,ax, color='blue')

        # draw_list_of_bounding_boxes(list_of_bounding_box,ax, color='blue')

        # draw_element_list(element_lists, draw_list=[],ax =ax)
        pass


    setting = schematic_info['setting']
    visualize_grid_from_setting(setting,ax=ax,draw_grid=draw_grid)


    ax.set_xlabel(None)
    ax.set_ylabel(None)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    if save_path:
        print("plt saved")
        plt.savefig(save_path, bbox_inches='tight')
        
    fig = plt.gcf()
    fig.canvas.draw()


    src_xlim = ax.get_xlim()
    src_ylim = ax.get_ylim()
    src_width = src_xlim[1] - src_xlim[0]
    src_height = src_ylim[1] - src_ylim[0]



    bbox_inches = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    pixel_width = bbox_inches.width * fig.dpi
    pixel_height = bbox_inches.height * fig.dpi


    pixels_per_unit_x = pixel_width / src_width
    pixels_per_unit_y = pixel_height / src_height
    return pixels_per_unit_x, pixels_per_unit_y, src_width, src_height, pixel_width, pixel_height, list_of_bounding_box, xlim, ylim



# Example

In [ ]:
if __name__ == "__main__":
    # Load and parse the XML file for the specified IC
    # folder_path = r"/Users/linkaiyuan/文件/PSU/test_single_sch/A111_Pulsed_Radar_Breakout_SparkFun-Connectors_RASPBERRYPI-26-PIN-GPIO.sch"
    # # folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    # IC_name = "ICM-20948"
    # sch_file_name = f"{folder_path}\\{IC_name}\\{IC_name}.sch"
    # # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"
    # visualize_schematic(folder_path)


    output_dir = '/Users/taitinglu/Desktop/IMG2SCH/test1.png'
    img_path = r"C:\Users\Taiting\Desktop\yolo_training_24\data\ArtemisDevKit.png"
    # sch_file = "F:\GitHub\IMG2SCH\sample\ArtemisDevKit.sch" 
    sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\yolo_training_24\dataset\val\sch\MagJack_Breakout.sch"
    # visualize_schematic_tiled(sch_file, output_dir, rows=3, cols=3, dpi=300)
    # sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\yolo_training_24\dataset\val\sch\Breadboard Power Supply - 5-3.3SMD_V14.sch"
    sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\data\full sch\EasyDriver_-_Stepper_Motor_Driver.sch"
    sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\data\ARGOS_Satellite_Transceiver_Shield_-_ARTIC_R2.sch"
    # sch_file = "/Users/linkaiyuan/Library/Containers/com.tencent.xinWeChat/Data/Documents/xwechat_files/wxid_cpkwgyls8hon32_7a57/msg/file/2025-08/LilyPad_E-Sewing_ProtoSnap.sch"
    sch_file = "/Users/linkaiyuan/文件/PSU/full sch unzip/ARGOS_Satellite_Transceiver_Shield_-_ARTIC_R2.sch"
    sch_file = "/Users/taitinglu/Desktop/IMG2SCH/full sch unzip/IOIO-OTG_-_v2.2.sch"
    visualize_schematic(sch_file, save_path=output_dir,draw_grid=False,draw_bounding_bbox=True)

# Add unit ic to schematic

In [ ]:
import xml.etree.ElementTree as ET

def get_symbol_element(schematic_library, library, deviceset):
    """
    Extracts detailed gate symbols from the schematic information based on the library, deviceset, and device.
    Includes deviceset_prefix in the output.

    Args:
        schematic_library (list): The schematic information containing IC libraries.
        library (str): The name of the library.
        deviceset (str): The name of the deviceset.
        device (str): The name of the device.

    Returns:
        dict: A dictionary with the deviceset_prefix and gate details in the format:
              {deviceset_prefix: prefix, gate_name: {symbol_name: str, description: str, wire: list, text: list}}
    """
    # Extract the library data based on the library name
    library_data = next((lib for lib in schematic_library if lib['name'] == library), None)

    if library_data:
        # Extract the deviceset list
        deviceset_list = library_data.get("devicesets", {}).get("deviceset", [])
        if not isinstance(deviceset_list, list):
            deviceset_list = [deviceset_list]

        # Find the specific deviceset
        deviceset_data = next((ds for ds in deviceset_list if ds['name'] == deviceset), None)
        if deviceset_data:
            # Extract the deviceset prefix
            deviceset_prefix = deviceset_data.get("prefix", "")

            # Extract gates and their corresponding detailed symbols
            gates = deviceset_data.get("gates", {}).get("gate", [])
            if not isinstance(gates, list):
                gates = [gates]

            gate_details = {}
            for gate in gates:
                gate_name = gate['name']
                symbol_name = gate['symbol']

                # Find the detailed symbol in the library
                symbols = library_data.get("symbols", {}).get("symbol", [])
                if not isinstance(symbols, list):
                    symbols = [symbols]

                detailed_symbol = next((sym for sym in symbols if sym['name'] == symbol_name), None)
                if detailed_symbol:
                    gate_details[gate_name] = {
                        "symbol_name": symbol_name,
                        "description": detailed_symbol.get("description", ""),
                        "wire": detailed_symbol.get("wire", []),
                        "text": detailed_symbol.get("text", [])
                    }

            return {
                "deviceset_prefix": deviceset_prefix,
                **gate_details
            }
        else:
            print(f"Deviceset '{deviceset}' not found in library '{library}'.")
    else:
        print(f"Library '{library}' not found.")
    return {}


def get_text_from_symbol(schematic_library, library, deviceset_name, gate_name):
    """
    Extracts the x and y coordinates of all text elements from a symbol in the schematic library.

    Args:
        schematic_library (list): The list of library dictionaries.
        deviceset_name (str): The name of the deviceset to search for.
        device_name (str): The name of the device inside the deviceset.
        library_name (str): The name of the library to match.

    Returns:
        dict: A dictionary where keys are text strings and values are tuples (x, y),
              or an empty dictionary if not found.
    """
    symbol_element = get_symbol_element(schematic_library, library, deviceset_name).get(gate_name, {})
    # print("text_elements:",gate_name,":",symbol_element['text'])
    if symbol_element:
        text_elements = symbol_element.get('text', [])
        result = {}
        for text in text_elements:
            if isinstance(text, dict) and 'text' in text and 'x' in text and 'y' in text:
                result[text['text']] = (float(text['x']), float(text['y']))
            else:
                print(f"Invalid text element format: {text}")
        return result
    return {}


def get_position_for_new_symbol(schematic_info, new_symbol_element, spacing=20, max_width=500):
    """
    Places a new symbol using a bottom-left strategy.
    It wraps to a new column if width exceeds max_width.

    Args:
        schematic_info (dict): Schematic info including parts and IC_library.
        new_symbol_element (dict): Symbol to place.
        spacing (int): Minimum spacing between symbols.
        max_width (int): Maximum horizontal extent allowed.

    Returns:
        list: [x, y] position to place the new symbol (bottom-left placement).
    """
    parts_info = schematic_info.get("parts", {})
    ic_library = schematic_info.get("IC_library", {})

    placed_boxes = []

    min_x = float('inf')

    for part_name, part_data in parts_info.items():
        for instance in part_data.get("instances", []):
            gate = instance.get("gate", "")
            symbol = get_symbol_element_of_instance_from_library(ic_library, part_name, gate, parts_info)
            if not symbol:
                continue

            inst_x = float(instance["x"])
            inst_y = float(instance["y"])
            rot = instance.get("rot", "R0")

            xlim, ylim = get_bounding_box_for_symbol_instance(symbol, inst_x, inst_y, rot)
            placed_boxes.append((xlim, ylim))
            min_x = min(min_x, xlim[0])

    # Default starting point if no symbols placed yet
    if not placed_boxes:
        min_x = 0

    # New symbol dimensions at origin
    symbol_xlim, symbol_ylim = get_bounding_box_for_symbol_instance(new_symbol_element, 0, 0, "R0")
    symbol_width = symbol_xlim[1] - symbol_xlim[0]
    symbol_height = symbol_ylim[1] - symbol_ylim[0]

    # Bottom-left placement within width constraint
    for y in range(0, 10000):  # large enough to cover practical height
        for x in range(int(min_x), int(min_x + max_width - symbol_width) + 1):
            proposed_xlim = [x, x + symbol_width]
            proposed_ylim = [y, y + symbol_height]

            collides = False
            for existing_xlim, existing_ylim in placed_boxes:
                if not (proposed_xlim[1] + spacing <= existing_xlim[0] or
                        proposed_xlim[0] >= existing_xlim[1] + spacing or
                        proposed_ylim[1] + spacing <= existing_ylim[0] or
                        proposed_ylim[0] >= existing_ylim[1] + spacing):
                    collides = True
                    break

            if not collides:
                return [x - symbol_xlim[0], y - symbol_ylim[0]]  # Adjust for symbol origin

    return [0, 0]  # Fallback


def find_unique_part_name(schematic_file, new_symbol_element):
    """
    Finds a unique part name that does not appear in the schematic file's part list.
    The prefix is determined based on the library name.

    Args:
        schematic_file (str): Path to the schematic file.
        library_name (str): Name of the library.
        default_prefix (str): Default prefix for the part name (default is "U").

    Returns:
        str: A unique part name.
    """
    prefix = new_symbol_element.get('deviceset_prefix',"IC")
    # Process the schematic file to get the latest part list
    schematic_info = process_schematic_file(schematic_file)
    part_names = list(schematic_info['parts'].keys())

    # Find a unique part name
    index = 1
    while f"{prefix}{index}" in part_names:
        index += 1
    return f"{prefix}{index}"


def add_unit_to_schematic_full(schematic_file, library, deviceset, device, value, name, x_pos, y_pos):
    """
    Adds a new unit to the schematic file, placing the <part> element
    after the last existing part with a name like 'U[number]' within the <parts> section.

    Args:
        schematic_file (str): Path to the schematic file in XML format.
        library (str): Library name for the part.
        deviceset (str): Deviceset name.
        device (str): Device name.
        value (str): Value of the part.
        name (str): Name of the part.
        x_pos (float): X-coordinate for the instance.
        y_pos (float): Y-coordinate for the instance.
    """
    schematic_info = process_schematic_file(schematic_file)
    schematic_library = schematic_info['IC_library']
    symbol_element = get_symbol_element(schematic_library, library, deviceset)

    # Parse the schematic file
    tree = ET.parse(schematic_file)
    root = tree.getroot()

    # Navigate to the parts and instances sections
    parts_section = root.find(".//parts")
    instances_section = root.find(".//instances")

    if parts_section is None or instances_section is None:
        raise ValueError("Invalid schematic file structure. Missing parts or instances section.")

    # Check if the part already exists
    existing_part = parts_section.find(f"./part[@name='{name}']")
    if existing_part is not None:
        print(f"Part with name '{name}' already exists. Skipping addition.")
        return

    # Find the index to insert the new part
    last_u_part_index = -1
    parts_list = list(parts_section)
    for index, part in enumerate(parts_list):
        if part.tag == 'part' and part.get('name', '').startswith('U') and part.get('name', '')[1:].isdigit():
            last_u_part_index = index

    # Create the new <part> element
    part_element = ET.Element("part", {
        "name": name,
        "library": library,
        "deviceset": deviceset,
        "device": device,
        "value": value
    })
    part_element.tail = "\n"

    if last_u_part_index != -1:
        parts_section.insert(last_u_part_index + 1, part_element)
    else:
        parts_section.insert(0, part_element)

    # Ensure all parts have newlines between them
    for part in parts_section.findall("part"):
        part.tail = "\n"

    # Create attributes and instances for each gate
    for gate_name, gate_data in symbol_element.items():
        if gate_name == "deviceset_prefix":
            continue  # Skip the deviceset_prefix key

        # Create the <instance> element
        instance_element = ET.Element("instance", {
            "part": name,
            "gate": gate_name,
            "x": str(x_pos),
            "y": str(y_pos),
            "smashed": "yes"
        })
        instance_element.text = "\n"   # Add newline after opening <instance>
        instance_element.tail = "\n"   # Add newline after closing </instance>

        # Get text positions for the current gate
        text_pos = get_text_from_symbol(schematic_library, library, deviceset, gate_name)
        for text_key, text_coordinates in text_pos.items():
            if ">" in text_key:
                value_attribute_pos = text_coordinates
                sanitized_text_key = text_key.replace(">", "")

                # Add attributes for each text position
                attribute_element = ET.Element("attribute", {
                    "name": sanitized_text_key,
                    "x": str(int(x_pos - value_attribute_pos[0])),
                    "y": str(int(y_pos - value_attribute_pos[1])),
                    "size": "1.778",
                    "layer": "96",
                    "font": "vector",
                    "align": "top-left"
                })
                attribute_element.tail = "\n"
                instance_element.append(attribute_element)

        # Add instance to <instances>
        instances_section.append(instance_element)

    # Ensure all existing instances and attributes have newlines
    for inst in instances_section.findall("instance"):
        inst.tail = "\n"
        if inst.text is None or not inst.text.strip():
            inst.text = "\n"
        for child in inst:
            child.tail = "\n"

    # Write to file (compact with line-by-line structure)
    tree.write(schematic_file, encoding="utf-8", xml_declaration=True)


def add_unit_to_schematic(schematic_file,library, deviceset, device, value):
    """
    Adds a new unit to the schematic file, placing the <part> element
    after the last existing part with a name like 'U[number]' within the <parts> section.

    Args:
        schematic_file (str): Path to the schematic file in XML format.
        library (str): Library name for the part.
        deviceset (str): Deviceset name.
        device (str): Device name.
        value (str): Value of the part.
    """
    # Determine a unique part name
    schematic_info = process_schematic_file(schematic_file)
    schematic_library = schematic_info['IC_library']

    # Check if the library exists
    library_data = next((lib for lib in schematic_library if lib['name'] == library), None)
    print(library)
    if not library_data:
        print(f"Library '{library}' not found in the schematic library.")
        return

    # Check if the deviceset exists in the library
    deviceset_list = library_data.get("devicesets", {}).get("deviceset", [])
    if not isinstance(deviceset_list, list):
        deviceset_list = [deviceset_list]
    deviceset_data = next((ds for ds in deviceset_list if ds['name'] == deviceset), None)
    if not deviceset_data:
        print(f"Deviceset '{deviceset}' not found in library '{library}'.")
        return

    # Check if the device exists in the deviceset
    device_list = deviceset_data.get("devices", {}).get("device", [])
    if not isinstance(device_list, list):
        device_list = [device_list]
    device_data = next((dev for dev in device_list if dev['name'] == device), None)
    if not device_data:
        print(f"Device '{device}' not found in deviceset '{deviceset}'.")
        return
    
    new_symbol_element = get_symbol_element(schematic_info['IC_library'], library, deviceset)
    part_name = find_unique_part_name(schematic_file, new_symbol_element)

    # Get the position for the new instance
    position = get_position_for_new_symbol(schematic_info, new_symbol_element)
    print(f"Suggested position for new symbol '{part_name}': {position}")
    x_pos = position[0] # Replace with desired x-coordinate
    y_pos = position[1] # Replace with desired y-coordinate

    # Add the unit to the schematic
    add_unit_to_schematic_full(schematic_file, library, deviceset, device, value, part_name, x_pos, y_pos)

In [ ]:
import xml.etree.ElementTree as ET

def get_gate_symbol_for_layout(library_xml, part_name, gate_name, part_library):
    """
    Return the raw symbol dict for a single gate, suitable for layout.

    Args:
        library_xml (list): schematic_info['IC_library']
        part_name (str): the <part>@name from your schematic
        gate_name (str): which gate (e.g. 'A', 'B', etc.) to fetch
        part_library (dict): schematic_info['parts']
    Returns:
        dict: the gate’s symbol data (with 'wire','pin','polygon',…), or None if not found.
    """
    part_info = part_library.get(part_name)
    if not part_info:
        print(f"[get_gate_symbol_for_layout] part '{part_name}' not in library")
        return None

    libname   = part_info.get('library')
    deviceset = part_info.get('deviceset')
    if not libname or not deviceset:
        print(f"[get_gate_symbol_for_layout] missing library/deviceset for '{part_name}'")
        return None

    # get the full gate→symbol map for that deviceset
    full_symbol_map = get_symbol_element(library_xml, libname, deviceset)
    gate_sym = full_symbol_map.get(gate_name)
    if gate_sym is None:
        print(f"[get_gate_symbol_for_layout] gate '{gate_name}' not found in symbol map")
    return gate_sym


def get_position_for_new_gate(schematic_info, gate_symbol, spacing=20, max_width=500):
    """
    Place a single gate using a bottom-left strategy, avoiding collisions with
    existing instances.

    Args:
        schematic_info (dict): Schematic info including 'parts' and 'IC_library'.
        gate_symbol (dict): The symbol definition for one gate.
        spacing (int): Minimum gap between bounding boxes.
        max_width (int): Max horizontal extent before wrapping.

    Returns:
        [x, y]: Coordinates at which to place this gate.
    """
    parts_info = schematic_info.get("parts", {})
    ic_library = schematic_info.get("IC_library", {})
    placed_boxes = []

    # 1) Gather existing instance boxes
    for part_name, part_data in parts_info.items():
        for inst in part_data.get("instances", []):
            sym = get_symbol_element_of_instance_from_library(
                ic_library, part_name, inst.get("gate"), parts_info
            )
            if not sym:
                continue
            ix, iy = float(inst["x"]), float(inst["y"])
            rot = inst.get("rot", "R0")
            xlim, ylim = get_bounding_box_for_symbol_instance(sym, ix, iy, rot)
            placed_boxes.append((xlim, ylim))

    # 2) Determine starting x
    min_x = min((box[0][0] for box in placed_boxes), default=0)

    # 3) Compute this gate’s size at origin
    gate_xlim, gate_ylim = get_bounding_box_for_symbol_instance(gate_symbol, 0, 0, "R0")
    gate_w = gate_xlim[1] - gate_xlim[0]
    gate_h = gate_ylim[1] - gate_ylim[0]

    # 4) Scan for the first free spot
    for y in range(0, 10000, spacing):
        for x in range(int(min_x), int(min_x + max_width - gate_w) + 1, spacing):
            prop_xlim = [x, x + gate_w]
            prop_ylim = [y, y + gate_h]
            collision = False
            for ex_xlim, ex_ylim in placed_boxes:
                if not (
                    prop_xlim[1] + spacing <= ex_xlim[0] or
                    prop_xlim[0] >= ex_xlim[1] + spacing or
                    prop_ylim[1] + spacing <= ex_ylim[0] or
                    prop_ylim[0] >= ex_ylim[1] + spacing
                ):
                    collision = True
                    break
            if not collision:
                # adjust by symbol origin
                return [x - gate_xlim[0], y - gate_ylim[0]]

    # fallback
    return [0, 0]


def add_unit_to_schematic_full(schematic_file, library, deviceset, device, value, name, x_pos, y_pos):
    """
    Adds a new unit to the schematic, placing each gate using collision-aware layout.
    """
    schematic_info   = process_schematic_file(schematic_file)
    schematic_library= schematic_info['IC_library']
    symbol_element   = get_symbol_element(schematic_library, library, deviceset)

    tree = ET.parse(schematic_file)
    root = tree.getroot()
    parts_section    = root.find(".//parts")
    instances_section= root.find(".//instances")
    if parts_section is None or instances_section is None:
        raise ValueError("Missing <parts> or <instances> in schematic.")

    # skip if already exists
    if parts_section.find(f"./part[@name='{name}']") is not None:
        print(f"Part '{name}' exists; skipping.")
        return

    # find insert index for 'U#' parts
    parts_list = list(parts_section)
    last_u = max(
        (i for i,p in enumerate(parts_list)
         if p.tag=='part' and p.get('name','').startswith('U') and p.get('name')[1:].isdigit()),
        default=-1
    )

    # create and insert <part>
    part_el = ET.Element("part", {
        "name": name, "library": library,
        "deviceset": deviceset, "device": device,
        "value": value
    })
    part_el.tail = "\n"
    idx = last_u+1 if last_u!=-1 else 0
    parts_section.insert(idx, part_el)
    for p in parts_section.findall("part"):
        p.tail = "\n"

    # register the new part in schematic_info
    schematic_info.setdefault('parts', {})[name] = {
        'library': library,
        'deviceset': deviceset,
        'device': device,
        'instances': []
    }

    # for each gate, get its symbol and place it
    for gate_name in symbol_element:
        if gate_name == "deviceset_prefix":
            continue

        gate_sym = get_gate_symbol_for_layout(
            schematic_library, name, gate_name, schematic_info['parts']
        )
        if gate_sym is None:
            continue

        # compute non-colliding position
        xg, yg = get_position_for_new_gate(schematic_info, gate_sym)
        print(f"Placing gate '{gate_name}' at ({xg}, {yg}) for part '{name}'")
        inst = ET.Element("instance", {
            "part": name, "gate": gate_name,
            "x": str(xg), "y": str(yg),
            "smashed": "yes"
        })
        inst.text = "\n"
        inst.tail = "\n"

        # build each text attribute at gate-relative offsets
        text_pos = get_text_from_symbol(schematic_library, library, deviceset, gate_name)
        for key, (dx, dy) in text_pos.items():
            if ">" not in key:
                continue

            clean = key.replace(">", "")
            attr = ET.Element("attribute", {
                "name":  clean,
                "x":     str(int(xg + dx)),
                "y":     str(int(yg + dy)),
                "size":  "1.778",
                "layer": "96",
                "font":  "vector",
                "align": "top-left"
            })
            attr.tail = "\n"
            inst.append(attr)

        instances_section.append(inst)

        # update schematic_info so next gate won't overlap
        schematic_info['parts'][name]['instances'].append({
            'gate': gate_name, 'x': xg, 'y': yg, 'rot': 'R0'
        })

    # normalize tails on instances/attrs
    for inst in instances_section.findall("instance"):
        inst.tail = "\n"
        if not (inst.text or "").strip():
            inst.text = "\n"
        for ch in inst:
            ch.tail = "\n"

    tree.write(schematic_file, encoding="utf-8", xml_declaration=True)


## test2

In [ ]:
import xml.etree.ElementTree as ET

def get_gate_symbol_for_layout(library_xml, part_name, gate_name, part_library):
    """
    Return the raw symbol dict for a single gate, suitable for layout.
    """
    part_info = part_library.get(part_name)
    if not part_info:
        print(f"[get_gate_symbol_for_layout] part '{part_name}' not in library")
        return None

    libname   = part_info.get('library')
    deviceset = part_info.get('deviceset')
    if not libname or not deviceset:
        print(f"[get_gate_symbol_for_layout] missing library/deviceset for '{part_name}'")
        return None

    full_symbol_map = get_symbol_element(library_xml, libname, deviceset)
    gate_sym = full_symbol_map.get(gate_name)
    if gate_sym is None:
        print(f"[get_gate_symbol_for_layout] gate '{gate_name}' not found in symbol map")
    return gate_sym


def get_position_for_new_gate(schematic_info, gate_symbol, spacing=20, max_width=500):
    """
    Place a single gate using a bottom-left strategy, avoiding collisions
    with every other instance already in schematic_info.
    """
    parts_info = schematic_info.get("parts", {})
    ic_library = schematic_info.get("IC_library", {})
    placed_boxes = []

    # 1) Gather bounding boxes for every existing instance
    for p_name, p_data in parts_info.items():
        for inst in p_data.get("instances", []):
            # pull the exact gate-symbol for this existing instance
            sym = get_gate_symbol_for_layout(
                ic_library,
                p_name,
                inst.get("gate"),
                parts_info
            )
            if not sym:
                continue

            ix, iy = float(inst["x"]), float(inst["y"])
            rot     = inst.get("rot", "R0")
            xlim, ylim = get_bounding_box_for_symbol_instance(sym, ix, iy, rot)
            placed_boxes.append((xlim, ylim))

    # 2) Find the leftmost x among placed boxes (or 0 if none)
    min_x = min((box[0][0] for box in placed_boxes), default=0)

    # 3) Measure the new gate at origin
    gate_xlim, gate_ylim = get_bounding_box_for_symbol_instance(gate_symbol, 0, 0, "R0")
    gate_w = gate_xlim[1] - gate_xlim[0]
    gate_h = gate_ylim[1] - gate_ylim[0]

    # 4) Scan for the first free slot
    for yy in range(0, 10000, spacing):
        for xx in range(int(min_x), int(min_x + max_width - gate_w) + 1, spacing):
            prop_xlim = [xx,       xx + gate_w]
            prop_ylim = [yy,       yy + gate_h]
            collision = False

            for ex_xlim, ex_ylim in placed_boxes:
                if not (
                    prop_xlim[1] + spacing <= ex_xlim[0] or
                    prop_xlim[0]        >= ex_xlim[1] + spacing or
                    prop_ylim[1] + spacing <= ex_ylim[0] or
                    prop_ylim[0]        >= ex_ylim[1] + spacing
                ):
                    collision = True
                    break

            if not collision:
                # adjust for the symbol’s native origin
                return [xx - gate_xlim[0], yy - gate_ylim[0]]

    # fallback
    return [0, 0]


def add_unit_to_schematic_full(schematic_file, library, deviceset, device, value, name, x_pos, y_pos):
    """
    Adds a new unit to the schematic, placing each gate using collision-aware layout.
    """
    schematic_info    = process_schematic_file(schematic_file)
    ic_lib            = schematic_info['IC_library']
    symbol_element    = get_symbol_element(ic_lib, library, deviceset)

    tree              = ET.parse(schematic_file)
    root              = tree.getroot()
    parts_sec         = root.find(".//parts")
    insts_sec         = root.find(".//instances")
    if parts_sec is None or insts_sec is None:
        raise ValueError("Missing <parts> or <instances> in schematic.")

    # 1) Insert <part> exactly as before
    if parts_sec.find(f"./part[@name='{name}']") is not None:
        print(f"Part '{name}' exists; skipping.")
        return

    parts_list = list(parts_sec)
    last_u = max(
        (i for i,p in enumerate(parts_list)
         if p.tag=='part'
         and p.get('name','').startswith('U')
         and p.get('name')[1:].isdigit()),
        default=-1
    )

    part_el = ET.Element("part", {
        "name":      name,
        "library":   library,
        "deviceset": deviceset,
        "device":    device,
        "value":     value
    })
    part_el.tail = "\n"
    idx = last_u + 1 if last_u != -1 else 0
    parts_sec.insert(idx, part_el)
    for p in parts_sec.findall("part"):
        p.tail = "\n"

    # 2) Register the new part in schematic_info
    schematic_info.setdefault('parts', {})[name] = {
        'library':   library,
        'deviceset': deviceset,
        'device':    device,
        'instances': []
    }

    # 3) For each gate, compute a non‐colliding spot and place it
    for gate_name in symbol_element:
        if gate_name == "deviceset_prefix":
            continue

        gate_sym = get_gate_symbol_for_layout(ic_lib, name, gate_name, schematic_info['parts'])
        if gate_sym is None:
            continue

        xg, yg = get_position_for_new_gate(schematic_info, gate_sym)
        print(f"Placing gate '{gate_name}' at ({xg}, {yg}) for part '{name}'")

        inst = ET.Element("instance", {
            "part":    name,
            "gate":    gate_name,
            "x":       str(xg),
            "y":       str(yg),
            "smashed": "yes"
        })
        inst.text = "\n"
        inst.tail = "\n"

        # 3a) add its attributes
        text_pos = get_text_from_symbol(ic_lib, library, deviceset, gate_name)
        for key, (dx, dy) in text_pos.items():
            clean = key.replace(">", "")
            attr = ET.Element("attribute", {
                "name":  clean,
                "x":     str(int(xg + dx)),
                "y":     str(int(yg + dy)),
                "size":  "1.778",
                "layer": "96",
                "font":  "vector",
                "align": "top-left"
            })
            attr.tail = "\n"
            inst.append(attr)

        insts_sec.append(inst)

        # 3b) record it so that future gates see it
        schematic_info['parts'][name]['instances'].append({
            'gate': gate_name,
            'x':     xg,
            'y':     yg,
            'rot':   'R0'
        })

    # 4) Normalize newlines and write out
    for inst in insts_sec.findall("instance"):
        inst.tail = "\n"
        if not (inst.text or "").strip():
            inst.text = "\n"
        for ch in inst:
            ch.tail = "\n"

    tree.write(schematic_file, encoding="utf-8", xml_declaration=True)


# Example

In [ ]:
if __name__ == "__main__":
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\Module_library"
    # folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    IC_name = "ICM-20948"
    sch_file_name = f"{folder_path}\\{IC_name}\\{IC_name}.sch"
    sch_file_name = r"/Users/linkaiyuan/文件/PSU/template_sch/template.sch"
    library = "SparkFun-Sensors"
    deviceset = 'ICM-20948'
    device = ""
    value = "9DoF IMU"

    # library = "SparkFun-Aesthetics"
    # deviceset = 'FRAME-LEDGER'
    # device = ""
    # value = ""

    # library = "SparkFun-Capacitors"
    # deviceset = '0.1UF'
    # device = "-0402-16V-10%"
    # value = ""
    
    merge_schematic_libraries(source_sch_file, destination_sch_file, debug=False)
    for i in range(5):
        add_unit_to_schematic(sch_file_name,library, deviceset, device, value)
    # add_unit_to_schematic(sch_file_name,library, deviceset, device, value)


    visualize_schematic(sch_file_name)

# Add module to schematic

In [ ]:
import xml.etree.ElementTree as ET


def has_label_in_segment(net_element):
    """
    Checks if a <label> element exists inside a <segment> element under the given <net> element.

    Args:
        net_element (xml.etree.ElementTree.Element): The <net> element to check.

    Returns:
        bool: True if a <label> element exists, False otherwise.
    """
    for segment in net_element.findall("segment"):
        if segment.find("label") is not None:
            return True
    return False


def udpate_net_name(existing_nets, net_name):
    """
    Checks if the given net_name exists in existing_nets and contains a label in its segment.
    If not, generates a new net_name with the prefix N$[number].

    Args:
        existing_nets (dict): Dictionary of existing nets.
        net_name (str): The net name to check.

    Returns:
        str: The original net_name if it contains a label or matches specific names,
             otherwise a new net_name with the prefix N$[number].
    """
    # Check if the net_name is in the special list
    special_nets = {"3.3V", "GND", "1.8V"}
    net_element = existing_nets.get(net_name)
    if net_name in special_nets or has_label_in_segment(net_element):
        return net_name

    # Generate a new net_name with the prefix N$[number]
    existing_net_names = set(existing_nets.keys())
    index = 1
    while f"N${index}" in existing_net_names:
        index += 1
    return f"N${index}"



def merge_schematic_libraries(source_schematic, destination_schematic, debug=False):
    """
    Merges libraries from the source schematic into the destination schematic.
    Ensures line-by-line formatting without indentation.
    """
    if debug:
        print(f"------------Start Merge------------")

    # Parse source and destination schematics
    source_tree = ET.parse(source_schematic)
    source_root = source_tree.getroot()
    destination_tree = ET.parse(destination_schematic)
    destination_root = destination_tree.getroot()

    # Locate the libraries sections
    source_libraries = source_root.find(".//libraries")
    destination_libraries = destination_root.find(".//libraries")

    if source_libraries is None or destination_libraries is None:
        raise ValueError("Invalid schematic file structure. Missing libraries section.")

    if debug:
        print("Located libraries in both source and destination schematics.")

    destination_library_dict = {
        lib.get("name"): lib for lib in destination_libraries.findall("library")
    }

    if debug:
        print(f"Destination libraries found: {list(destination_library_dict.keys())}")

    for source_library in source_libraries.findall("library"):
        source_library_name = source_library.get("name")

        if debug:
            print(f"Processing source library: {source_library_name}")

        if source_library_name in destination_library_dict:
            destination_library = destination_library_dict[source_library_name]

            # Merge packages
            src_pkgs = source_library.find("packages")
            dst_pkgs = destination_library.find("packages")
            if src_pkgs is not None and dst_pkgs is not None:
                dst_pkg_names = {pkg.get("name") for pkg in dst_pkgs.findall("package")}
                for pkg in src_pkgs.findall("package"):
                    if pkg.get("name") not in dst_pkg_names:
                        if debug:
                            print(f"Adding missing package '{pkg.get('name')}'")
                        pkg.tail = "\n"
                        dst_pkgs.append(pkg)

            # Merge symbols
            src_syms = source_library.find("symbols")
            dst_syms = destination_library.find("symbols")
            if src_syms is not None and dst_syms is not None:
                dst_sym_names = {sym.get("name") for sym in dst_syms.findall("symbol")}
                for sym in src_syms.findall("symbol"):
                    if sym.get("name") not in dst_sym_names:
                        if debug:
                            print(f"Adding missing symbol '{sym.get('name')}'")
                        sym.tail = "\n"
                        dst_syms.append(sym)

            # Merge devicesets
            src_dsets = source_library.find("devicesets")
            dst_dsets = destination_library.find("devicesets")
            if src_dsets is not None and dst_dsets is not None:
                dst_dset_names = {ds.get("name") for ds in dst_dsets.findall("deviceset")}
                for dset in src_dsets.findall("deviceset"):
                    if dset.get("name") not in dst_dset_names:
                        if debug:
                            print(f"Adding missing deviceset '{dset.get('name')}'")
                        dset.tail = "\n"
                        dst_dsets.append(dset)
        else:
            if debug:
                print(f"Adding entire library '{source_library_name}' to destination.")
            source_library.text = "\n"
            source_library.tail = "\n"
            destination_libraries.append(source_library)

    # Ensure clean line-by-line formatting for all child elements
    for lib in destination_libraries.findall("library"):
        if lib.text is None or not lib.text.strip():
            lib.text = "\n"
        lib.tail = "\n"
        for section in ['packages', 'symbols', 'devicesets']:
            sub = lib.find(section)
            if sub is not None:
                if sub.text is None or not sub.text.strip():
                    sub.text = "\n"
                sub.tail = "\n"
                for child in sub:
                    child.tail = "\n"

    destination_libraries.tail = "\n"

    # Write final result
    destination_tree.write(destination_schematic, encoding="utf-8", xml_declaration=True)

    if debug:
        print(f"------------Merge completed------------")




def find_unique_part_name_for_module(source_part_names, destination_part_names, old_part_name, new_symbol_element):
    """
    Determines a unique part name to avoid conflicts when merging schematics.

    Args:
        source_part_names (list): Part names from source schematic.
        destination_part_names (list): Part names from destination schematic.
        old_part_name (str): The original part name in source schematic.
        new_symbol_element (dict): Contains deviceset info including prefix.

    Returns:
        str: A part name that avoids conflict.
    """
    if old_part_name not in destination_part_names:
        return old_part_name  # No conflict

    prefix = new_symbol_element.get('deviceset_prefix', 'IC')
    index = 1
    while True:
        new_part_name = f"{prefix}{index}"
        if new_part_name not in source_part_names and new_part_name not in destination_part_names:
            return new_part_name
        index += 1


def get_position_for_schematic_integration(source_sch_file,destination_sch_file, spacing=20, max_width=500, debug=False):
    import copy

    destination_info = process_schematic_file(destination_sch_file)
    source_info = process_schematic_file(source_sch_file)
    # print("Destination schematic info:", source_info['nets'])
    parts_info = destination_info.get("parts", {})
    ic_library = destination_info.get("IC_library", {})

    dest_boxes = []

    for part_name, part_data in parts_info.items():
        for instance in part_data.get("instances", []):
            gate = instance.get("gate", "")
            symbol = get_symbol_element_of_instance_from_library(ic_library, part_name, gate, parts_info)
            if not symbol:
                continue
            inst_x = float(instance["x"])
            inst_y = float(instance["y"])
            rot = instance.get("rot", "R0")
            xlim, ylim = get_bounding_box_for_symbol_instance(symbol, inst_x, inst_y, rot)
            dest_boxes.append((xlim, ylim))

    src_xlim, src_ylim = get_schematic_bounding_box_from_schematic_info(source_info)
    source_width = src_xlim[1] - src_xlim[0]
    source_height = src_ylim[1] - src_ylim[0]

    if debug:
        print("Original source schematic bounding box:", src_xlim, src_ylim)

    if not dest_boxes:
        dx = int(-src_xlim[0])
        dy = int(-src_ylim[0])
    else:
        min_x = min(xlim[0] for xlim, _ in dest_boxes)
        dx, dy = None, None
        for y in range(0, 10000):
            for x in range(int(min_x), int(min_x + max_width - source_width) + 1):
                proposed_xlim = [x, x + source_width]
                proposed_ylim = [y, y + source_height]
                collides = False
                for dest_xlim, dest_ylim in dest_boxes:
                    if not (proposed_xlim[1] + spacing <= dest_xlim[0] or
                            proposed_xlim[0] >= dest_xlim[1] + spacing or
                            proposed_ylim[1] + spacing <= dest_ylim[0] or
                            proposed_ylim[0] >= dest_ylim[1] + spacing):
                        collides = True
                        break
                if not collides:
                    dx = int(x - src_xlim[0])
                    dy = int(y - src_ylim[0])
                    break
            if dx is not None:
                break

        if dx is None:
            dx = int(-src_xlim[0])
            dy = int(-src_ylim[0])

    if debug:
        print("Offset applied to source schematic:", dx, dy)

    updated_parts = {}
    part_name_map = {}
    original_parts = source_info.get("parts", {})
    source_part_names = list(original_parts.keys())
    destination_part_names = list(destination_info.get("parts", {}).keys())

    for part_name, part_data in original_parts.items():
        copied_part_data = copy.deepcopy(part_data)
        new_symbol_element = get_symbol_element(destination_info['IC_library'], part_data['library'], part_data["deviceset"])
        new_part_name = find_unique_part_name_for_module(source_part_names, destination_part_names, part_name, new_symbol_element)
        part_name_map[part_name] = new_part_name
        if debug:
            print(f"Renaming part '{part_name}' to '{new_part_name}'")
        destination_part_names.append(new_part_name)

        for instance in copied_part_data.get("instances", []):
            old_x = instance["x"]
            old_y = instance["y"]
            instance["x"] = str(float(instance["x"]) + dx)
            instance["y"] = str(float(instance["y"]) + dy)
            instance["part"] = new_part_name
            if debug:
                print(f"Updated instance {part_name} from ({old_x}, {old_y}) to ({instance['x']}, {instance['y']})")

            attr_data = instance.get("attributes")
            if isinstance(attr_data, dict):
                attr_list = [attr_data]
            elif isinstance(attr_data, list):
                attr_list = attr_data
            else:
                attr_list = []

            alt_attr_data = instance.get("attribute")
            if isinstance(alt_attr_data, dict):
                attr_list.append(alt_attr_data)
            elif isinstance(alt_attr_data, list):
                attr_list.extend(alt_attr_data)

            for attr in attr_list:
                if "x" in attr:
                    old_x_val = attr["x"]
                    attr["x"] = str(float(attr["x"]) + dx)
                    if debug:
                        print(f"Updated attribute x from {old_x_val} to {attr['x']}")
                if "y" in attr:
                    old_y_val = attr["y"]
                    attr["y"] = str(float(attr["y"]) + dy)
                    if debug:
                        print(f"Updated attribute y from {old_y_val} to {attr['y']}")

        copied_part_data["name"] = new_part_name
        updated_parts[new_part_name] = copied_part_data

    source_info["parts"] = updated_parts

    for netname, net_items in source_info.get("nets", {}).items():
        if debug:
            print(f"Processing net '{netname}'")
        for item in net_items:
            for pinref in item.get("pinref", []):
                old_part = pinref.get("part")
                if old_part in part_name_map:
                    pinref["part"] = part_name_map[old_part]
                    if debug:
                        print(f"Updated pinref part from {old_part} to {pinref['part']}")
            for wire in item.get("wire", []):
                for key in ["x1", "y1", "x2", "y2"]:
                    if key in wire:
                        old_val = wire[key]
                        wire[key] = str(float(wire[key]) + (dx if 'x' in key else dy))
                        if debug:
                            print(f"Updated wire {key} from {old_val} to {wire[key]}")
            for label in item.get("label", []):
                for key in ["x", "y"]:
                    if key in label:
                        old_val = label[key]
                        label[key] = str(float(label[key]) + (dx if key == 'x' else dy))
                        if debug:
                            print(f"Updated label {key} from {old_val} to {label[key]}")
            for junction in item.get("junction", []):
                for key in ["x", "y"]:
                    if key in junction:
                        old_val = junction[key]
                        junction[key] = str(float(junction[key]) + (dx if key == 'x' else dy))
                        if debug:
                            print(f"Updated junction {key} from {old_val} to {junction[key]}")

    return source_info

def add_parts_and_instances_to_sch_file(update_sch_source_info, destination_sch_file):
    """
    Adds parts, instances, and nets from source_info to the destination .sch file (XML format).

    Args:
        destination_sch_file (str): Path to the destination schematic XML file.
        source_info (dict): Updated source_info with parts, instances, and nets.
    """
    import xml.etree.ElementTree as ET

    # Parse destination schematic XML
    tree = ET.parse(destination_sch_file)
    root = tree.getroot()

    parts_section = root.find(".//parts")
    instances_section = root.find(".//instances")
    sheets_section = root.find(".//sheets")
    sheet = sheets_section.find("sheet") if sheets_section is not None else None

    if parts_section is None or instances_section is None or sheet is None:
        raise ValueError("Invalid schematic file structure. Missing <parts>, <instances>, or <sheet> section.")

    nets_section = sheet.find("nets")
    if nets_section is None:
        nets_section = ET.SubElement(sheet, "nets")

    existing_nets = {net.get("name"): net for net in nets_section.findall("net")}

    # Add each part and its instances
    for part_name, part_data in update_sch_source_info.get("parts", {}).items():
        # print(f"Adding part '{part_name}'")
        # print(f"Part data: {part_data}")
        part_element = ET.Element("part", {
            "name": part_name,
            "library": part_data.get("library", ""),
            "deviceset": part_data.get("deviceset", ""),
            "device": part_data.get("device", "")
        })

        # Include additional attributes if present in part_data
        if "value" in part_data:
            part_element.set("value", part_data["value"])
        if "library_urn" in part_data:
            part_element.set("library_urn", part_data["library_urn"])
        if "package3d_urn" in part_data:
            part_element.set("package3d_urn", part_data["package3d_urn"])
        if part_data.get("deviceset") not in ['GND','3.3V','1.8V']:
            part_element.set("value", part_data.get("value", ""))

        part_element.tail = "\n"
        parts_section.append(part_element)

        for instance in part_data.get("instances", []):
            instance_element = ET.Element("instance", {
                "part": instance.get("part", part_name),
                "gate": instance.get("gate", "G$1"),
                "x": instance.get("x", "0"),
                "y": instance.get("y", "0"),
                "smashed": instance.get("smashed", "yes"),
            })
            if "rot" in instance:
                instance_element.set("rot", instance["rot"])
            instance_element.tail = "\n"
            instance_element.text = "\n"

            attr_data = instance.get("attributes")
            if isinstance(attr_data, dict):
                attributes = [attr_data]
            elif isinstance(attr_data, list):
                attributes = attr_data
            else:
                attributes = []

            alt_attr_data = instance.get("attribute")
            if isinstance(alt_attr_data, dict):
                attributes.append(alt_attr_data)
            elif isinstance(alt_attr_data, list):
                attributes.extend(alt_attr_data)

            for attr in attributes:
                attr_element = ET.Element("attribute", {
                    "name": attr.get("name", ""),
                    "x": attr.get("x", "0"),
                    "y": attr.get("y", "0"),
                    "size": attr.get("size", "1.778"),
                    "layer": attr.get("layer", "96")
                })
                if "font" in attr:
                    attr_element.set("font", attr["font"])
                if "align" in attr:
                    attr_element.set("align", attr["align"])
                if "rot" in attr:
                    attr_element.set("rot", attr["rot"])
                attr_element.tail = "\n"
                instance_element.append(attr_element)

            instances_section.append(instance_element)

    # Add nets
    for net_name, segments in update_sch_source_info.get("nets", {}).items():
        existing_nets = {net.get("name"): net for net in nets_section.findall("net")}
        # print(net_name,list(existing_nets.keys()))
        
        if net_name in existing_nets:
            new_net_name = udpate_net_name(existing_nets, net_name)
            # print(f"'{net_name}' to '{new_net_name}'")
            if new_net_name != net_name:
                net_element = ET.Element("net", {"name": new_net_name, "class": "0"})
                net_element.text = "\n"
                net_element.tail = "\n"
                nets_section.append(net_element)
            else:
                net_element = existing_nets[net_name]
        else:
            net_element = ET.Element("net", {"name": net_name, "class": "0"})
            net_element.text = "\n"
            net_element.tail = "\n"
            nets_section.append(net_element)

        for segment in segments:
            segment_element = ET.SubElement(net_element, "segment")
            segment_element.text = "\n"
            segment_element.tail = "\n"

            for pinref in segment.get("pinref", []):
                pinref_element = ET.SubElement(segment_element, "pinref", {
                    "part": pinref.get("part", ""),
                    "gate": pinref.get("gate", ""),
                    "pin": pinref.get("pin", "")
                })
                pinref_element.tail = "\n"

            for wire in segment.get("wire", []):
                wire_element = ET.SubElement(segment_element, "wire", wire)
                wire_element.tail = "\n"

            for label in segment.get("label", []):
                label_element = ET.SubElement(segment_element, "label", label)
                label_element.tail = "\n"

            for junction in segment.get("junction", []):
                junction_element = ET.SubElement(segment_element, "junction", junction)
                junction_element.tail = "\n"

    for inst in instances_section.findall("instance"):
        inst.tail = "\n"
        if inst.text is None or not inst.text.strip():
            inst.text = "\n"

    for part in parts_section.findall("part"):
        part.tail = "\n"

    tree.write(destination_sch_file, encoding="utf-8", xml_declaration=True)
    # print(f"Parts, instances, and nets added to '{destination_sch_file}'")


def add_module_to_sch(source_sch_file, destination_sch_file, spacing=20, max_width=500):
    """
    Merges a module from source schematic to destination schematic.

    Args:
        source_sch_file (str): Path to the source schematic XML file.
        destination_sch_file (str): Path to the destination schematic XML file.
        debug (bool): If True, prints debug information.
    """

    merge_schematic_libraries(source_sch_file, destination_sch_file, debug=False)
    update_sch_source_info = get_position_for_schematic_integration(source_sch_file,destination_sch_file, spacing, max_width,debug=False)
    add_parts_and_instances_to_sch_file(update_sch_source_info,destination_sch_file)

# Example1

In [ ]:
if __name__ == "__main__":
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\Module_library"
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    IC_name = "TMP117MAIDRVT"
    sch_file_name_source = f"{folder_path}\\{IC_name}\\{IC_name}.sch"
    sch_file_name_source = r"C:\Users\Taiting\Desktop\yolo_training_24\data\text.sch"
    sch_file_name_dest = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"
    sch_file_name_dest = r"C:\Users\Taiting\Desktop\yolo_training_24\data\text2.sch"

    add_module_to_sch(sch_file_name_source, sch_file_name_dest)
    img_path = r"C:\Users\Taiting\Desktop\yolo_training_24\data\text.png"
    visualize_schematic(sch_file_name_dest,img_path,draw_grid=False)

# Example2

In [ ]:
if __name__ == "__main__":
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    # List of source schematic files
    sch_file_name_sources = [
        f"{folder_path}\\ICM-20948\\ICM-20948.sch",
        f"{folder_path}\\ICS-41350\\ICS-41350.sch",
        f"{folder_path}\\BMI270\\BMI270.sch",
        f"{folder_path}\\BME680\\BME680.sch",
        f"{folder_path}\\BMA400\\BMA400.sch"
    ]

    sch_file_name_dest = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"

    # Add all source schematic files to the destination schematic
    for sch_file_name_source in sch_file_name_sources:
        add_module_to_sch(sch_file_name_source, sch_file_name_dest, spacing=5, max_width=400)

    # Visualize the final schematic
    visualize_schematic(sch_file_name_dest)


# Clean sch file

In [ ]:
def clean_schematic_file(sch_file_path):
    """
    Removes all libraries, parts, instances, and nets from the given schematic file.

    Args:
        sch_file_path (str): Path to the Eagle .sch file.
    """
    tree = ET.parse(sch_file_path)
    root = tree.getroot()

    drawing = root.find("drawing")
    if drawing is None:
        raise ValueError("Invalid schematic: missing <drawing>")

    schematic = drawing.find("schematic")
    if schematic is None:
        raise ValueError("Invalid schematic: missing <schematic>")

    # Remove all libraries
    libraries = schematic.find("libraries")
    if libraries is not None:
        for lib in list(libraries.findall("library")):
            libraries.remove(lib)

    # Remove all parts
    parts = schematic.find("parts")
    if parts is not None:
        for part in list(parts.findall("part")):
            parts.remove(part)

    # Remove all instances
    sheets = schematic.find("sheets")
    if sheets is not None:
        sheet = sheets.find("sheet")
        if sheet is not None:
            instances = sheet.find("instances")
            if instances is not None:
                for instance in list(instances.findall("instance")):
                    instances.remove(instance)

            # Remove all nets
            nets = sheet.find("nets")
            if nets is not None:
                for net in list(nets.findall("net")):
                    nets.remove(net)

    # Save cleaned schematic
    tree.write(sch_file_path, encoding="utf-8", xml_declaration=True)
    print(f"Cleaned schematic file: {sch_file_path}")


# Example

In [ ]:
if __name__ == "__main__":
    sch_file_name_dest = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"
    clean_schematic_file(sch_file_name_dest)
    visualize_schematic(sch_file_name_dest)

# Preprocessing Schematic file Update label name

In [ ]:
def has_label_in_segment(net_element):
    """
    Checks if a <label> element exists inside a <segment> element under the given <net> element.

    Args:
        net_element (xml.etree.ElementTree.Element): The <net> element to check.

    Returns:
        bool: True if a <label> element exists, False otherwise.
    """
    for segment in net_element.findall("segment"):
        if segment.find("label") is not None:
            return True
    return False



def update_net_names_in_schematic(sch_file_name, ic_name):
    """
    Updates net names in the schematic file by prefixing them with the IC name if they have labels
    and the IC name is not already present in the net name.

    Args:
        sch_file_name (str): Path to the schematic file.
        ic_name (str): Name of the IC to prefix the net names.
    """

    # Parse the schematic file
    tree = ET.parse(sch_file_name)
    root = tree.getroot()

    # Locate the nets section
    sheets_section = root.find(".//sheets")
    sheet = sheets_section.find("sheet") if sheets_section is not None else None
    nets_section = sheet.find("nets") if sheet is not None else None

    if nets_section is None:
        print("No nets section found in the schematic file.")
        return

    # Get existing nets
    existing_nets = {net.get("name"): net for net in nets_section.findall("net")}

    # Update net names
    for net_name, net_element in existing_nets.items():
        # Check if the net has a label in its segments using has_label_in_segment
        if has_label_in_segment(net_element):
            # Only update the net name if ic_name is not already in the net name
            if ic_name not in net_name:
                new_name = f"{ic_name}#{net_name}"
                net_element.set("name", new_name)
                print(f"Updated net name: {net_name} -> {new_name}")
            else:
                print(f"Net name '{net_name}'Skipping update.")

    # Save the updated schematic file
    tree.write(sch_file_name, encoding="utf-8", xml_declaration=True)
    print(f"Updated schematic file saved: {sch_file_name}")


def revert_net_names_in_schematic(sch_file_name, ic_name):
    """
    Reverts net names in the schematic file by removing the IC name prefix if present.

    Args:
        sch_file_name (str): Path to the schematic file.
        ic_name (str): Name of the IC to remove from the net names.
    """

    # Parse the schematic file
    tree = ET.parse(sch_file_name)
    root = tree.getroot()

    # Locate the nets section
    sheets_section = root.find(".//sheets")
    sheet = sheets_section.find("sheet") if sheets_section is not None else None
    nets_section = sheet.find("nets") if sheet is not None else None

    if nets_section is None:
        print("No nets section found in the schematic file.")
        return

    # Get existing nets
    existing_nets = {net.get("name"): net for net in nets_section.findall("net")}

    # Revert net names
    for net_name, net_element in existing_nets.items():
        # Check if the net name starts with the IC name followed by '#'
        prefix = f"{ic_name}#"
        if net_name.startswith(prefix):
            original_name = net_name[len(prefix):]
            net_element.set("name", original_name)
            print(f"Reverted net name: {net_name} -> {original_name}")
        else:
            print(f"Net name '{net_name}' Skipping revert.")

    # Save the updated schematic file
    tree.write(sch_file_name, encoding="utf-8", xml_declaration=True)
    print(f"Reverted schematic file saved: {sch_file_name}")


# Example1

In [ ]:
if __name__ == "__main__":
    # Example usage
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library_net_with_ic_name"
    IC_name = "ICS-41350"
    sch_file_name_source = f"{folder_path}\\{IC_name}\\{IC_name}.sch"
    update_net_names_in_schematic(sch_file_name_source, IC_name)

In [ ]:
import os

def update_net_names_in_all_subfolders(folder_path):
    """
    Iterates through all subfolders in the given folder path, extracts the subfolder name,
    and calls update_net_names_in_schematic for each subfolder.

    Args:
        folder_path (str): Path to the folder containing subfolders with schematic files.
    """
    for subfolder_name in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder_name)
        if os.path.isdir(subfolder_path):  # Check if it's a directory
            sch_file_name_source = os.path.join(subfolder_path, f"{subfolder_name}.sch")
            if os.path.exists(sch_file_name_source):  # Check if the schematic file exists
                print(f"Updating net names for: {sch_file_name_source}")
                update_net_names_in_schematic(sch_file_name_source, subfolder_name)
            else:
                print(f"Schematic file not found in: {subfolder_path}")

# Call the function
if __name__ == "__main__":
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library_net_with_ic_name"
    update_net_names_in_all_subfolders(folder_path)

# Example2

In [ ]:
if __name__ == "__main__":
    # Example usage
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library_net_with_ic_name"
    IC_name = "ICS-41350"
    sch_file_name_source = f"{folder_path}\\{IC_name}\\{IC_name}.sch"
    revert_net_names_in_schematic(sch_file_name_source, IC_name)

# Update attributes and net class

In [ ]:
import xml.etree.ElementTree as ET

def remove_elements_under_attributes(sch_file_path, debug=False):
    """
    Removes all elements under the 'attributes' tag in the schematic file,
    leaving the 'attributes' tag empty.

    Args:
        sch_file_path (str): Path to the Eagle .sch file.
        debug (bool): If True, print debug messages.
    """
    tree = ET.parse(sch_file_path)
    root = tree.getroot()

    attributes_section = root.find(".//attributes")
    if attributes_section is not None:
        for child in list(attributes_section):
            attributes_section.remove(child)
        if debug:
            print("Removed all elements under the 'attributes' tag.")
    else:
        if debug:
            print("No 'attributes' element found in the schematic.")

    tree.write(sch_file_path, encoding="utf-8", xml_declaration=True)
    if debug:
        print(f"Attribute Updated schematic file saved: {sch_file_path}")

def update_net_classes(sch_file, class_number, net_name=None, debug=False):
    """
    Updates the class attribute for all nets or a specific net in the given schematic file.

    Args:
        sch_file (str): Path to the schematic XML file.
        class_number (str): The class number to set for the net(s).
        net_name (str, optional): The name of the specific net to update. If None, updates all nets.
        debug (bool): If True, print debug messages.
    """
    tree = ET.parse(sch_file)
    root = tree.getroot()

    sheets_section = root.find(".//sheets")
    sheet = sheets_section.find("sheet") if sheets_section is not None else None

    if sheet is None:
        raise ValueError("Invalid schematic file structure. Missing <sheet> section.")

    nets_section = sheet.find("nets")
    if nets_section is None:
        raise ValueError("Invalid schematic file structure. Missing <nets> section.")

    if net_name:
        net = nets_section.find(f"net[@name='{net_name}']")
        old_class_number = net.get("class") if net is not None else None
        if net is not None:
            if old_class_number != str(class_number):
                net.set("class", str(class_number))
                if debug:
                    print(f"Net '{net_name}' class '{old_class_number}' updated to '{class_number}'.")
            else:
                if debug:
                    print(f"Net '{net_name}' already has class {class_number}")
        else:
            if debug:
                print(f"Net '{net_name}' not found in the schematic.")
    else:
        for net in nets_section.findall("net"):
            net.set("class", str(class_number))
        if debug:
            print(f"All net classes updated to '{class_number}'.")

    tree.write(sch_file, encoding="utf-8", xml_declaration=True)

def check_net_class_exists(sch_file, class_name, class_number):
    """
    Checks if a net class with the specified name and number exists in the schematic file.

    Args:
        sch_file (str): Path to the schematic XML file.
        class_name (str): The name of the net class to check.
        class_number (int): The number of the net class to check.
        debug (bool): If True, print debug messages.

    Returns:
        bool: True if the net class with the specified name and number exists, False otherwise.
    """
    tree = ET.parse(sch_file)
    root = tree.getroot()

    classes_section = root.find(".//classes")
    if classes_section is None:
        return False

    for net_class in classes_section.findall("class"):
        if net_class.get("name") == class_name and net_class.get("number") == str(class_number):
            return True

    return False

def get_all_net_names(sch_file):
    """
    Extracts all net names from the given schematic file.

    Args:
        sch_file (str): Path to the schematic XML file.
        debug (bool): If True, print debug messages.

    Returns:
        list: A list of net names found in the schematic.
    """
    tree = ET.parse(sch_file)
    root = tree.getroot()

    sheets_section = root.find(".//sheets")
    sheet = sheets_section.find("sheet") if sheets_section is not None else None

    if sheet is None:
        raise ValueError("Invalid schematic file structure. Missing <sheet> section.")

    nets_section = sheet.find("nets")
    if nets_section is None:
        raise ValueError("Invalid schematic file structure. Missing <nets> section.")

    net_names = [net.get("name") for net in nets_section.findall("net")]
    return net_names

import xml.etree.ElementTree as ET

def add_net_class_to_schematic(sch_file, class_number, class_name, width="0.1016", drill="0", debug=False):
    """
    Adds a new net class to the <classes> section of the schematic file.

    Args:
        sch_file (str): Path to the schematic XML file.
        class_number (str): The class number for the new net class.
        class_name (str): The name of the new net class.
        width (str): The width attribute for the new net class.
        drill (str): The drill attribute for the new net class.
        debug (bool): If True, print debug messages.
    """
    tree = ET.parse(sch_file)
    root = tree.getroot()

    classes_section = root.find(".//classes")
    if classes_section is None:
        schematic = root.find(".//schematic")
        if schematic is None:
            raise ValueError("Invalid schematic file structure. Missing <schematic> section.")
        classes_section = ET.SubElement(schematic, "classes")
        classes_section.text = "\n"

    for existing_class in classes_section.findall("class"):
        if existing_class.get("number") == str(class_number):
            if debug:
                print(f"Class with number '{class_number}' already exists. Skipping addition.")
            return False
        if existing_class.get("name") == class_name:
            if debug:
                print(f"Class with name '{class_name}' already exists. Skipping addition.")
            return False

    new_class = ET.Element("class", {
        "number": str(class_number),
        "name": class_name,
        "width": str(width),
        "drill": str(drill)
    })
    new_class.text = "\n"   # Add line break inside <class>...</class>
    new_class.tail = "\n"   # Add line break after </class>

    classes_section.append(new_class)

    tree.write(sch_file, encoding="utf-8", xml_declaration=True, short_empty_elements=False)
    return True


def delete_net_class_from_schematic(sch_file, class_name, debug=False):
    """
    Deletes a net class with the specified class name from the <classes> section of the schematic file.

    Args:
        sch_file (str): Path to the schematic XML file.
        class_name (str): The name of the net class to delete.
        debug (bool): If True, print debug messages.
    """
    tree = ET.parse(sch_file)
    root = tree.getroot()

    classes_section = root.find(".//classes")
    if classes_section is None:
        if debug:
            print("No <classes> section found in the schematic file.")
        return

    class_found = False
    for net_class in classes_section.findall("class"):
        if net_class.get("name") == class_name:
            classes_section.remove(net_class)
            class_found = True
            if debug:
                print(f"Deleted net class '{class_name}' from the schematic file.")
            break

    if not class_found and debug:
        print(f"Net class '{class_name}' not found in the schematic file.")

    tree.write(sch_file, encoding="utf-8", xml_declaration=True)
    if debug:
        print(f"Updated schematic file saved: {sch_file}")

def delete_all_net_classes_except_default(sch_file, default_class_name, debug=False):
    """
    Deletes all net classes from the <classes> section of the schematic file,
    except for the specified default class name.

    Args:
        sch_file (str): Path to the schematic XML file.
        default_class_name (str): The name of the net class to retain.
        debug (bool): If True, print debug messages.
    """
    tree = ET.parse(sch_file)
    root = tree.getroot()

    classes_section = root.find(".//classes")
    if classes_section is None:
        if debug:
            print("No <classes> section found in the schematic file.")
        return

    for net_class in list(classes_section.findall("class")):
        if net_class.get("name") != default_class_name:
            classes_section.remove(net_class)
            if debug:
                print(f"Deleted net class '{net_class.get('name')}' with '{net_class.get('number')}' from the schematic file.")

    tree.write(sch_file, encoding="utf-8", xml_declaration=True)

def update_net_classes_for_list(sch_file, class_number, class_name, net_names, debug=False):
    """
    Updates the class attribute for the specified list of nets in the given schematic file.

    Args:
        sch_file (str): Path to the schematic XML file.
        class_number (str): The class number to set for the nets.
        class_name (str): The name of the net class.
        net_names (list): List of net names to update.
        debug (bool): If True, print debug messages.
    """
    if check_net_class_exists(sch_file, class_name, class_number):
        for net_name in net_names:
            update_net_classes(sch_file, class_number, net_name, debug)
    else:
        if debug:
            print(f"Net class {class_name} does not exist.")


def polish_net_class_and_name(sch_file, debug =False):
    """
    Polishes the net class and name in the schematic file.

    Args:
        sch_file (str): Path to the schematic XML file.
        class_name (str): The name of the net class to polish.
    """
    remove_elements_under_attributes(sch_file,debug)
    net_names = get_all_net_names(sch_file)
    class_name = 'default'
    class_number = 0
    add_net_class_to_schematic(sch_file, class_number, class_name, debug=debug)
    update_net_classes_for_list(sch_file, class_number, class_name, net_names, debug)
    delete_all_net_classes_except_default(sch_file, class_name,debug)


# Example

In [ ]:
if __name__ == "__main__":
    folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    IC_name = "XC6222B331MR-G"
    sch_file_name_source = f"{folder_path}\\{IC_name}\\{IC_name}.sch"
    polish_net_class_and_name(sch_file_name_source,debug=True)

# Check if package3d exist in sch

In [ ]:
import os

def check_package3d_in_sch(folder_name):
    """
    Iterates through subfolders in the given folder, retrieves .sch files based on subfolder names,
    and checks if 'package3d' exists in the .sch files.

    Args:
        folder_name (str): Path to the folder containing subfolders.

    Returns:
        list: A list of .sch file paths where 'package3d' exists.
    """
    sch_with_package3d = []
    for subfolder in os.listdir(folder_name):
        subfolder_path = os.path.join(folder_name, subfolder)
        if os.path.isdir(subfolder_path):  # Check if it's a subfolder
            sch_file_path = os.path.join(subfolder_path, f"{subfolder}.sch")
            if os.path.exists(sch_file_path):  # Check if the .sch file exists
                with open(sch_file_path, 'r', encoding='utf-8', errors='ignore') as sch_file:
                    content = sch_file.read()
                    if 'package3d' in content:
                        sch_with_package3d.append(sch_file_path)
    return sch_with_package3d



# Example

In [ ]:
if __name__ == "__main__":
    # Example usage
    folder_name = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    sch_with_package3d = check_package3d_in_sch(folder_name)
    print("Schematic files containing 'package3d':")
    for sch_file in sch_with_package3d:
        print(sch_file)

# Remove all 3d package from sch

In [ ]:
import xml.etree.ElementTree as ET

def remove_packages3d_from_sch(sch_file):
    """
    Removes the 'packages3d' element, 'package3dinstances', and attributes containing 'package3d'
    from the specified .sch file.

    Args:
        sch_file (str): Path to the .sch file.

    Returns:
        None
    """
    try:
        # Parse the XML file
        tree = ET.parse(sch_file)
        root = tree.getroot()

        # Navigate to the 'packages3d' element
        libraries = root.find("./drawing/schematic/libraries")
        if libraries is not None:
            for library in libraries.findall("library"):
                # Remove 'packages3d' element
                packages3d = library.find("packages3d")
                if packages3d is not None:
                    library.remove(packages3d)
                    print(f"Removed 'packages3d' from library: {library.get('name')}")

                # Remove 'package3dinstances' element
                devicesets = library.find("devicesets")
                if devicesets is not None:
                    for deviceset in devicesets.findall("deviceset"):
                        devices = deviceset.find("devices")
                        if devices is not None:
                            for device in devices.findall("device"):
                                package3dinstances = device.find("package3dinstances")
                                if package3dinstances is not None:
                                    device.remove(package3dinstances)
                                    print(f"Removed 'package3dinstances' from device in library: {library.get('name')}")

        # Navigate to the 'parts' section and remove attributes containing 'package3d' or named 'override_package_urn'
        parts = root.find("./drawing/schematic/parts")
        if parts is not None:
            for part in parts.findall("part"):
                attributes_to_remove = [attr for attr in part.attrib if "package3d" in attr.lower() or attr == "override_package_urn"]
                for attr in attributes_to_remove:
                    del part.attrib[attr]
                    print(f"Removed attribute '{attr}' from part: {part.get('name')}")

        # Save the modified XML back to the file
        tree.write(sch_file, encoding="utf-8", xml_declaration=True)
        print(f"Updated .sch file saved: {sch_file}")

    except Exception as e:
        print(f"Error processing .sch file: {e}")



# Example

In [ ]:
if __name__ == "__main__":
    # Example usage
    sch_file_name_dest = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library\USB4110-GF-A\USB4110-GF-A.sch"
    remove_packages3d_from_sch(sch_file_name_dest)

In [ ]:
import glob
import zipfile
import os

root_dir = r"/Users/linkaiyuan/文件/PSU/products"
for zip_path in glob.glob(os.path.join(root_dir, "**", "*.zip"), recursive=True):
    print(zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(zip_path))
    

In [ ]:
root_dir = r"/Users/linkaiyuan/文件/PSU/products"
all_sch_files = []
count = 0
for sch_file_path in glob.glob(os.path.join(root_dir, "**", "*.sch"), recursive=True):
    # all_sch_files.append(process_schematic_file(sch_file_path))
    try:
        all_sch_files.append(process_schematic_file(sch_file_path))
    except Exception as e:
        print(f"Skipping {sch_file_path} due to error: {e}")
        count += 1
        continue

In [ ]:
print(count)
all_parts = []
for i in all_sch_files:
    if isinstance(i['parts'], dict):
        all_parts.append(i['parts'])
    elif isinstance(i['parts'], list):
        all_parts.extend(i['parts'])

In [ ]:
import json
from collections import Counter
import pandas as pd

# 1. Load JSON data
with open("text1.json", "r") as f:
    data = json.load(f)

counter = Counter()

# 2. Iterate through each schematic’s parts and build keys
for schematic in data:
    for part_ref, part_info in schematic.items():
        lib       = part_info.get("library", "").lower()
        deviceset = part_info.get("deviceset", "")
        package   = part_info.get("package", "")

        # 2a. If it’s a resistor, group under ("resistor",)
        if "resistor" in lib:
            key = ("resistor",)

        # 2b. If it’s a capacitor, group under ("capacitor",)
        elif "capacitor" in lib:
            key = ("capacitor",)

        # 2c. Otherwise, use (library, deviceset, package) as the key
        else:
            key = (lib, deviceset, package)

        counter[key] += 1

# 3. Convert counts into a DataFrame
rows = []
for key, count in counter.items():
    if len(key) == 1:
        # All resistors → show as “Group/Resistor/US”
        # All capacitors → show as “Group/Capacitor/EU”
        group_tag = key[0]  # either "resistor" or "capacitor"
        if group_tag == "resistor":
            rows.append({
                "Library/Group":      "Group",
                "DeviceSet/Category": "Resistor",
                "Package/Group":      "US",
                "Count":              count
            })
        else:
            rows.append({
                "Library/Group":      "Group",
                "DeviceSet/Category": "Capacitor",
                "Package/Group":      "EU",
                "Count":              count
            })
    else:
        lib, deviceset, package = key
        rows.append({
            "Library/Group":      lib,
            "DeviceSet/Category": deviceset,
            "Package/Group":      package,
            "Count":              count
        })

df = pd.DataFrame(rows).sort_values(by="Count", ascending=False)

# 4a. Print top 20 rows to console
print("\n—— Top 20 component groups by count ——\n")
print(df.head(20).to_string(index=False))

# 4b. Save the full table to CSV
output_csv = "component_counts.csv"
df.to_csv(output_csv, index=False)
print(f"\n✔ Full results written to {output_csv}")


# Extract IC from Schematic

In [ ]:
## /Users/linkaiyuan/文件/PSU/products/SparkFun_Environmental_Combo_Breakout_-_ENS160_BME
if __name__ == "__main__":
    # <pinref part="C2" gate="G$1" pin="1"/>
    # <wire x1="53.34" y1="220.98" x2="53.34" y2="210.82" width="0.1524" layer="91"/>
    # Output elements from a file according the path
    # Load and parse the XML file for the specified IC
    folder_path = r"/Users/linkaiyuan/文件/PSU/products/SparkFun_Environmental_Combo_Breakout_-_ENS160_BME/SparkFun_ENS160_BME280_Breakout.sch"
    # folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    # IC_name = "ICM-20948"
    # sch_file_name = f"{folder_path}/{IC_name}/{IC_name}.sch"
    sch_file_name = folder_path
    # print("schematic file name:", sch_file_name)
    # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"
    # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\temp_add.sch"
    schematic_info = process_schematic_file(sch_file_name)

### Save the generated json obj

In [ ]:
schematic_info['parts']

import json

with open("text3.json", "w") as f:
    json.dump(schematic_info, f, indent=2)
# print keys of librarys

# single part generate (ori, not working for instances list, only instance json)


In [ ]:
from xml.etree.ElementTree import ParseError
import os
import stat
if __name__ == "__main__":

    root_dir = "/Users/linkaiyuan/文件/PSU/products"
    temp_path = "/Users/linkaiyuan/文件/PSU/template_sch/template.sch"
    # os.makedirs(temp_path, exist_ok=True)
    os.chmod(temp_path, stat.S_IRWXU)
    os.chmod(root_dir, stat.S_IRWXU)
    import shutil
    import glob
    import os
    import json
    import stat

    unique_symbols = set()
    count = 0

    for sch_file_path in glob.glob(os.path.join(root_dir, "**", "*.sch"), recursive=True):
        # if count > 1:
        #     break
        name = os.path.basename(sch_file_path)
        stem = os.path.splitext(name)[0]
        os.chmod(sch_file_path, stat.S_IRWXU)
        try:
            schematic_info = process_schematic_file(sch_file_path)
            for part_name in schematic_info['parts'].items():
                # count += 1
                # if count > 1:
                #     break
                # print(part_name)
                symbol = part_name[1]['instances'][0]['symbol']
                lib = part_name[1]['library']
                if (symbol, lib) in unique_symbols:
                    continue
                unique_symbols.add((symbol, lib))

                new_name = f"{os.path.splitext(name)[0]}_{lib}_{symbol}.sch".replace("/", "slash")
                if ("MGM240P_EFM32GG12B410F1024GL120_EFM32GG12B410F1024GL120_A" in new_name):
                    print("name find:",sch_file_path)
                # dst_path = os.path.join("/Users/linkaiyuan/文件/PSU/test_single_sch", new_name)
                dst_path = os.path.join("/Users/linkaiyuan/文件/PSU/test_single_sch", new_name)
                # os.chmod(dst_path, stat.S_IRWXU))
                if os.path.exists(dst_path):
                    os.chmod(dst_path, stat.S_IRWXU)
                shutil.copy(temp_path, dst_path)
                os.chmod(dst_path, stat.S_IRWXU)
                # print(dst_path)

                    
                deviceset = part_name[1]['deviceset']
                device = part_name[1]['device']
                value = part_name[1]['value']
                print(sch_file_path, dst_path, lib, deviceset, device, value)
                merge_schematic_libraries(sch_file_path, dst_path, debug=False)
                add_unit_to_schematic(dst_path,lib, deviceset, device, value)
                print(f"Orgional schematic file: {sch_file_path}")
                print(f"New schematic file: {dst_path}")
                print(f"added unit: {lib}, {deviceset}, {device}, {value}")
    # # device = ""
    # # value = ""
        except AttributeError as e:
            print(f"Skipping {sch_file_path} due to error: {e}")
            continue
        except ParseError as e:
            print(f"Skipping {sch_file_path} due to XML parsing error: {e}")
            continue
        except TypeError as e:
            print(f"Skipping {sch_file_path} due to TypeError: {e}")
            continue


# single part generate (Latest for multi instances)

In [ ]:
import os, glob, shutil, stat
from xml.etree.ElementTree import ParseError

if __name__ == "__main__":
    root_dir   = "/Users/linkaiyuan/文件/PSU/full sch unzip"
    temp_path  = "/Users/linkaiyuan/文件/PSU/template_sch/template.sch"
    out_dir    = "/Users/linkaiyuan/文件/PSU/test/multi_instances8_7"
    os.makedirs(out_dir, exist_ok=True)
    part = {}
    unique_keys = set()
    for sch_path in glob.glob(os.path.join(root_dir, "**", "*.sch"), recursive=True):
        try:
            schematic_info = process_schematic_file(sch_path)
        except (AttributeError, ParseError, TypeError, ValueError):
            continue

        stem = os.path.splitext(os.path.basename(sch_path))[0]
        part_dict = schematic_info['parts']
        part = {}
        # iterate over every part...
        for part_name, part_info in part_dict.items():
            if part_name in part:
                
                continue
            part[part_name] = part_info
            print("Here is part_name:", part_name)
            lib       = part_info['library']
            deviceset = part_info['deviceset']
            device    = part_info['device']
            value     = part_info['value']

            # now iterate _all_ instances
            for idx, inst in enumerate(part_info.get('instances', [])):
                symbol = inst['symbol']

                # if you still want to dedupe by (lib,symbol) you can:
                key = (lib, symbol, idx)
                if key in unique_keys:
                    continue
                unique_keys.add(key)

                # build a unique filename per instance
                new_name = f"{stem}#{lib}#{symbol}.sch".replace("/", "slash")
                dst = os.path.join(out_dir, new_name)

                # copy your template & merge/add as before
                os.chmod(temp_path, stat.S_IRWXU)
                shutil.copy(temp_path, dst)
                os.chmod(dst,       stat.S_IRWXU)

                # merge & add one unit
                merge_schematic_libraries(sch_path, dst, debug=False)
                add_unit_to_schematic(dst, lib, deviceset, device, value)

                print(f"""
    Source:    {sch_path}
    Instance:  [{idx}] symbol={symbol}, lib={lib}, deviceset={deviceset}, device={device}, value={value}
    Written to: {dst}
    """)


# Generate Training Data (Images for each single symbols)

### Function to filter out the blank symols

In [ ]:
from PIL import Image
import numpy as np
import os
import shutil


def is_blank_image(path, white_tol=250, blank_frac=0.99):
    """
    Returns True if more than blank_frac of the pixels are
    (R,G,B) >= white_tol.
    """
    im = Image.open(path).convert("RGB")
    arr = np.array(im)
    # mask of pixels that are “almost white”
    white = (arr[:,:,0] >= white_tol) & (arr[:,:,1] >= white_tol) & (arr[:,:,2] >= white_tol)
    return white.mean() >= blank_frac

### loop over single instances to generate single symbol images

In [ ]:
import glob
import os
import shutil


folder_path = r"/Users/linkaiyuan/文件/PSU/test/multi_instances8_7"

# img_dir   = r"/Users/linkaiyuan/文件/PSU/problem_image_debug/"
img_dir     = r"/Users/linkaiyuan/文件/PSU/test/train/images"
lbl_dir     = r"/Users/linkaiyuan/文件/PSU/test/train/labels"
sch_store_dir = r"/Users/linkaiyuan/文件/PSU/test/train/sch"




# save_path = "/Users/linkaiyuan/Downloads/"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/Bus_Pirate_-_v3.6a/BusPirate-v3.6a.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/multi_instances/MicroMod_Processor_STM32WB5MMG_stm32wb5mmg_STM32WB5MMG_COMMS.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_Qwiic_Buzzer/SparkFun_Qwiic_Buzzer.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_MicroMod_WiFi_Function_Board_-_DA16200/SparkFun_MicroMod_DA16200_Function.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/WAV_Trigger/wavtrigger_v11.sch"


limit = 50

for p in (folder_path, img_dir, lbl_dir, sch_store_dir):
    os.makedirs(p, exist_ok=True)

class_order = 0
class_map   = {}
with open("/Users/linkaiyuan/文件/PSU/wrong_image_list.txt", "r") as f:
        wrong_list = [os.path.splitext(os.path.basename(line.strip()))[0] for line in f]
# print(wrong_list)

base = ""


for sch_file in glob.glob(os.path.join(folder_path, "*.sch")):
    # if os.path.splitext(os.path.basename(sch_file))[0] not in wrong_list:
    #         # print(f"Skipping {os.path.splitext(os.path.basename(sch_file))[0]} as it is not in the wrong list.")
    #     continue
    limit = limit - 1
    # if limit == 0:
    #     break
    try:
        # parse schematic and get its full‐sheet bounds
        schematic_info = process_schematic_file(sch_file)
        src_xlim, src_ylim = get_schematic_bounding_box_from_schematic_info(schematic_info)
        # print(sch_file)
        # render and save
        # png_path = os.path.join(img_dir, "version_with_bounding" + os.path.basename(sch_file).replace(".sch", ".png"))
        base = os.path.splitext(os.path.basename(sch_file))[0]
        png_path = os.path.join(img_dir, base + ".png")
        sch_file_path = os.path.join(sch_store_dir, base + ".sch")
        # sizes    = visualize_schematic(sch_file, png_path)
        sizes   = visualize_schematic(sch_file, save_path=png_path)
        px_per_x, px_per_y, _, _, img_w, img_h = sizes
        if is_blank_image(png_path):
                print(f"Blank → removing {base}")
                os.remove(png_path)
                continue
        else:
                print(f"Kept → {base}")
                shutil.copy(sch_file, sch_file_path)

        key = '#'.join(base.split('#')[1:])
        
        if key not in class_map:
                class_map[key] = class_order
                class_order += 1
        cls_id = class_map[key]
        png_path = os.path.join(img_dir, base + ".png")

        label_file = ""
        ic_library = schematic_info.get('IC_library')
        parts_info = schematic_info.get('parts')

        # loop over each part instance
        for part_name, part_data in parts_info.items():
            for instance in part_data.get('instances', []):
                symbol_element = get_symbol_element_of_instance_from_library(
                    ic_library, part_name, instance.get('gate', ''), parts_info
                )
                if not symbol_element:
                    continue

                # get this symbol’s world‐space bbox
                rot_inst = instance['rot']
                x_inst   = float(instance['x'])
                y_inst   = float(instance['y'])
                (x0, x1), (y0, y1) = get_bounding_box_for_symbol_instance(
                    symbol_element, x_inst, y_inst, rot_inst
                )

                # convert to pixels
                w_px  = (x1 - x0) * px_per_x
                h_px  = (y1 - y0) * px_per_y
                cx_px = ((x0 + x1)/2 - src_xlim[0]) * px_per_x
                cy_px = ((y0 + y1)/2 - src_ylim[0]) * px_per_y

                # normalize for YOLO (flip y so origin is top‐left)
                norm_x_center = cx_px / img_w
                norm_y_center = 1.0 - (cy_px / img_h)
                norm_width    = w_px    / img_w
                norm_height   = h_px    / img_h

                # lookup class
                lib          = part_data['library']
                symbol       = instance['symbol']
                base_name    = os.path.splitext(os.path.basename(sch_file))[0]
                key          = f"{lib}#{symbol}"
                if class_map is not None:
                    class_number = class_map.get(key, -1)
                    if class_number == -1:
                        continue

                label_file += f"{class_number} {norm_x_center:.6f} {norm_y_center:.6f} {norm_width:.6f} {norm_height:.6f}\n"

        # write out labels
        label_filename = os.path.join(lbl_dir, base + ".txt")
        with open(label_filename, "w") as f:
            f.write(label_file)

    except Exception as e:
        print(f"Skipping {sch_file} due to error: {e}")

### Generate yaml file for yolo training

In [ ]:
import yaml

dataset_root = "/Users/linkaiyuan/文件/PSU/test/"
data_yaml_path = os.path.join(dataset_root, "data.yaml")
class_map['SheetInfo'] = class_order
class_map['Wire'] = class_order + 1
index_to_name = {v: k for k, v in class_map.items()} 

data_yaml = {
    "path": "/Users/linkaiyuan/文件/PSU/test/",
    "train": "train",
    "val": "val",
    "names": index_to_name
}

with open("/Users/linkaiyuan/文件/PSU/test/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False, allow_unicode=True)


## generate img and label for all full sch

In [ ]:
import glob
import os
import traceback

save_path = "/Users/linkaiyuan/Downloads/"
sch_file = '/Users/linkaiyuan/文件/PSU/products/SparkFun_Artemis_Development_Kit_with_Camera/ArtemisDevKit.sch'
# sch_file = "/Users/linkaiyuan/文件/PSU/multi_instances/MicroMod_Processor_STM32WB5MMG_stm32wb5mmg_STM32WB5MMG_COMMS.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_Qwiic_Buzzer/SparkFun_Qwiic_Buzzer.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_MicroMod_WiFi_Function_Board_-_DA16200/SparkFun_MicroMod_DA16200_Function.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/WAV_Trigger/wavtrigger_v11.sch"


try:
    # parse schematic and get its full‐sheet bounds
    schematic_info = process_schematic_file(sch_file)
    src_xlim, src_ylim = get_schematic_bounding_box_from_schematic_info(schematic_info)
    print(sch_file)
    # render and save
    png_path = os.path.join(save_path, "version_with_bounding" + os.path.basename(sch_file).replace(".sch", ".png"))
    sizes    = visualize_schematic(sch_file, png_path)
    # sizes   = visualize_schematic_nobounding(sch_file, save_path=png_path)
    px_per_x, px_per_y, _, _, img_w, img_h, bounding_box_list = sizes


    ic_library = schematic_info.get('IC_library')
    parts_info = schematic_info.get('parts')

    # loop over each part instance
    for box in bounding_box_list:
        
            symbol_element = get_symbol_element_of_instance_from_library(
                ic_library, part_name, instance.get('gate', ''), parts_info
            )
            if not symbol_element:
                continue

            # get this symbol’s world‐space bbox
            part_name = box['part_name']



            x_center = box['x_center']
            y_center = box['y_center']

            box_length = box['length']
            box_width = box['width']

            x0 = x_center - box_length/2
            y0 = y_center - box_width/2

            x1 = x_center + box_length/2
            y1 = y_center + box_width/2

            # rot_inst = instance['rot']
            # x_inst   = float(instance['x'])
            # y_inst   = float(instance['y'])
            # (x0, x1), (y0, y1) = get_bounding_box_for_symbol_instance(
            #     symbol_element, x_inst, y_inst, rot_inst
            # )

            # convert to pixels
            w_px  = (x1 - x0) * px_per_x
            h_px  = (y1 - y0) * px_per_y
            cx_px = ((x0 + x1)/2 - src_xlim[0]) * px_per_x
            cy_px = ((y0 + y1)/2 - src_ylim[0]) * px_per_y

            # normalize for YOLO (flip y so origin is top‐left)
            norm_x_center = cx_px / img_w
            norm_y_center = 1.0 - (cy_px / img_h)
            norm_width    = w_px    / img_w
            norm_height   = h_px    / img_h

            # lookup class
            # lib          = part_data['library']
            # symbol       = instance['symbol']
            # base_name    = os.path.splitext(os.path.basename(sch_file))[0]
            # key          = f"{base_name}_{lib}_{symbol}"


            if part_name == 'SheetInfo':
                key = "SheetInfo"
            elif box.get('cat', '') == 'symbol':
                key =  box.get('key', '')
            else:
                key = 'Wire'

            if class_map is not None:
                class_number = class_map.get(key, -1)
                if class_number == -1:
                    continue

            label_file += f"{class_number} {norm_x_center:.6f} {norm_y_center:.6f} {norm_width:.6f} {norm_height:.6f}\n"

    # write out labels
    # label_filename = 
    with open(label_filename, "w") as f:
        f.write(label_file)

except Exception as e:
    print(f"Skipping {sch_file} due to error: {e}")
    traceback.print_exc()

# generate Images for full sch files

## Preprocess eagle.zip

In [41]:
import os

def check_zip_in_subfolders(folder_path):
    """
    Checks if each subfolder inside folder_path contains a 'eagle.zip' file.
    Returns a list of subfolders that contain 'eagle.zip'.
    """
    result = []
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            expected_zip = os.path.join(subfolder_path, "eagle.zip")
            if os.path.isfile(expected_zip):
                result.append(subfolder)
    return result


def unzip_eagle_zips_in_subfolders(folder_path):
    """
    Unzips 'eagle.zip' in each subfolder of the given folder_path.
    After unzipping, prints a message if .sch file is not found in the subfolder.
    If no .sch file is found but a folder is present, prints a message.
    """
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        print(f"Processing subfolder: {subfolder}")
        if subfolder not in ['.DS_Store'] and os.path.isdir(subfolder_path):
            zip_path = os.path.join(subfolder_path, "eagle.zip")
            if os.path.isfile(zip_path):
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(subfolder_path)
                sch_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith('.sch')]
                if not sch_files:
                    # Check for new folders created after unzip
                    new_folders = [f for f in os.listdir(subfolder_path) if os.path.isdir(os.path.join(subfolder_path, f))]
                    # Ignore folders named 'MAXOSX'
                    new_folders = [f for f in new_folders if f.upper() != "_MAXOSX"]
                    if new_folders:
                        print(f"No .sch file found, but new folder(s) found in: {subfolder_path} → {new_folders}")
                        # Move all .sch and .brd files from new folders to subfolder_path
                        for nf in new_folders:
                            nf_path = os.path.join(subfolder_path, nf)
                            for file in os.listdir(nf_path):
                                if file.lower().endswith(('.sch', '.brd')):
                                    src = os.path.join(nf_path, file)
                                    dst = os.path.join(subfolder_path, file)
                                    try:
                                        os.rename(src, dst)
                                        print(f"Moved {file} from {nf_path} to {subfolder_path}")
                                    except Exception as e:
                                        print(f"Failed to move {file}: {e}")
                    else:
                        print(f"No .sch file found in: {subfolder_path}")
            else:
                print(f"no eagle.zip in: {subfolder_path}")
            # After moving, check again for .sch and .brd files in subfolder_path
            sch_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith('.sch')]
            brd_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith('.brd')]
            if len(sch_files) == 1 and len(brd_files) == 1:
                # Only one .sch and one .brd file found, delete all other files and folders
                keep_files = {sch_files[0], brd_files[0]}
                for item in os.listdir(subfolder_path):
                    item_path = os.path.join(subfolder_path, item)
                    if item not in keep_files:
                        try:
                            if os.path.isfile(item_path):
                                os.remove(item_path)
                            elif os.path.isdir(item_path):
                                shutil.rmtree(item_path)
                            print(f"Deleted: {item_path}")
                        except Exception as e:
                            print(f"Failed to delete {item_path}: {e}")
            else:
                print(f"Error: Expected exactly one .sch and one .brd in {subfolder_path}, found {len(sch_files)} .sch and {len(brd_files)} .brd")
            
            # Finally, rename .sch and .brd files to match the subfolder name
            for ext in ['.sch', '.brd']:
                files = [f for f in os.listdir(subfolder_path) if f.lower().endswith(ext)]
                for f in files:
                    new_name = f"{subfolder}{ext}"
                    src = os.path.join(subfolder_path, f)
                    dst = os.path.join(subfolder_path, new_name)
                    if f != new_name:
                        try:
                            os.rename(src, dst)
                            print(f"Renamed {f} to {new_name} in {subfolder_path}")
                        except Exception as e:
                            print(f"Failed to rename {f}: {e}")

            print(f"-----------------------")

def check_sch_brd_in_subfolders(folder_path):
    """
    Checks each subfolder in folder_path for exactly one .sch and one .brd file,
    both named the same as the subfolder. Prints 'pass' if true, else 'no' and saves fail cases.
    Returns a list of failed subfolders.
    """
    fail_cases = []
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            sch_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith('.sch')]
            brd_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith('.brd')]
            if len(sch_files) == 1 and len(brd_files) == 1 and sch_files[0] == f"{subfolder}.sch" and brd_files[0] == f"{subfolder}.brd":
                pass
            else:
                fail_cases.append(subfolder)
    return fail_cases


def copy_all_sch_files(source_folder, destination_folder):
    """
    Copies all .sch files inside subfolders of source_folder to destination_folder.
    If the file already exists in destination_folder, it will not be copied.
    Returns a tuple: (count_copied, count_skipped)
    """
    os.makedirs(destination_folder, exist_ok=True)
    count_copied = 0
    count_skipped = 0
    for subfolder in os.listdir(source_folder):
        subfolder_path = os.path.join(source_folder, subfolder)
        if os.path.isdir(subfolder_path):
            for file in os.listdir(subfolder_path):
                if file.lower().endswith('.sch'):
                    src_file = os.path.join(subfolder_path, file)
                    dst_file = os.path.join(destination_folder, file)
                    if not os.path.exists(dst_file):
                        shutil.copy2(src_file, dst_file)
                        # print(f"Copied: {src_file} -> {dst_file}")
                        count_copied += 1
                    else:
                        # print(f"Skipped (already exists): {dst_file}")
                        count_skipped += 1
    print(f"Total copied: {count_copied}, Total skipped: {count_skipped}")
    return count_copied, count_skipped

In [ ]:
import os
import csv
from collections import Counter
import pathlib
import requests
from urllib.parse import urlparse

def combine_csv_files(src_folder, output_csv):
    """
    Combine all CSV files in a folder into a single CSV file.
    Assumes all CSVs have the same header.

    Args:
        src_folder (str): Path to folder containing CSV files
        output_csv (str): Path to save the combined CSV
    """
    csv_files = [f for f in os.listdir(src_folder) if f.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("No CSV files found in the source folder.")

    header_written = False

    with open(output_csv, "w", newline="", encoding="utf-8") as out_f:
        writer = None

        for csv_file in csv_files:
            csv_path = os.path.join(src_folder, csv_file)
            with open(csv_path, newline="", encoding="utf-8") as in_f:
                reader = csv.reader(in_f)
                header = next(reader)

                if not header_written:
                    writer = csv.writer(out_f)
                    writer.writerow(header)
                    header_written = True

                for row in reader:
                    writer.writerow(row)

    print(f"Combined {len(csv_files)} CSV files into: {output_csv}")



def get_zip_links_from_csv(csv_path):
    """
    Get a list of ZIP file links from the fourth column of a CSV file.

    Args:
        csv_path (str): Path to the CSV file

    Returns:
        list[str]: List of ZIP file URLs from the fourth column
    """
    zip_links = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader, None)  # skip header
        for row in reader:
            if len(row) >= 4:
                url = row[3].strip()
                if url.lower().endswith(".zip"):
                    zip_links.append(url)
    return zip_links




def find_duplicate_links(links):
    """
    Find duplicate links in a list.

    Args:
        links (list[str]): List of links

    Returns:
        list[str]: List of duplicate links (each appears once in output)
    """
    counter = Counter(links)
    duplicates = [link for link, count in counter.items() if count > 1]
    return duplicates


def remove_duplicates(links):
    """
    Remove duplicate links from a list while preserving order.

    Args:
        links (list[str]): List of links

    Returns:
        list[str]: List with duplicates removed
    """
    seen = set()
    unique_links = []
    for link in links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)
    return unique_links


def download_zips_to_named_folders(links, dest_folder, overwrite=False, timeout=30):
    """
    Download ZIP files from a list of URLs and save each into its own subfolder.
    Folder name is based on the ZIP name (without extension) from the URL.
    The ZIP file inside is always saved as 'eagle.zip'.
    If folder exists and overwrite=False, add ##1, ##2... until unique.

    Args:
        links (list[str]): List of ZIP URLs
        dest_folder (str): Destination base folder
        overwrite (bool): Overwrite if file exists
        timeout (int): Timeout for download in seconds

    Returns:
        tuple: (list of saved paths, list of failed URLs)
    """
    os.makedirs(dest_folder, exist_ok=True)
    saved_paths = []
    failed_urls = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }

    total = len(links)
    for idx, url in enumerate(links, start=1):
        try:
            # Base folder name from ZIP file name (without .zip)
            base_folder_name = pathlib.Path(urlparse(url).path).stem or f"file_{idx}"

            # Ensure unique folder if overwrite=False
            zip_folder = os.path.join(dest_folder, base_folder_name)
            if not overwrite:
                counter = 1
                while os.path.exists(zip_folder):
                    zip_folder = os.path.join(dest_folder, f"{base_folder_name}##{counter}")
                    counter += 1

            os.makedirs(zip_folder, exist_ok=True)
            dest_path = os.path.join(zip_folder, "eagle.zip")

            print(f"[{idx}/{total}] Downloading: {url} -> {dest_path}")

            with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                tmp_path = dest_path + ".part"
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 64):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp_path, dest_path)

            saved_paths.append(dest_path)

        except Exception as e:
            print(f"[{idx}/{total}] Failed to download {url}: {e}")
            failed_urls.append(url)

    # Summary report
    print("\n=== Download Summary ===")
    print(f"✅ Successful downloads: {len(saved_paths)}")
    print(f"❌ Failed downloads: {len(failed_urls)}")
    if failed_urls:
        print("Failed URLs:")
        for u in failed_urls:
            print("  -", u)

    return saved_paths, failed_urls



In [42]:
src_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"
output_csv = "/Users/taitinglu/Desktop/IMG2SCH/sparkfun_combined.csv"
dest_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip"

combine_csv_files(src_folder, output_csv)

Combined 13 CSV files into: /Users/taitinglu/Desktop/IMG2SCH/sparkfun_combined.csv


In [45]:
links = get_zip_links_from_csv(output_csv)
duplicates = find_duplicate_links(links)
print(len(duplicates))
unique_links = remove_duplicates(links)
print("Unique links:", len(unique_links))
download_zips_to_named_folders(unique_links, dest_folder)

508
Unique links: 577
[1/577] Downloading: https://cdn.sparkfun.com/datasheets/Robotics/BigEasyDriver_v16a.zip -> /Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/BigEasyDriver_v16a/eagle.zip
[2/577] Downloading: https://docs.sparkfun.com/SparkFun_Three_Phase_Motor_Driver-TMC6300/board_files/eagle_files.zip -> /Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/eagle_files/eagle.zip
[3/577] Downloading: https://cdn.sparkfun.com/assets/d/e/1/c/0/SparkFun_Auto_pHAT.zip -> /Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/SparkFun_Auto_pHAT/eagle.zip
[4/577] Downloading: http://www.sparkfun.com/datasheets/Robotics/TB6612FNG%20Breakout%20v11.zip -> /Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/TB6612FNG%20Breakout%20v11/eagle.zip
[4/577] Failed to download http://www.sparkfun.com/datasheets/Robotics/TB6612FNG%20Breakout%20v11.zip: 404 Client Error: Not Found for url: https://www.sparkfun.com/datasheets/Robotics/TB6612FNG%20Breakout%20v11.zip
[5/577] Downloading: https://cdn.sparkfun.com/datasheets/Robo

['/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/BigEasyDriver_v16a/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/eagle_files/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/SparkFun_Auto_pHAT/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/Haptic_Motor_Driver_DRV2605L/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/SparkFun_Servo_Trigger_v10a/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/EasyDriver_v45/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/Qwiic_Motor_Driver_Eagle/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/eagle_files/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/Pi_Servo_pHAT_v21/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/SparkFun_Servo_Trigger_Con_Rot/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/SparkFun_Ardumoto_v20/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip/RedBot-v14/eagle.zip',
 '/Users/taitinglu/Desktop/IMG2SCH/Spar

## Example

In [27]:
# Example usage:
folder_path       = r"C:\Users\Taiting\Desktop\yolo_training_24\full sch zip"
source_folder_path = r"/Users/taitinglu/Desktop/IMG2SCH/eagle_merged"
print(check_zip_in_subfolders(source_folder_path))

unzip_eagle_zips_in_subfolders(source_folder_path)


# fail_list = check_sch_brd_in_subfolders(folder_path)
# print("No Fail cases:", fail_list)


# Move all .sch files from subfolders to a single destination folder
source_folder_path = r"/Users/taitinglu/Desktop/IMG2SCH/eagle_merged"
destination_folder_path = r"/Users/taitinglu/Desktop/IMG2SCH/eagle_merged unzip"
# copy_all_sch_files(source_folder_path, destination_folder_path)

[]
Processing subfolder: Wireless_Motor_Driver_Shield
no eagle.zip in: /Users/taitinglu/Desktop/IMG2SCH/eagle_merged/Wireless_Motor_Driver_Shield
Error: Expected exactly one .sch and one .brd in /Users/taitinglu/Desktop/IMG2SCH/eagle_merged/Wireless_Motor_Driver_Shield, found 0 .sch and 0 .brd
-----------------------
Processing subfolder: SparkFun_Qwiic_Dual_Solid_State_Relay
no eagle.zip in: /Users/taitinglu/Desktop/IMG2SCH/eagle_merged/SparkFun_Qwiic_Dual_Solid_State_Relay
Error: Expected exactly one .sch and one .brd in /Users/taitinglu/Desktop/IMG2SCH/eagle_merged/SparkFun_Qwiic_Dual_Solid_State_Relay, found 0 .sch and 0 .brd
-----------------------
Processing subfolder: Buzzer-v15
no eagle.zip in: /Users/taitinglu/Desktop/IMG2SCH/eagle_merged/Buzzer-v15
Error: Expected exactly one .sch and one .brd in /Users/taitinglu/Desktop/IMG2SCH/eagle_merged/Buzzer-v15, found 0 .sch and 0 .brd
-----------------------
Processing subfolder: SparkFun_ProDriver_TC78H670FTG_eagle_files
no eagle.zi

## Process full sch

In [ ]:
def check_eagle_version(file_path, target_version="9.6.2"):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if 'eagle version' in line:
                if f'version="{target_version}"' in line:
                    print(f"File is already in Eagle version {target_version}")
                    return True
                else:
                    print(f"File is in a different version: {line.strip()}")
                    print("Please open and save it in Eagle version 9.6.2.")
                    return False
    print("Version info not found.")
    return False


def convert_sch_folder_to_images(sch_folder, img_folder,fail_csv):
    """
    Converts all .sch files in sch_folder to images in img_folder using visualize_schematic.
    Skips files that fail and saves their names and errors to a list.
    """
    os.makedirs(img_folder, exist_ok=True)
    sch_files = sorted([f for f in os.listdir(sch_folder) if f.lower().endswith('.sch')])[:-1]
    failed_files = []
    for idx, sch_file in enumerate(sch_files):
        sch_path = os.path.join(sch_folder, sch_file)
        img_path = os.path.join(img_folder, f"{os.path.splitext(sch_file)[0]}.png")
        print(f"[{idx}] Processing: {sch_file}")
        try:
            if check_eagle_version(sch_path):
                if not os.path.exists(img_path):
                    visualize_schematic(sch_path, save_path=img_path, draw_grid=False,draw_bounding_bbox=True)
                else:
                    continue  # Skip if image already exists
            else:
                print(f"Skipping {sch_file} due to version mismatch.")
                failed_files.append((sch_file, "version mismatch"))
        except Exception as e:
            print(f"Failed to process {sch_file}: {e}")
            failed_files.append((sch_file, str(e)))
        
        print("-----------------------")
        
    csv_path = os.path.join(img_folder, fail_csv)
    with open(csv_path, "w", newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["File", "Error"])
        writer.writerows(failed_files)
    print(f"Failed files saved to: {csv_path}")
    return failed_files

## Example

In [ ]:
# Example usage:
sch_folder = r"/Users/taitinglu/Desktop/IMG2SCH/full sch unzip"
img_folder = r"/Users/taitinglu/Desktop/IMG2SCH/full img unzip"
fail_csv = r"/Users/taitinglu/Desktop/IMG2SCH/failed_sch.csv"
sch_to_img_check =convert_sch_folder_to_images(sch_folder, img_folder,fail_csv)

# PCB Stat Cal

## Stats calculate funs

In [ ]:
import xml.etree.ElementTree as ET
import json
def total_pins(file_path):
    file = process_schematic_file(file_path)
    parts = file.get("parts", {})
    libraries = file.get("IC_library", [])

    # Step 1: Map component name → (library, deviceset, device)
    component_to_dev = {
        comp: {
            "deviceset": data.get("deviceset"),
            "device": data.get("device"),
            "library": data.get("library")
        }
        for comp, data in parts.items()
    }
    # Step 2: Extract device-to-pin mapping from libraries
    device_to_pins = {}
    for lib in libraries:
        lib_name = lib.get("name")
        devicesets = lib.get("devicesets", {}).get("deviceset", [])
        if isinstance(devicesets, dict):
            devicesets = [devicesets]

        for dset in devicesets:
            dset_name = dset.get("name")
            devices = dset.get("devices", {}).get("device", [])
            if isinstance(devices, dict):
                devices = [devices]

            gates = dset.get("gates", {}).get("gate", [])
            if isinstance(gates, dict):
                gates = [gates]

            symbols = lib.get("symbols", {}).get("symbol", [])
            if isinstance(symbols, dict):
                symbols = [symbols]

            # Step 3: Get pin names from symbol used in gates
            pin_names = set()
            
            for gate in gates:
                symbol_name = gate.get("symbol")
                print(f"Gate symbol: {symbol_name}")
                for sym in symbols:
                    if sym.get("name") == symbol_name:
                        pin_data = sym.get("pin", [])
                        if isinstance(pin_data, dict):
                            pin_data = [pin_data]
                        for pin in pin_data:
                            name = pin.get("name")
                            if name:
                                pin_names.add(name)

            for dev in devices:
                device_name = dev.get("name")
                key = (lib_name, dset_name, device_name)
                device_to_pins[key] = pin_names
    # Step 4: Combine component with its pin count
    component_pin_info = []
    for comp_name, info in component_to_dev.items():
        key = (info["library"], info["deviceset"], info["device"])
        pin_names = device_to_pins.get(key, set())
        if pin_names:  # Only keep components with non-zero pins
            component_pin_info.append({
                "Component": comp_name,
                "Library": info["library"],
                "Deviceset": info["deviceset"],
                "Device": info["device"],
                "Pin Count": len(pin_names),
                "Pins": sorted(pin_names)
            })

    # Step 5: Print result
    # for entry in component_pin_info:
    #     print(f"{entry['Component']} ({entry['Device']}): {entry['Pin Count']} pins → {entry['Pins']}")
    total_p = sum(entry['Pin Count'] for entry in component_pin_info)
    return total_p

def parse_eagle_brd_src(file_path, src_file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    file = process_schematic_file(src_file_path)

    # BOARD → PLAIN → DIMENSION or directly bounding box in ELEMENTS or SIGNALS
    drawing = root.find('drawing')
    board = drawing.find('board')
    layers = drawing.find('layers')
    # --- Get Board Dimensions ---
    min_x, min_y, max_x, max_y = float('inf'), float('inf'), float('-inf'), float('-inf')

    for elem in board.findall(".//element"):
        x = float(elem.get('x', 0))
        y = float(elem.get('y', 0))
        min_x = min(min_x, x)
        min_y = min(min_y, y)
        max_x = max(max_x, x)
        max_y = max(max_y, y)

    width = round(max_x - min_x, 2)
    height = round(max_y - min_y, 2)

    # --- Get Number of Layers ---
    
    layercount = 0
    for layer in layers.findall('layer'):
        number = int(layer.get('number', -1))
        active = layer.get('active', 'no')
        if 1 <= number <= 16 and active.lower() == 'yes':
            layercount += 1
    # print(layercount)

    # --- Get Number of Components ---
    elements = board.find('elements')
    num_components = len(elements.findall('element')) if elements is not None else 0


    # print(file['nets'])
    # for net in file['nets']['net']:
    #     print(net)


    # print(type(file["nets"]))
    # print(file["nets"])

    net_dict = file.get("nets", {})
    num_nets = len(net_dict)
    # print("Number of nets:", num_nets)
    # if isinstance(net_list, dict):
    #     net_list = [net_list]

        
    unique_pins = set()

    for net_name, net_data in net_dict.items():
        net_data_list = net_data if isinstance(net_data, list) else [net_data]

        for nd in net_data_list:
            # print(nd)
            pinrefs = nd.get("pinref", [])
            # if isinstance(segments, dict):
            #     segments = [segments]

            # for segment in segments:
            #     pinrefs = segment.get("pinref", [])
                
            if isinstance(pinrefs, dict):
                pinrefs = [pinrefs]
            
            for pinref in pinrefs:
                part = pinref.get("part", "?")
                pin = pinref.get("pin", "?")
                unique_pins.add(f"{part}.{pin}")

    
    a = total_pins(src_file_path)

    return {
        'Width': width,
        'Height': height,
        'Num Layers': layercount,
        'Num Components': num_components,
        'Num Pads': len(unique_pins),
        'Num Nets': num_nets,
        'Num of total pins': a
    }

# Example usage
brd_path = '/Users/linkaiyuan/文件/PSU/products/Arduino_Nano_Every/NANOEveryV3.0.brd'
src_path = '/Users/linkaiyuan/文件/PSU/products/Arduino_Nano_Every/NANOEveryV3.0.sch'
info = parse_eagle_brd_src(brd_path, src_path)
print(info)





## Stats Generate and save

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Collect all results
folder = "/Users/linkaiyuan/文件/PSU/products"
results = []
file_16_layer = []
bad_file = 0
for brd_file in glob.glob(os.path.join(folder, "**", "*.brd"), recursive=True):
    try:
        sch_file = brd_file.replace(".brd", ".sch")
        if not os.path.exists(sch_file):
            print(f"Missing .sch for {brd_file}")
            continue
        info = parse_eagle_brd_src(brd_file, sch_file)
        if info['Num Layers'] == 4:
            file_16_layer.append(brd_file)
        info['Filename'] = os.path.basename(brd_file)
        results.append(info)
    except Exception as e:
        print(f"Error processing {brd_file}: {e}")
        bad_file+=1
        continue

# Convert to DataFrame and save
df = pd.DataFrame(results)
csv_path = "/Users/linkaiyuan/文件/PSU/eagle_stats.csv"
df.to_csv(csv_path, index=False)
print(f"Saved stats to {csv_path}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
csv_path = "/Users/linkaiyuan/文件/PSU/eagle_stats.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()  # Strip spaces

# List of fields to plot
fields = [
    'Num Components',
    'Num Nets',
    'Num Pads',
    'Num Layers',
    'Num of total pins',
    'Width',
    'Height'
]

# Convert all to numeric (in case of object dtype)
for field in fields:
    df[field] = pd.to_numeric(df[field], errors='coerce')

# Plot and save each field
for field in fields:
    df_sorted = df.sort_values(by=field, ascending=False).reset_index(drop=True)
    
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(df_sorted)), df_sorted[field])
    plt.title(f"{field} per Board")
    plt.xlabel("Board Index")
    plt.ylabel(field)
    plt.tight_layout()

    # Save to PNG
    safe_field_name = field.replace(" ", "_").lower()
    save_path = f"{safe_field_name}_per_board.png"
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved plot: {save_path}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
csv_path = "/Users/linkaiyuan/文件/PSU/eagle_stats.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()  # Strip spaces

# List of fields to plot
fields = [
    'Num Components',
    'Num Nets',
    'Num Pads',
    'Num Layers',
    'Num of total pins',
    'Width',
    'Height'
]

# Convert all to numeric (in case of object dtype)
for field in fields:
    df[field] = pd.to_numeric(df[field], errors='coerce')

# Plot and save each field as a frequency plot (histogram)
for field in fields:
    plt.figure(figsize=(10, 5))
    
    # Plot frequency (histogram) of each field
    plt.hist(df[field].dropna(), bins=15, color='blue', edgecolor='black', alpha=0.7)
    
    plt.title(f"Frequency Distribution of {field}")
    plt.xlabel(field)
    plt.ylabel("Frequency")
    plt.tight_layout()

    # Save to PNG
    safe_field_name = field.replace(" ", "_").lower()
    save_path = f"{safe_field_name}_frequency.png"
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved plot: {save_path}")


In [ ]:
import json
schematic_info = process_schematic_file("/Users/linkaiyuan/文件/PSU/products/Bus_Pirate_-_v3.6a/BusPirate-v3.6a.sch")


output_path = "/Users/linkaiyuan/文件/PSU/test_multi_gate_schematic_info.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(schematic_info, f, ensure_ascii=False, indent=4)

print(f"Saved schematic info to {output_path}")

# test for multi gate sch generate

In [ ]:
from xml.etree.ElementTree import ParseError
import os
import stat
if __name__ == "__main__":


    root_dir = "/Users/linkaiyuan/文件/PSU/products"
    temp_path = "/Users/linkaiyuan/文件/PSU/template_sch/template.sch"
    # os.makedirs(temp_path, exist_ok=True)
    os.chmod(temp_path, stat.S_IRWXU)
    os.chmod(root_dir, stat.S_IRWXU)
    import shutil
    import glob
    import os
    import json
    import stat

    unique_symbols = set()
    count = 0

    for sch_file_path in glob.glob(os.path.join(root_dir, "**", "*.sch"), recursive=True):
        # if count > 1:
        #     break
        name = os.path.basename(sch_file_path)
        stem = os.path.splitext(name)[0]
        os.chmod(sch_file_path, stat.S_IRWXU)
        try:
            schematic_info = process_schematic_file(sch_file_path)
            for part_name in schematic_info['parts'].items():
                # count += 1
                # if count > 1:
                #     break
                # print(part_name)
                symbol = part_name[1]['instances'][0]['symbol']
                lib = part_name[1]['library']
                if (symbol, lib) in unique_symbols:
                    continue
                unique_symbols.add((symbol, lib))

                new_name = f"{os.path.splitext(name)[0]}_{lib}_{symbol}.sch".replace("/", "slash")
                if ("MGM240P_EFM32GG12B410F1024GL120_EFM32GG12B410F1024GL120_A" in new_name):
                    print("name find:",sch_file_path)
                # dst_path = os.path.join("/Users/linkaiyuan/文件/PSU/test_single_sch", new_name)
                dst_path = os.path.join("/Users/linkaiyuan/文件/PSU/test_multigate", new_name)
                # os.chmod(dst_path, stat.S_IRWXU)
                os.chmod(temp_path, stat.S_IRWXU)
                if os.path.exists(dst_path):
                    os.chmod(dst_path, stat.S_IRWXU)
                shutil.copy(temp_path, dst_path)
                os.chmod(dst_path, stat.S_IRWXU)
                # print(dst_path)

                    
                deviceset = part_name[1]['deviceset']
                device = part_name[1]['device']
                value = part_name[1]['value']
                print(sch_file_path, dst_path, lib, deviceset, device, value)
                merge_schematic_libraries(sch_file_path, dst_path, debug=False)
                add_unit_to_schematic(dst_path,lib, deviceset, device, value)
                print(f"Orgional schematic file: {sch_file_path}")
                print(f"New schematic file: {dst_path}")
                print(f"added unit: {lib}, {deviceset}, {device}, {value}")
    # # device = ""
    # # value = ""
        except AttributeError as e:
            # print(f"Skipping {sch_file_path} due to error: {e}")
            continue
        except ParseError as e:
            # print(f"Skipping {sch_file_path} due to XML parsing error: {e}")
            continue
        except TypeError as e:
            # print(f"Skipping {sch_file_path} due to TypeError: {e}")
            continue


## Single Sch to Image Tesing

In [ ]:
import glob
import os
import traceback

save_path = "/Users/linkaiyuan/Downloads/"
sch_file = "/Users/linkaiyuan/文件/PSU/full sch zip/Qwiic_Fuel_Gauge_-_MAX17048/Qwiic_Fuel_Gauge_-_MAX17048.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/full sch unzip/SparkFun_GPS_Dead_Reckoning_Breakout_-_NEO-M8U_(Qw.sch"
# sch_file = '/Users/linkaiyuan/文件/PSU/products/SparkFun_Artemis_Development_Kit_with_Camera/ArtemisDevKit.sch'
# sch_file = "/Users/linkaiyuan/文件/PSU/multi_instances/MicroMod_Processor_STM32WB5MMG_stm32wb5mmg_STM32WB5MMG_COMMS.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_Qwiic_Buzzer/SparkFun_Qwiic_Buzzer.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_MicroMod_WiFi_Function_Board_-_DA16200/SparkFun_MicroMod_DA16200_Function.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/WAV_Trigger/wavtrigger_v11.sch"


try:
    # parse schematic and get its full‐sheet bounds
    schematic_info = process_schematic_file(sch_file)
    # src_xlim, src_ylim = get_schematic_bounding_box_from_schematic_info(schematic_info)
    print(sch_file)
    # render and save
    png_path = os.path.join(save_path, "version_with_bounding" + os.path.basename(sch_file).replace(".sch", ".png"))
    sizes    = visualize_schematic(sch_file, png_path, draw_bounding_bbox = True)
    # sizes   = visualize_schematic_nobounding(sch_file, save_path=png_path)
    px_per_x, px_per_y, _, _, img_w, img_h, bounding_box_list, src_xlim, src_ylim = sizes


    ic_library = schematic_info.get('IC_library')
    parts_info = schematic_info.get('parts')

    # loop over each part instance
    for box in bounding_box_list:
            part_name = box['part_name']
            # symbol_element = get_symbol_element_of_instance_from_library(
            #     ic_library, part_name, '', parts_info
            # )
            # if not symbol_element:
            #     continue

            # get this symbol’s world‐space bbox
            


            key = box['key']
            x_center = box['x_center']
            y_center = box['y_center']

            box_length = box['length']
            box_width = box['width']

            x0 = x_center - box_length/2
            y0 = y_center - box_width/2

            x1 = x_center + box_length/2
            y1 = y_center + box_width/2

            # rot_inst = instance['rot']
            # x_inst   = float(instance['x'])
            # y_inst   = float(instance['y'])
            # (x0, x1), (y0, y1) = get_bounding_box_for_symbol_instance(
            #     symbol_element, x_inst, y_inst, rot_inst
            # )

            # convert to pixels
            w_px  = (x1 - x0) * px_per_x
            h_px  = (y1 - y0) * px_per_y
            cx_px = ((x0 + x1)/2 - src_xlim[0]) * px_per_x
            cy_px = ((y0 + y1)/2 - src_ylim[0]) * px_per_y

            # normalize for YOLO (flip y so origin is top‐left)
            norm_x_center = cx_px / img_w
            norm_y_center = 1.0 - (cy_px / img_h)
            norm_width    = w_px    / img_w
            norm_height   = h_px    / img_h

            # lookup class
            # lib          = part_data['library']
            # symbol       = instance['symbol']
            # base_name    = os.path.splitext(os.path.basename(sch_file))[0]
            # key          = f"{base_name}_{lib}_{symbol}"


            if part_name == 'SheetInfo':
                key = "SheetInfo"
            elif box.get('cat', '') == 'symbol':
                key =  box.get('key', '')
            else:
                key = 'Wire'

    #         if class_map is not None:
    #             class_number = class_map.get(key, -1)
    #             if class_number == -1:
    #                 continue

    #         label_file += f"{class_number} {norm_x_center:.6f} {norm_y_center:.6f} {norm_width:.6f} {norm_height:.6f}\n"

    # # write out labels
    # # label_filename = 
    # with open(label_filename, "w") as f:
    #     f.write(label_file)

except Exception as e:
    print(f"Skipping {sch_file} due to error: {e}")
    traceback.print_exc()

## Process full sch

In [ ]:
def check_eagle_version(file_path, target_version="9.6.2"):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if 'eagle version' in line:
                if f'version="{target_version}"' in line:
                    print(f"File is already in Eagle version {target_version}")
                    return True
                else:
                    print(f"File is in a different version: {line.strip()}")
                    print("Please open and save it in Eagle version 9.6.2.")
                    return False
    print("Version info not found.")
    return False


def convert_sch_folder_to_images(sch_folder, img_folder,fail_csv):
    """
    Converts all .sch files in sch_folder to images in img_folder using visualize_schematic.
    Skips files that fail and saves their names and errors to a list.
    """
    os.makedirs(img_folder, exist_ok=True)
    sch_files = [f for f in os.listdir(sch_folder) if f.lower().endswith('.sch')][:-1]
    failed_files = []
    for idx, sch_file in enumerate(sch_files):
        sch_path = os.path.join(sch_folder, sch_file)
        img_path = os.path.join(img_folder, f"{os.path.splitext(sch_file)[0]}.png")
        print(f"[{idx}] Processing: {sch_file}")
        try:
            if check_eagle_version(sch_path):
                if not os.path.exists(img_path):
                    visualize_schematic(sch_path, save_path=img_path, draw_grid=False,draw_bounding_bbox=True)
                else:
                    continue  # Skip if image already exists
            else:
                print(f"Skipping {sch_file} due to version mismatch.")
                failed_files.append((sch_file, "version mismatch"))
        except Exception as e:
            print(f"Failed to process {sch_file}: {e}")
            failed_files.append((sch_file, str(e)))
        
        print("-----------------------")
        
    csv_path = os.path.join(img_folder, fail_csv)
    with open(csv_path, "w", newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["File", "Error"])
        writer.writerows(failed_files)
    print(f"Failed files saved to: {csv_path}")
    return failed_files

In [ ]:
# Example usage:
sch_folder = "/Users/linkaiyuan/文件/PSU/full sch"
img_folder = "/Users/linkaiyuan/文件/PSU/test/img_100"
fail_csv = "/Users/linkaiyuan/文件/PSU/test/fail_csv.csv"
sch_to_img_check =convert_sch_folder_to_images(sch_folder, img_folder,fail_csv)

In [ ]:
import glob
import os
import traceback

save_path = "/Users/linkaiyuan/文件/PSU/test/img_100"
# sch_file = '/Users/linkaiyuan/文件/PSU/products/SparkFun_Artemis_Development_Kit_with_Camera/ArtemisDevKit.sch'
folder_path = '/Users/linkaiyuan/文件/PSU/test/full sch'
count = 0
for p in ([save_path]):
    os.makedirs(p, exist_ok=True)
sch_files = sorted(glob.glob(os.path.join(folder_path, "*.sch")))
for sch_file in sch_files:
# sch_file = "/Users/linkaiyuan/文件/PSU/multi_instances/MicroMod_Processor_STM32WB5MMG_stm32wb5mmg_STM32WB5MMG_COMMS.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_Qwiic_Buzzer/SparkFun_Qwiic_Buzzer.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/SparkFun_MicroMod_WiFi_Function_Board_-_DA16200/SparkFun_MicroMod_DA16200_Function.sch"
# sch_file = "/Users/linkaiyuan/文件/PSU/products/WAV_Trigger/wavtrigger_v11.sch"
    if count > 100:
        break
    count += 1
    try:
        # parse schematic and get its full‐sheet bounds
        schematic_info = process_schematic_file(sch_file)
        # src_xlim, src_ylim = get_schematic_bounding_box_from_schematic_info(schematic_info)
        print(sch_file)
        # render and save
        png_path = os.path.join(save_path,  os.path.basename(sch_file).replace(".sch", ".png"))
        sizes    = visualize_schematic(sch_file, png_path, draw_bounding_bbox = True)
        # sizes   = visualize_schematic_nobounding(sch_file, save_path=png_path)
        px_per_x, px_per_y, _, _, img_w, img_h, bounding_box_list, src_xlim, src_ylim = sizes


        ic_library = schematic_info.get('IC_library')
        parts_info = schematic_info.get('parts')

        # loop over each part instance
        for box in bounding_box_list:
                part_name = box['part_name']
                # symbol_element = get_symbol_element_of_instance_from_library(
                #     ic_library, part_name, '', parts_info
                # )
                # if not symbol_element:
                #     continue

                # get this symbol’s world‐space bbox
                


                key = box['key']
                x_center = box['x_center']
                y_center = box['y_center']

                box_length = box['length']
                box_width = box['width']

                x0 = x_center - box_length/2
                y0 = y_center - box_width/2

                x1 = x_center + box_length/2
                y1 = y_center + box_width/2

                # rot_inst = instance['rot']
                # x_inst   = float(instance['x'])
                # y_inst   = float(instance['y'])
                # (x0, x1), (y0, y1) = get_bounding_box_for_symbol_instance(
                #     symbol_element, x_inst, y_inst, rot_inst
                # )

                # convert to pixels
                w_px  = (x1 - x0) * px_per_x
                h_px  = (y1 - y0) * px_per_y
                cx_px = ((x0 + x1)/2 - src_xlim[0]) * px_per_x
                cy_px = ((y0 + y1)/2 - src_ylim[0]) * px_per_y

                # normalize for YOLO (flip y so origin is top‐left)
                norm_x_center = cx_px / img_w
                norm_y_center = 1.0 - (cy_px / img_h)
                norm_width    = w_px    / img_w
                norm_height   = h_px    / img_h

                # lookup class
                # lib          = part_data['library']
                # symbol       = instance['symbol']
                # base_name    = os.path.splitext(os.path.basename(sch_file))[0]
                # key          = f"{base_name}_{lib}_{symbol}"


                if part_name == 'SheetInfo':
                    key = "SheetInfo"
                elif box.get('cat', '') == 'symbol':
                    key =  box.get('key', '')
                else:
                    key = 'Wire'

        #         if class_map is not None:
        #             class_number = class_map.get(key, -1)
        #             if class_number == -1:
        #                 continue

        #         label_file += f"{class_number} {norm_x_center:.6f} {norm_y_center:.6f} {norm_width:.6f} {norm_height:.6f}\n"

        # # write out labels
        # # label_filename = 
        # with open(label_filename, "w") as f:
        #     f.write(label_file)

    except Exception as e:
        print(f"Skipping {sch_file} due to error: {e}")
        traceback.print_exc()

In [ ]:
sch_files

## val move to train

In [ ]:
import os
import shutil
import random

# Change this to your actual dataset root
dataset_dir = '/Users/linkaiyuan/文件/PSU/yolo_training_13/dataset'

val_images_dir = os.path.join(dataset_dir, 'val/images')
val_labels_dir = os.path.join(dataset_dir, 'val/labels')

train_images_dir = os.path.join(dataset_dir, 'train/images')
train_labels_dir = os.path.join(dataset_dir, 'train/labels')

# List all images in validation
val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.png'))]

# Shuffle and select N images to move
random.seed(42)
n_to_move = 200  # change as needed
selected = random.sample(val_images, min(n_to_move, len(val_images)))

for img_file in selected:
    label_file = os.path.splitext(img_file)[0] + '.txt'

    # Full source paths
    src_img = os.path.join(val_images_dir, img_file)
    src_lbl = os.path.join(val_labels_dir, label_file)

    # Full destination paths
    dst_img = os.path.join(train_images_dir, img_file)
    dst_lbl = os.path.join(train_labels_dir, label_file)

    # Move image and label
    shutil.move(src_img, dst_img)
    if os.path.exists(src_lbl):
        shutil.move(src_lbl, dst_lbl)
    else:
        print(f"[WARNING] No label found for {img_file}")


## version control: old eagle version to 9.6.2

In [ ]:
def check_eagle_version(file_path, target_version="9.6.2"):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if 'eagle version' in line:
                if f'version="{target_version}"' in line:
                    print(f"File is already in Eagle version {target_version}")
                    return True
                else:
                    print(f"File is in a different version: {line.strip()}")
                    print("Please open and save it in Eagle version 9.6.2.")
                    return False
    print("Version info not found.")
    return False

import glob
import os


output_dir = "./outversion_sch_test"
csv_path = os.path.join(output_dir, "outdated_sch_files.csv")
os.makedirs(output_dir, exist_ok=True)

folder_path       = r"/Users/linkaiyuan/文件/PSU/products"
outdated_files = []
for sch_file in glob.glob(os.path.join(folder_path, "**", "*.sch"), recursive=True):
    try:
        if not check_eagle_version(sch_file):
            outdated_files.append(sch_file)
    except UnicodeDecodeError as e:
        outdated_files.append(sch_file)
        continue

import csv
csv_path = "./outversion_sch_test/outdated_sch_files.csv"
with open(csv_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Outdated Schematic Files"])
    for path in outdated_files:
        writer.writerow([path])

print(f"\nSaved outdated .sch files to {csv_path}")

## save sch to 9.6.2 version

In [ ]:
#!/usr/bin/env python3
import subprocess
import csv

# Configuration
CSV_FILE = "/path/to/your/files.csv"  # Update this path
eagle_path = "/Applications/EAGLE-9.6.2/eagle.app/Contents/MacOS/Eagle"
ulp_path = "/Users/linkaiyuan/文件/PSU/outversion_sch_test/dummy_edit.ulp"

# Read file paths from CSV
files = []
with open(CSV_FILE, 'r', encoding='utf-8') as csvfile:
    reader = csv.reader(csvfile)
    
    # Skip header row if your CSV has one
    # next(reader)  # Uncomment this line if you have headers
    
    for row in reader:
        if row and row[0].strip():  # row[0] is first column
            files.append(row[0].strip())

print(f"Found {len(files)} files in CSV")

# Process each file
for i, sch_file in enumerate(files, 1):
    print(f"[{i}/{len(files)}] Processing: {sch_file}")
    
    cmd = [
        eagle_path,
        "-C", 
        f"RUN {ulp_path}; QUIT;",
        sch_file
    ]
    
    try:
        result = subprocess.run(cmd, check=True)
        print(f"✓ Success: {sch_file}")
    except subprocess.CalledProcessError as e:
        print(f"✗ Failed: {sch_file} (return code: {e.returncode})")
    except Exception as e:
        print(f"✗ Error: {sch_file} - {e}")

print("Done!")

## extract single component to sch

In [ ]:
import os, glob, shutil, stat
from xml.etree.ElementTree import ParseError

if __name__ == "__main__":
    root_dir   = "/Users/linkaiyuan/文件/PSU/full sch unzip"
    img_check_root = '/Users/linkaiyuan/文件/PSU/full img'
    temp_path  = "/Users/linkaiyuan/文件/PSU/template_sch/template.sch"
    # out_dir    = "/Users/linkaiyuan/文件/PSU/single_deivce_test"
    out_dir    = "/Users/linkaiyuan/文件/PSU/test/drawable_sch_device_test"
    os.makedirs(out_dir, exist_ok=True)
    part = {}
    unique_keys = set()
    count = 1
    for sch_path in glob.glob(os.path.join(root_dir, "**", "*.sch"), recursive=True):
        if count >100:
            break
        count +=1
        try:
            # sch_path = "/Users/linkaiyuan/文件/PSU/full sch unzip/Ambient_Light_Sensor_-_VEML7700_(Qwiic).sch"
            schematic_info = process_schematic_file(sch_path)
        except (AttributeError, ParseError, TypeError, ValueError):
             continue

        stem = os.path.splitext(os.path.basename(sch_path))[0]
        print(stem)

        img_path = os.path.join(img_check_root, stem + ".png")
        if not os.path.isfile(img_path):
            print("file ",sch_path, "unable to print", "no image in:", img_path)
            
            continue
        lab_dict = schematic_info['IC_library']
        part = {}
        # iterate over every part...
        for lib in lab_dict:
            name = lib['name']
            # des = lib['description']
            if not lib.get('packages', None):
                continue
            if lib['packages'].get('package', None):
                pack_list = lib['packages']['package'] if isinstance(lib['packages']['package'], list) else [lib['packages']['package']]
                symbol_list = lib['symbols']['symbol'] if isinstance(lib['symbols']['symbol'], list) else [lib['symbols']['symbol']]
                deviceset_list = lib['devicesets']['deviceset'] if isinstance(lib['devicesets']['deviceset'], list) else [lib['devicesets']['deviceset']]

                
                # print("Here is part_name:", part_name)
                # lib       = part_info['library']
                # deviceset = part_info['deviceset']
                # device    = part_info['device']
                # value     = part_info['value']

                # now iterate _all_ instances
                for idx, deviceset in enumerate(deviceset_list):
                    deviceset_name = deviceset['name']

                    # if you still want to dedupe by (lib,symbol) you can:
                    # deviceset_prefix = deviceset['prefix']
                    # deviceset_des = deviceset['description']
                    deviceset_gate_list = deviceset['gates']['gate'] if isinstance(deviceset['gates']['gate'], list) else [deviceset['gates']['gate']]
                    deviceset_device_list = deviceset['devices']['device'] if isinstance(deviceset['devices']['device'], list) else [deviceset['devices']['device']]
                    for idx_d, device in enumerate(deviceset_device_list):
                        device_name = device['name']
                        # device_package = device['package']
                        # device_connect = device['connects']
                        device_tech_name = ''
                        device_tech_value = ''
                        device_tech_list = device['technologies']['technology'] if isinstance(device['technologies']['technology'], list) else [device['technologies']['technology']]
                        if device_tech_list[0].get('attribute', None):
                            device_attr = device_tech_list[0]['attribute'] if isinstance(device_tech_list[0]['attribute'], list) else [device_tech_list[0]['attribute']]
                            device_tech_name = device_attr[0]['name']
                            device_tech_value = device_attr[0]['value']
                        new_name = f"{stem}#{name}#{deviceset_name}.sch".replace("/", "slash")



                        key = (stem, name, deviceset_name)
                        if key in unique_keys:
                            continue
                        unique_keys.add(key)

                    # build a unique filename per instance

                        dst = os.path.join(out_dir, new_name)

                    # copy your template & merge/add as before
                        os.chmod(temp_path, stat.S_IRWXU)
                        shutil.copy(temp_path, dst)
                        os.chmod(dst,       stat.S_IRWXU)

                    # merge & add one unit
                        try:
                            merge_schematic_libraries(sch_path, dst, debug=False)
                            add_unit_to_schematic(dst, name, deviceset_name, device_name, device_tech_value)

                            print(f"""
                                Source:    {sch_path}
                                Instance:  [{idx}] symbol={device_name}, lib={lib}, deviceset={deviceset_name}, device={device}, value={value}
                                Written to: {dst}
                                """)
                        except Exception as e:
                            print(sch_path)
                            break

## check correctness of single sch

In [ ]:
#!/usr/bin/env python3
import subprocess
import glob
import os

# Configuration
DIRECTORY = "/Users/linkaiyuan/文件/PSU/test/drawable_sch_device_test"  # Update this path to your directory
eagle_path = "/Applications/EAGLE-9.6.2/eagle.app/Contents/MacOS/Eagle"
ulp_path = "/Users/linkaiyuan/文件/PSU/outversion_sch_test/dummy_edit.ulp"

# Find all .sch files in the directory
# Option 1: Only files in the specified directory (not subdirectories)
files = glob.glob(os.path.join(DIRECTORY, "*.sch"))

# Option 2: Include all .sch files in subdirectories (recursive)
# files = glob.glob(os.path.join(DIRECTORY, "**/*.sch"), recursive=True)

print(f"Found {len(files)} .sch files in directory")

# Process each file
for i, sch_file in enumerate(files, 1):
    print(f"[{i}/{len(files)}] Processing: {sch_file}")
    
    cmd = [
        eagle_path,
        "-C", 
        f"RUN {ulp_path}; QUIT;",
        sch_file
    ]
    
    try:
        result = subprocess.run(cmd, check=True)
        print(f"✓ Success: {sch_file}")
    except subprocess.CalledProcessError as e:
        print(f"✗ Failed: {sch_file} (return code: {e.returncode})")
    except Exception as e:
        print(f"✗ Error: {sch_file} - {e}")

print("Done!")